# Practice 3 全解 · 中文翻译 + 逐句讲解

> 这份笔记本是**学习/复习用**的，不是交作业用的那一份。两份文件的分工：
>
> | 文件 | 用途 | 语言 |
> |---|---|---|
> | `homework-practice-03-linclass-solved.ipynb` | 交作业（已修错、补全所有问答） | 英文 |
> | `practice3-中文翻译与讲解.ipynb`（本文件） | 看懂每一个知识点、每一个变量 | 中文 |
>
> 建议的读法：先读本文件对应小节 → 再回作业文件看代码 → 有疑问再回来读「为什么」。

## 这份笔记怎么用

- 每个任务都按 **📖 原文 → 🇨🇳 翻译 → 🧠 讲解 → 💻 代码里的每个名字** 四段组织。
- 讲解里的代码块都是**能独立运行的小实验**（很快，几秒钟），跑一遍比读十遍有用。
- 第六章是**错误清单**：原解答里哪些地方是错的、我改成了什么、为什么。
- 第七章是**变量名 / 函数速查表**，卡住的时候直接查。

## 目录

1. 前置知识速成（如果你完全没基础，先读这一章）
2. Part 1：SVM、逻辑回归、概率校准
3. Part 2：类别变量的处理（one-hot 与计数器）
4. Part 3：特征选择（embedded / filter / wrapper）
5. Part 4：模型的经济效益（利润、阈值）
6. 原解答的错误清单与修改记录
7. 附录：速查表、常见报错

---
# 1. 前置知识速成

这一章不讲作业，只把后面要用到的概念铺好。每一节都很短，看完再往下走。

## 1.1 机器学习在干什么

一句话：**给你一堆「有过答案的例子」，让你学会对「没答案的新例子」做判断。**

- 每个例子叫一个**样本**（sample / object），作业里就是「一个客户」或「一个人造数据点」。
- 描述样本的那些数字叫**特征**（feature），全部样本的特征摞在一起就是矩阵 `X`，形状是 `(样本数, 特征数)`。
- 我们要预测的答案叫**标签**（label / target），记作 `y`，长度等于样本数。
- 二分类问题里 `y` 只有两个取值：`0` 和 `1`（作业里常写成「负类 / 正类」，正类就是我们关心的那件事，比如「客户答应了」）。

作业里的实际形状：
- Part 1 人造数据：`X` 是 `(10000, 10)`，`y` 是 `(10000,)`。
- Part 2~4 银行数据：`X` 是 `(41188, 19)`，`y` 是「客户有没有认购定期存款」。

## 1.2 为什么要分成训练集和测试集

这是整个机器学习里最容易犯、后果最严重的一个错。

> 类比：老师给你 100 道练习题的答案让你背，然后**用同一批题**考你，你考 100 分。这能说明你学会了吗？不能，你只是背下来了。

所以标准做法是：把数据切成两份。

| 名字 | 用途 | 作业里的变量 |
|---|---|---|
| 训练集 train | 模型从里面学参数 | `X_train, y_train` |
| 测试集 test | 只在最后用一次，估计「将来在真实世界能有多好」 | `X_test, y_test` |

铁律：**测试集不能用来做任何决定**（不能用来选超参数、不能用来选特征、不能用来选阈值）。一旦你用测试集做了决定，它就不再是「没见过的数据」，你报出来的分数就是虚高的。

后面 Part 3 的 wrapper 方法、Part 4 的调阈值，作业都特意强调这一点——这不是啰嗦，这是本课的核心考点之一。

## 1.3 模型的两种输出：硬标签 vs 分数

初学者最容易混的一点，先记住这张表：

| 方法 | 返回什么 | 例子 | 用途 |
|---|---|---|---|
| `model.predict(X)` | 硬标签 0/1 | `[0, 1, 1, 0]` | 最终决策 |
| `model.predict_proba(X)` | 每个类的概率，形状 `(n, 2)` | `[[0.7, 0.3], ...]` | 排序、算 AUC、调阈值 |
| `model.decision_function(X)` | 到分隔面的**有符号距离** | `[-1.2, 0.4, 2.7]` | 只有 SVM 这类模型有 |

`predict_proba(X)[:, 1]` 里那个 `[:, 1]` 的意思是「所有行，第 1 列」——因为第 0 列是「属于负类的概率」，第 1 列才是「属于正类的概率」。这是全作业出现频率最高的一个写法。

**为什么分数很重要？** 因为 0/1 丢掉了很多信息。两个客户，一个「99% 会答应」，一个「51% 会答应」，`predict` 都说 1，看不出差别；但只给你 100 个电话额度时，你肯定先打 99% 那个。**分数保留了排序，排序才是钱。**

## 1.4 过拟合（overfitting）

模型在训练集上表现很好，在测试集上表现差，叫过拟合。原因通常是模型「记住了训练数据里的偶然细节」而不是「学到了规律」。

作业里会反复看到这个模式：**训练集分数很高、测试集分数明显更低 = 过拟合**。Part 2 的「计数器泄漏」就是最典型的例子。

对付过拟合的常见手段（作业里全都会出现）：
- 正则化（`C` 参数就是管这个的，见 1.5）
- 加噪声（Part 2 Task 6）
- 交叉验证 + 只用训练集调参

## 1.5 正则化与参数 `C`

线性模型（逻辑回归、线性 SVM）本质上都在找一组权重 `w`，让「预测错」的总惩罚最小。但光最小化错误还不够——如果允许权重无限大，模型可以对训练数据 100% 正确，但那是死记硬背。

所以目标函数长这样：

$$\text{Loss}(w) = \underbrace{\sum_i \text{error}_i}_{\text{拟合数据}} + \underbrace{\frac{1}{2C}\lVert w\rVert^2}_{\text{别太极端}}$$

`C` 是**你自己设定的数**（超参数），含义反直觉，务必记牢：

| `C` 很小（如 0.001） | `C` 很大（如 100） |
|---|---|
| 正则化**强**，权重被压得很小 | 正则化**弱**，权重几乎不受限制 |
| 模型「保守」，容易欠拟合 | 模型「激进」，容易过拟合 |

一句话记法：**`C` 是「我允许模型多自由」的旋钮，`C` 越大越自由。**

> 为什么作业要用交叉验证去「选 C」？因为 C 不是模型自己学出来的，是我们手动试出来的。而试的时候只能用训练集，不能碰测试集。

## 1.6 评价指标：这四个词必须分清

假设模型说「这 100 个客户会买」，实际只有 20 个真买了：

|  | 真实是正类 | 真实是负类 |
|---|---|---|
| **预测为正类** | TP = 20（真正例，猜对了） | FP = 80（假正例，白打了电话） |
| **预测为负类** | FN = 30（假负例，漏掉的真客户） | TN = 剩余（真负例，正确地没打） |

由这四个数派生出的指标：

$$\text{precision（精确率）} = \frac{TP}{TP+FP} = \frac{\text{我打的电话里有多少是真的}}{\text{我打出去的电话数}}$$

$$\text{recall（召回率）} = \frac{TP}{TP+FN} = \frac{\text{真的客户我找到了多少}}{\text{所有真客户}}$$

$$\text{FPR（假正率）} = \frac{FP}{FP+TN} = \frac{\text{无辜的人被我打扰的比例}}{\text{所有无辜的人}}$$

直观记忆：
- **precision 关心「我说是的时候有多准」**——怕冤枉好人。
- **recall 关心「该抓的我漏了多少」**——怕放过坏人。
- 这两个天然打架：你把阈值降到 0（全都预测为正），recall = 1，但 precision 掉到谷底。所以永远要一起看。

## 1.7 ROC 曲线与 PR 曲线

模型给的是分数，不是 0/1。**阈值**（threshold）就是「分数大于多少算正类」。

- 阈值从 1 慢慢降到 0，每个阈值都算出一组 `(FPR, TPR)`，把这些点连起来就是 **ROC 曲线**。
- 同样地，每个阈值算出一组 `(recall, precision)`，连起来就是 **PR 曲线**。

**AUC** = Area Under Curve，曲线下的面积，取值 0~1，越大越好：
- `AUC-ROC = 0.5` 表示和抛硬币一样，`1.0` 表示完美。
- `AUC-PR` 的「随机基线」不是 0.5，而是**正类的比例**（本作业银行数据是 0.11）。这点非常关键，见下面的小实验。

In [ ]:
# 小实验：为什么正类稀少时要用 AUC-PR 而不是 AUC-ROC
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

rng = np.random.RandomState(0)
n = 20000

for pos_rate in [0.5, 0.11, 0.02]:
    y = (rng.rand(n) < pos_rate).astype(int)
    score = rng.rand(n)                      # 完全无用的「模型」：随机分数
    print("正类比例 %5.2f | AUC-ROC = %.3f | AUC-PR = %.3f"
          % (pos_rate, roc_auc_score(y, score), average_precision_score(y, score)))

看输出你会发现一件很有意思的事：

- **AUC-ROC 永远是 0.5**，无论正类多稀少——因为随机模型就是随机模型。
- **AUC-PR 会一路掉到 0.02**，恰好等于正类比例。

但真正的问题在反方向：如果模型「稍微有点用」，正类稀少的场景下 AUC-ROC 会显得**虚高**。原因是 FPR 的分母 `FP+TN` 里 TN 特别大（98% 都是负类），所以 FPR 天然就很小，ROC 曲线很容易向左上角贴。而 PR 曲线的 precision 分母是 `TP+FP`，完全不受大量负类「稀释」的影响。

一句话：**正类少 + 错误代价不对称 → 用 PR 系指标；类别均衡、两类错误同等重要 → ROC 也行。**

## 1.8 三个「两套算法」的坑（作业里专门问了）

作业里有个问题：「sklearn 里同一个指标有两种算法，差别在哪？」这是本作业设置的一个陷阱，必须先讲清楚。

**AUC-PR 的两种算法：**

| 写法 | 面积怎么算 | 特点 |
|---|---|---|
| `auc(recall, precision)` | **梯形法**：把相邻点用直线连起来，算梯形面积之和 | 曲线光滑时和下面几乎一样；**曲线退化成两点时会严重虚高** |
| `average_precision_score(y, score)` | **阶梯法**：$\sum_k (R_k - R_{k+1})P_k$ | 严格对应「取前 k 个时的 precision」的定义，是标准做法 |

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, average_precision_score, auc

rng = np.random.RandomState(1)
y = (rng.rand(3000) < 0.5).astype(int)

for name, score in [("全是同一个分数 0.5", np.full(3000, 0.5)),
                    ("随机分数", rng.rand(3000)),
                    ("有点用的分数", y * 0.4 + rng.rand(3000))]:
    prec, rec, thr = precision_recall_curve(y, score)
    print("%-20s 阈值个数 %4d | 梯形 %.4f | 阶梯 %.4f"
          % (name, len(thr), auc(rec, prec), average_precision_score(y, score)))

三行分别是三种情况，很能说明问题：

- **全是同一个分数**：只有 1 个阈值，曲线退化成 2 个点。`precision_recall_curve` 会自动补上一个「起点」$(recall=0, precision=1)$，于是梯形法在这两点之间画了一条斜线，面积 = $\frac{1 + \text{正类比例}}{2} \approx 0.75$。**这就是作业里那个 0.7528 的来源**（正类比例 0.5057，$(1+0.5057)/2 = 0.7528$，一模一样）。
- **随机分数**：阈值很多，两种算法几乎一样。
- **有点用的分数**：两种算法也几乎一样。

> 所以那个「虚高」的根源是**曲线退化**，不是「曲线抖动」。梯形的本质是「用直线连接相邻点」，当曲线只剩两点时，它就连出了一条根本没有数据支持的斜线。
>
> 结论：写报告一律用 `average_precision_score`。作业里的 `auc(recall, precision)` 只是照着模板给的写法，理解它可以，别用它比较模型。

**AUC-ROC 的两种算法**（作业没重点问，但顺手记住）：
- `roc_auc_score(y, score)`：基于排序，标准做法。
- `auc(fpr, tpr)`：梯形法。因为 ROC 曲线单调不减，两种结果几乎完全相同，这里没有坑。

## 1.9 交叉验证（Cross-Validation）

Part 3 和 Part 4 全靠它，必须懂。

问题：只切一次训练/测试，结果波动很大——万一这次切到的测试集特别简单呢？

做法：把训练集切成 K 份（作业用 5 份），轮流拿其中 1 份当「小测试集」（叫 validation），其余 K-1 份训练，最后把 K 次结果平均。这样每个样本都当过一次验证集，结果稳得多。

```
数据:  [1][2][3][4][5]
第1轮: 验证=1, 训练=2,3,4,5
第2轮: 验证=2, 训练=1,3,4,5
...
第5轮: 验证=5, 训练=1,2,3,4
=> 5 个分数，报告「平均值 ± 标准差」
```

关键参数：
- `StratifiedKFold`：**分层** K 折，保证每折里正类比例和数据整体一致。正类只有 11% 时必须用它，否则某一折可能几乎没有正类。
- `shuffle=True, random_state=500`：先打乱再切。作业 Part 4 指定了这两个值，是为了让结果可复现。
- `n_splits=5`：切 5 份。

> 「标准差」是什么：5 个分数之间的离散程度。平均值告诉你「大概多好」，标准差告诉你「这个数有多可信」。标准差大 = 结果不稳，别拿小数点后第三位去比较模型。

## 1.10 本篇会用到的 sklearn 三件套

后面代码里 `Pipeline`、`ColumnTransformer` 出现得非常频繁，先认脸。

```python
Pipeline([
    ("pre", 预处理),       # 比如编码、标准化
    ("lr",  模型),         # 比如 LogisticRegression
])
```
`Pipeline` 的作用：把「预处理 + 模型」焊成一个整体。好处是 `fit` 一次，预处理只在训练集上学统计量（比如均值、类别均值），永远不会偷看测试集——**这是防泄漏的机械保障**。

```python
ColumnTransformer([
    ("cat", OneHotEncoder(), 类别列名列表),   # 这几列这样处理
    ("num", "passthrough",   数值列名列表),   # 这几列原样保留
])
```
`ColumnTransformer` 的作用：对不同的列用不同的处理方式，然后把结果横向拼起来。`"passthrough"` 是「原样通过」的意思。

记住这三个名字，Part 2 之后的代码就都能读了：`Pipeline` 里第一个元素永远是预处理（`"pre"`），第二个永远是模型（`"lr"`）。

---
# 2. Part 1：SVM、逻辑回归、概率校准（2 分 + 0.5 加分）

## 📖 原文

> In this assignment you:
> - get familiar with what happens "inside" support vector machine and logistic regression
> - get familiar with probability calibration
> - learn variable transformation methods and feature selection methods
> - try to evaluate the economic effect of the model
>
> #### Deadline:
> - Soft deadline 1-Mart-2024
> - Hard deadline 8-Mart-2024

## 🇨🇳 翻译

这次作业你会：
- 了解支持向量机（SVM）和逻辑回归「内部」到底在做什么
- 了解**概率校准**（probability calibration）
- 学习**变量变换**方法和**特征选择**方法
- 尝试评估模型的**经济效果**

**截止日期（Deadline）：**
- 软截止（Soft deadline）：2024 年 3 月 1 日——过了还能交，可能扣分
- 硬截止（Hard deadline）：2024 年 3 月 8 日——过了就不收了

> 一句话总览：Part 1 讲「两种最基础的分类器长什么样、它们的输出能不能当概率用」；Part 2 讲「类别变量怎么变成数字」；Part 3 讲「特征太多怎么删」；Part 4 讲「模型到底能赚多少钱」。

## Task 1.1 · 原文

> Let's generate synthetic data.
>
> __Random classifier__：First, let's define the simplest model, which produces a random answer for each object. Using the test sample, we calculate AUC-ROC, AUC-PR.
>
> **Question:** We are solving a binary classification problem, but y_random contains some fractional numbers, not 0/1. Why?

## 🇨🇳 翻译

先造一批人造数据。

**随机分类器**：先定义最简单的模型——它对每个样本给出一个随机答案。用测试集计算 AUC-ROC 和 AUC-PR。

**问题**：我们做的是二分类问题，但 `y_random` 里是一堆小数而不是 0/1，为什么？

> *Below is an **example** of working* with the built-in `sklearn` functions for drawing ROC and PR curves and saving metrics. The pipeline can be changed as you wish.
>
> **翻译**：*下面是一个使用 `sklearn` 内置函数画 ROC、PR 曲线并记录指标的**示例**。这套流程你可以随意改动。*
>
> 也就是说作业提供的 `depict_pr_roc` 只是「参考实现」，不是标准答案——你可以自己组织代码，只要最后有曲线和指标。

### 🧠 先补概念：人造数据 `make_classification` 的每个参数

作业第一句就是造数据，这个函数的参数必须看懂，否则后面所有数字都没有意义。

```python
X, y = make_classification(
    n_samples=10000,     # 造 10000 个样本
    n_features=10,       # 每个样本 10 个特征
    n_informative=5,     # 其中 5 个是「真正有用」的
    n_redundant=5,       # 另外 5 个是「有用的那 5 个的随机线性组合」（冗余，不带新信息）
    random_state=42)     # 随机种子：固定它，每次跑出来数据一模一样
```

关键点：
- `n_informative` 是有信息量的特征，模型学到它们就能分对。
- `n_redundant` 是**人为制造的共线性**：第 6 个特征可能是 `0.3*x1 - 0.7*x2`。它看起来像新特征，其实是老特征的换装。这正是后面 Part 3 讨论「相关特征会分散权重」的伏笔。
- `random_state` 就是「随机种子」。计算机的随机是伪随机：种子相同，结果就完全相同。**作业里凡是出现 `random_state` 的地方，都是为了让你我的结果能对上。**

In [ ]:
from sklearn.datasets import make_classification
import numpy as np

X, y = make_classification(n_samples=10000, n_features=10, n_informative=5,
                           n_redundant=5, random_state=42)
print("X 的形状:", X.shape, "-> 10000 个样本，每个 10 维")
print("y 的形状:", y.shape, "-> 每个样本一个标签")
print("y 的取值:", np.unique(y))
print("正类比例:", y.mean())
print("前 3 个样本的前 4 个特征:\n", np.round(X[:3, :4], 2))

### 🧠 为什么 `y_random` 是小数（这是作业的第一个问答题）

先看 `DummyClassifier` 是什么：它是一个**什么都不学**的「假模型」，只用来做基准线（baseline）。如果我们的真模型连它都打不过，说明模型完全是废的。

`strategy='uniform'` 的意思是：**模型对类别没有任何信息，就按均匀先验给出「每个类 1/K 的概率」**。

- `predict` → 在两类之间**随机抽**一个，得到 0 或 1。
- `predict_proba` → 返回**常数 0.5 / 0.5**（二分类时 $1/2$），因为「均匀」意味着两类一样可能。

⚠️ 这里有个很多人误解的点：`predict` 是随机的，但 `predict_proba` **不是随机数，而是恒定值 0.5**。`y_random` 里的那些小数，是「概率」，不是「随机抽样结果」。所以它既不是 0/1（那是标签），也不是 `[0,1]` 上乱跳的随机数（那是随机分数），而是**一个恒等于 0.5 的常数**——这个细节在 1.8 节解释「为什么梯形 AUC-PR 会变成 0.75」时是关键。

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

dummy = DummyClassifier(strategy='uniform', random_state=42).fit(X_train, y_train)

print("predict        给出硬标签 :", dummy.predict(X_test)[:8])
print("predict_proba  给出概率   :\n", np.round(dummy.predict_proba(X_test)[:4], 3))
print("predict_proba[:, 1] 就是正类那一列:", np.round(dummy.predict_proba(X_test)[:8, 1], 3))
print("\npredict 给的是硬标签（0/1），它的「随机」由 random_state 控制；")
print("而 predict_proba 是恒定的 0.5，完全与 random_state 无关：")
print("第二次 predict:", dummy.predict(X_test)[:8])

> **答案（写进作业的话）**：`predict_proba` 返回的是**每个类的概率估计**（一个 `(n, 2)` 的数组），而不是最终标签。`DummyClassifier(strategy='uniform')` 没有任何信息，于是报告均匀先验：两个类各 0.5，所以 `[:, 1]` 这一列全是 0.5 这样的小数。要 0/1 请用 `predict`；算 ROC/PR 曲线必须用 `predict_proba`，因为曲线需要的是**分数**，不是决策。

### 📄 修正版答案格全文翻译（solved 版「y_random 为什么是小数」Answer）

> `predict_proba` returns a **probability vector**, not a label: for every object it reports $[p(y=0),\,p(y=1)]$.
>
> `DummyClassifier(strategy='uniform')` has no information at all, so it reports the **uniform prior** — $1/K$ for each class, i.e. 0.5 for both classes, identically for every object. That is why `y_random` is a constant column of 0.5 rather than 0/1 values.

**翻译**：`predict_proba` 返回的是**概率向量**，不是标签：对每个样本它报告 $[p(y=0),\,p(y=1)]$。`DummyClassifier(strategy='uniform')` 什么信息都没有，所以它报告**均匀先验**——每个类 $1/K$，也就是两个类各 0.5，对每个样本都一样。这就是为什么 `y_random` 是一列恒定的 0.5，而不是 0/1。

> Note the distinction, it matters later: `predict` **does** produce random labels (it samples a class for each object), but `predict_proba` is deterministic and constant. So "random classifier" here means "a classifier that knows nothing", not "a classifier that outputs random numbers".
>
> The distinction `predict` vs `predict_proba` is also the reason ROC/PR curves are built from the latter: a curve is traced by moving the threshold, and you cannot move a threshold through a 0/1 label.

**翻译**：注意这个区别，后面很重要：`predict` **确实**产生随机标签（它为每个样本随机抽一个类），但 `predict_proba` 是确定性的常数。所以这里的「随机分类器」意思是「一个什么都不知道的分类器」，而不是「输出随机数的分类器」。`predict` 与 `predict_proba` 的区别也是 ROC/PR 曲线必须用后者的原因：曲线是靠**移动阈值**扫出来的，而 0/1 标签上没有阈值可以移动。

### 🧠 为什么 ROC / PR 必须用分数

因为整条曲线是「**把阈值从 1 慢慢降到 0**」画出来的。如果只有 0/1 两种值，阈值只可能切出一个点，没有曲线可言。

## 工具函数 `depict_pr_roc` 逐行讲解

作业里给了一个画图 + 算指标的模板函数，长得像这样：

```python
def depict_pr_roc(y_true, y_pred, classifier_name='Some Classifier', ax=None):
  if ax is None:
    fig, ax = plt.subplots(1, 2, figsize=(11, 5))       # 一行两列的画布

  print(classifier_name, 'metrics')
  PrecisionRecallDisplay.from_predictions(y_true, y_pred, ax=ax[0], name=classifier_name)
  precision, recall, _ = precision_recall_curve(y_true, y_pred)
  print('AUC-PR: %.5f' % auc(recall, precision))
  ax[0].set_title("PRC")
  ax[0].set_ylim(0, 1.1)

  RocCurveDisplay.from_predictions(y_true, y_pred, ax=ax[1], name=classifier_name)
  print('AUC-ROC: %.5f' % roc_auc_score(y_true, y_pred))
  ax[1].set_title("ROC")
  ax[1].set_ylim(0, 1.1)
```

| 名字 | 作用 |
|---|---|
| `ax` | 画布的「子图」对象。`ax[0]` 是左边那张，`ax[1]` 是右边那张 |
| `PrecisionRecallDisplay.from_predictions` | sklearn 现成的画 PR 曲线工具，一行搞定 |
| `RocCurveDisplay.from_predictions` | 同上，画 ROC |
| `precision_recall_curve(y, score)` | 返回三个数组 `(precision, recall, thresholds)` |
| `auc(recall, precision)` | 算 PR 曲线下面积（梯形法，见 1.8 的坑） |
| `roc_auc_score(y, score)` | 直接算 ROC 的 AUC |
| `%.5f` | 保留 5 位小数的格式化占位符 |

⚠️ **易错点**：`precision_recall_curve` 返回的顺序是 **(precision, recall)**，但 `auc()` 的惯例是横轴在前。横轴是 recall，所以必须写成 `auc(recall, precision)`。写反了数不变，但语义错；更危险的是 `roc_curve` 返回 `(fpr, tpr)`，写 `auc(tpr, fpr)` 就会得到 1 减去正确答案——这是历年作业最常见的错。

## 📄 修正版补充说明格全文翻译：「两条曲线到底在量什么」

> 修正版 solved notebook 里紧跟着随机分类器有一整格英文说明（标题 *What the two curves actually measure*），这里给出逐句翻译。

> Before comparing anything, fix the definitions (TP = predicted 1 and truly 1):
>
> $$\text{precision} = \frac{TP}{TP+FP}, \qquad \text{recall (TPR)} = \frac{TP}{TP+FN}, \qquad FPR = \frac{FP}{FP+TN}$$

**翻译**：在比较任何东西之前，先固定定义（TP = 预测为 1 且真为 1）：precision（精确率）= 预测为正的里面真的为正的比例；recall（召回率，也叫 TPR）= 真正例里面被找出来的比例；FPR（假正率）= 真负例里面被冤枉的比例。

> * A classifier with `predict_proba` does not give a decision, it gives a **ranking**. Moving the threshold from 1 down to 0 traces a whole family of confusion matrices.

**翻译**：一个有 `predict_proba` 的分类器给出的不是「决策」，而是一个**排序**。把阈值从 1 一路降到 0，就扫出一整族混淆矩阵（每个阈值对应一张）。

> * The **ROC** curve plots TPR against FPR over all thresholds; its area (AUC-ROC) is the probability that a random positive object gets a higher score than a random negative one.

**翻译**：**ROC** 曲线是在所有阈值下「TPR 对 FPR」的图；它的面积（AUC-ROC）有一个漂亮的概率含义：**随机抓一个正样本，它的分数比随机抓的一个负样本高的概率**。

> * The **PR** curve plots precision against recall over all thresholds. Its area (AUC-PR) answers a different question: "if I call the top-k objects, what share of them will be positive?" — precisely the business question of Part 4.

**翻译**：**PR** 曲线是在所有阈值下「precision 对 recall」的图。它的面积（AUC-PR）回答的是另一个问题：「如果我打电话给分数最高的前 k 个人，其中有多大比例会成交？」——这正是 Part 4 的业务问题。

> That is why for a rare positive class the PR curve is the honest one: FPR is divided by TN, and with 90% negatives TN is huge, so FPR stays tiny and the ROC curve looks beautiful no matter what.

**翻译**：这就是为什么正类稀少时 PR 曲线才是「诚实」的那个：FPR 的分母是 TN（真负例数），当 90% 都是负样本时 TN 巨大，于是 FPR 怎么算都很小，ROC 曲线不管模型多差都很好看。

## SVM 任务原文（solved 版三个连续格）+ 翻译

> __Support Vector Machine (Linear Kernel)__
>
> Train support vector machines. In the classroom, we went through the linear version without a kernel transition, which corresponds to the linear kernel (`LinearSVC` or `SVC(kernel='linear')` in `sklearn.svm`).
>
> Choose the regularization parameter `C` (you can use cross-validation or separate the validation sample from the training one).

**翻译**：**支持向量机（线性核）**。训练一个 SVM。课堂上我们讲的是不做核变换的线性版本，对应 `sklearn.svm` 里的线性核（`LinearSVC` 或 `SVC(kernel='linear')`）。选择正则化参数 `C`（可以用交叉验证，也可以从训练集里再切一块验证集）。

> On the test part,
> * calculate AUC-ROC, AUC-PR with an accuracy of 5 digits.
> * Plot ROC and PR curves

**翻译**：在测试集上：① 计算 AUC-ROC 和 AUC-PR，精确到小数点后 5 位；② 画出 ROC 和 PR 曲线。

> Compare them with the responses of a random classifier.
> Note that with `sklearn` you can calculate both metrics in two different ways - what are their differences? Compare results.

**翻译**：把结果和随机分类器比较。注意：用 `sklearn` 算这两个指标各有**两种不同的算法**——它们的区别是什么？把两种结果都比一比。（这正是下面「两种 AUC-PR 的算法」一节回答的问题。）

> 关键词汇：**kernel（核）** 是把数据映射到高维空间的技巧，「线性核」就是「不映射、直接在原空间找直线」；**validation sample（验证集）** 是从训练集里再切出来、专门用来挑超参数的一小块。

## SVM 是什么（先建立直觉，再看代码）

### 一句话

**在所有能把两类点分开的直线里，SVM 挑「离两边的点都尽量远」的那一条。**

> 类比：在两国之间划一条边界线。你可以贴着 A 国的哨兵划，也可以贴着 B 国的哨兵划，但那都不安全。最安全的划法是**正中**——两侧留出的空地（margin，间隔）一样宽。SVM 就是干这个的。

### 三个关键词

1. **分隔超平面**：$w\cdot x + b = 0$。二维是一条直线，十维是一个「平面」。
2. **间隔（margin）**：两条虚线 $w\cdot x+b=\pm 1$ 之间的距离，等于 $\frac{2}{\lVert w\rVert}$。SVM 想让这个距离尽可能大，也就是**让 $\lVert w\rVert$ 尽可能小**。
3. **支持向量（support vectors）**：刚好落在虚线上的那些点。**只有它们决定边界**，其他点删掉都不影响结果。

### 那「分错了怎么办」——hinge loss

现实数据不可能完全分开，所以 SVM 允许点越界，但要罚分。罚分函数叫 **hinge loss（合页损失）**：

$$\ell(z) = \max(0, 1 - z), \qquad z = y\cdot(w\cdot x + b)$$

读法：
- 当 $z \ge 1$（点不仅在正确一侧，而且已经在间隔外）→ 罚 0。**注意：罚 0，不再关心它有多远。**
- 当 $z < 1$（越界了或分错了）→ 按差距线性罚。

最后这句话是本作业 Part 1 **第二个问答题的答案根源**，一定要记住：**hinge loss 对「已经很对」的点完全不奖不罚**，所以 SVM 没有动力把分数推得更高。这就是它输出不了「概率」的原因。

### `C` 在这里是什么

$$\min_{w,b} \ \sum_i \max(0, 1 - y_i(w\cdot x_i + b)) + \frac{1}{2C}\lVert w\rVert^2$$

- `C` 小 → $\frac{1}{2C}$ 大 → 拼命压小 $\lVert w\rVert$ → 间隔宽、允许更多点越界（**宽松、稳健**）
- `C` 大 → 罚分优先 → 间隔窄、尽量不越界（**严格、容易过拟合**）

## 逻辑回归任务原文（solved 版两个连续格）+ 翻译

> __Logistic Regression__
>
> A similar task for logistic regression with L2 regularizer:
> * fit hyperparameter C using AUC-PR metric
> * draw ROC, PR curves for the test part
> * output metrics for test data and compare them with the results of a random classifier

**翻译**：**逻辑回归**。对带 L2 正则化的逻辑回归做同样的事：① 用 AUC-PR 作为指标调超参数 `C`；② 画出测试集上的 ROC、PR 曲线；③ 输出测试集指标，并与随机分类器的结果比较。

> Draw ROC, PR curves for test part for all 3 classifiers on one graph

**翻译**：把全部 3 个分类器（随机、SVM、逻辑回归）的 ROC 曲线画在同一张图上、PR 曲线画在同一张图上，方便直接对比。

> **Question:** Compare the results of LR and SVM in terms of all calculated quality criteria, explain the differences (if any).

**翻译**：**问题**：按所有算过的质量指标比较 LR 和 SVM 的结果，解释差异（如果有的话）。

> 关键词汇：**L2 regularizer（L2 正则化）** 就是在损失函数里加一项 $\frac{1}{2C}\lVert w\rVert^2$，惩罚过大的权重，防止过拟合（见 1.5 节）。

## 逻辑回归是什么

### 一句话

**逻辑回归不画线，它给每个点算一个「是正类的概率」，然后用 0.5 当分界线。**

模型形式：

$$p(y=+1\mid x) = \sigma(w\cdot x + b), \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

`σ` 叫 **sigmoid 函数**，形状是 S 形：$z$ 很大 → 输出接近 1；$z$ 很小 → 接近 0；$z=0$ → 正好 0.5。

### 损失函数：log loss（交叉熵）

$$\ell(z) = \log(1 + e^{-z})$$

和 hinge 的最大区别：**log loss 永远不为 0**。一个点即使已经分对了，只要它离边界还不够远，就仍然有罚分。所以逻辑回归会**持续把预测往 0 和 1 两端推**——这就是它天生输出「像概率的东西」的原因。

In [ ]:
# 直观对比：hinge 与 log loss
import numpy as np

z = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 5.0])   # z = y*(wx+b)，越大表示越「正确且自信」
hinge = np.maximum(0, 1 - z)
logloss = np.log1p(np.exp(-z))

print(" z      hinge     logloss")
for zi, h, l in zip(z, hinge, logloss):
    print("%5.1f  %7.3f  %10.3f" % (zi, h, l))

看最后一行：`z = 5`（非常自信地分对了）时，hinge 的惩罚是 **0**（完全不管了），log loss 还有 **0.0067**（还在推）。这个差别虽小，却决定了「输出是不是概率」。

## 选 `C`：为什么要用 `GridSearchCV`

作业要求「Choose the regularization parameter `C`（可以用交叉验证）」。做法是：把 C 的一系列候选值都试一遍，用交叉验证打分，选最好的那个。

```python
C_grid = np.logspace(-3, 2, 8)      # [0.001, 0.0037, ..., 100]，对数网格
svm_search = GridSearchCV(
    LinearSVC(max_iter=20000),
    param_grid={"C": C_grid},       # 要试的参数和候选值
    scoring="average_precision",    # 用什么指标判定谁好（这里用 AUC-PR）
    cv=3,                           # 3 折交叉验证
)
svm_search.fit(X_train, y_train)
svm = svm_search.best_estimator_    # 拿到「最好那个 C」训练好的模型
print(svm_search.best_params_)      # {'C': 3.73}
print(svm_search.best_score_)       # 交叉验证上的平均分
```

| 名字 | 作用 |
|---|---|
| `np.logspace(-3, 2, 8)` | 在 $10^{-3}$ 到 $10^{2}$ 之间取 8 个**等比**的数。为什么等比？因为 C 的影响是乘性的，0.001→0.01 和 1→10 效果类似 |
| `param_grid` | 字典，键是参数名，值是候选列表 |
| `scoring` | 评分函数名。`"average_precision"` 就是 AUC-PR |
| `cv=3` | 3 折交叉验证 |
| `best_estimator_` | 用最优参数、在**全部训练集**上重新训练好的模型 |
| 结尾的下划线 `_` | sklearn 的约定：**带下划线的属性是「fit 之后才存在」的**。这就是为什么 `best_params_` 只能在 `fit` 之后读 |

⚠️ `GridSearchCV` 内部只在 `X_train` 上做交叉验证，`X_test` 完全没参与——这是正确的做法。如果你把 `X`（含测试集）喂给它，就已经泄漏了。

## 两种 AUC-PR 的算法（作业的第二个问答题）

代码里同时出现了这两行：

```python
prec, rec, _ = precision_recall_curve(y_test, y_svm_score)
trapezoid = auc(rec, prec)                                # 梯形法
step_area = average_precision_score(y_test, y_svm_score)  # 阶梯法（标准）
```

| 方法 | 结果（本次运行，线性 SVM） |
|---|---|
| `auc(recall, precision)` 梯形法 | 0.82969 |
| `average_precision_score` 阶梯法 | 0.82995 |

模型曲线光滑时，两者差 0.0003，可以忽略。但换成**「随机分类器」**时：

| 方法 | 结果 |
|---|---|
| `auc(recall, precision)` | 0.75283 |
| `average_precision_score` | 0.50566 |

差到 0.25！原因不是「随机模型的曲线抖」，而是**它的分数是个常数 0.5**（见上一节）：

1. 所有样本分数相同 → `precision_recall_curve` 只找到 **1 个阈值**，输出 2 个点：$(recall=1,\ precision=0.5057)$ 和自动补上的起点 $(recall=0,\ precision=1)$。
2. 梯形法把这两个点**用一条直线连起来**，面积 = $\frac{1+0.5057}{2} = 0.7528$。
3. 而 `average_precision_score` 按定义只累加「真实的 recall 增量」，得到的正好是正类比例 0.5057 —— 这才是「抛硬币」的正确分数。

> 这就是为什么「随机分类器的 AUC-PR 是 0.75」这种说法是错的——它只是梯形法在退化曲线上画出来的一条假斜线。

> **答案（写进作业）**：两种写法分别是**梯形积分**和**阶梯积分**。前者把相邻点用直线连起来算面积；后者严格实现「precision@k」的定义、按 recall 的增量加权。当曲线退化成很少几个点（例如分数全相同时只有 2 个点）时，梯形法会凭空造出一条斜线，把面积从正确的 0.506 抬到 0.753。**报告里一律用 `average_precision_score`。**

### 📄 修正版答案格全文翻译（solved 版「The two ways to compute AUC-PR」）

> | function | how the area is computed |
> |---|---|
> | `auc(recall, precision)` | **trapezoid** rule over the points returned by `precision_recall_curve` — it connects neighbouring points with straight lines |
> | `average_precision_score(y, score)` | **step** (right-continuous) approximation $\sum_k (R_k - R_{k+1})\,P_k$ — it reproduces exactly the "precision of the top-k" definition |

**翻译（表格）**：

| 函数 | 面积是怎么算的 |
|---|---|
| `auc(recall, precision)` | 对 `precision_recall_curve` 返回的点用**梯形**法则——把相邻的点用直线连起来 |
| `average_precision_score(y, score)` | **阶梯**（右连续）近似 $\sum_k (R_k - R_{k+1})\,P_k$——它精确复现了「前 k 名的 precision」这一定义 |

> Here they differ only by ~3e-4 — the curve is smooth. But on the **random classifier** they differ by 0.25, and the reason is worth understanding: its score is the constant 0.5 (see Task 1.1's first question), so `precision_recall_curve` finds a single threshold and returns only two points — the real one $(recall=1,\ precision=0.50567)$ and the conventional anchor $(recall=0,\ precision=1)$. The trapezoid rule joins them with a straight line and reports $\frac{1+0.50567}{2}=0.75283$, while the step rule honestly reports the base rate $0.50567$.

**翻译**：在这个（真实模型的）曲线上，两种算法只差约 3e-4——因为曲线光滑。但在**随机分类器**上两者差 0.25，原因值得理解：它的分数是常数 0.5（见 Task 1.1 的第一个问题），所以 `precision_recall_curve` 只找到**一个**阈值，只返回两个点——真实的那点 $(recall=1, precision=0.50567)$ 和按惯例补的锚点 $(recall=0, precision=1)$。梯形法把两点用直线相连，报出 $\frac{1+0.50567}{2}=0.75283$；而阶梯法诚实地报出基准率 0.50567。

> So the inflation is not caused by "wiggly curves": it is caused by a **degenerate curve** being interpolated. `average_precision_score` is the standard; `auc(recall, precision)` is acceptable only when the curve has enough points and is smooth.

**翻译**：所以虚高不是「曲线抖动」造成的，而是一条**退化曲线被插值**造成的。`average_precision_score` 才是标准做法；`auc(recall, precision)` 只在曲线有足够多点且光滑时勉强可用。

## 问答：LR 和 SVM 的结果比较

**本次运行结果（测试集 3000 个样本）：**

| 模型 | AUC-PR（梯形） | AUC-PR（阶梯） | AUC-ROC |
|---|---|---|---|
| 随机分类器 | 0.75283 | 0.50566 | 0.50000 |
| 线性 SVM（C = 3.73） | 0.82969 | 0.82995 | 0.84764 |
| 逻辑回归（C = 0.001） | 0.83360 | 0.83411 | 0.84533 |

**怎么读这张表：**

1. **两者几乎打平**。差距 0.004 左右，完全在一次随机切分的噪声范围内。这不奇怪：两者都在找**线性**边界、用的是同一批特征，只是损失函数不同（hinge vs log loss）。当数据只有中等可分（AUC≈0.85）时，绝大部分排序能力来自数据本身，而不是损失函数的选择。
2. **真正的差别在输出上**：LR 有 `predict_proba`，数字可以直接当概率读；`LinearSVC` 只有 `decision_function`，是「到分隔面的距离」，没有概率含义。这正是 Part 1 后半段（校准）要解决的问题。
3. **哪些样本有用**：LR 用到全部样本；SVM 只保留支持向量（这里是 53 个），所以预测更快、对远处的离群点更不敏感。
4. **C 不能跨模型比较**：LR 选了 0.001、SVM 选了 3.73，看起来差 4 个数量级，但两种损失的尺度不同，这两个数**不可比**。

> 表里那个 0.75283 vs 0.50566 的对比就是作业埋的坑，见上一节。

### 📄 修正版答案格全文翻译（solved 版 LR vs SVM 长答案）

> **1. LR and SVM are practically tied.** Measured on the test set (3,000 objects):

**翻译**：**1. LR 和 SVM 几乎打平。** 在测试集（3000 个样本）上测得：

（结果表格与上面「问答：LR 和 SVM 的结果比较」一节中的中文表格完全相同：随机分类器 0.75283/0.50567/0.50000，SVM 0.82969/0.82995/0.84764，LR 0.83360/0.83376/0.84533。）

> The gap is ~0.004 in AUC-PR and ~0.002 in AUC-ROC — far inside the noise of a single 3,000-object split. This is expected: both methods fit a **linear** decision boundary on the same features, they just use different loss functions (hinge vs logistic). When the data is only moderately separable (AUC ≈ 0.85) most of the ordering of objects is determined by the data, not by the loss.

**翻译**：AUC-PR 差约 0.004、AUC-ROC 差约 0.002——远小于一次 3000 样本切分的噪声。这是预料之中的：两种方法在同一批特征上拟合的都是**线性**决策边界，只是损失函数不同（hinge 对 logistic）。当数据只有中等可分（AUC ≈ 0.85）时，样本排序的绝大部分由**数据本身**决定，而不是由损失函数决定。

> **2. Where they genuinely differ:**
> * **Outputs.** LR returns `predict_proba` — numbers that can be read as probabilities. SVM (LinearSVC) returns only `decision_function` — a signed distance to the hyperplane, which has no probabilistic meaning. Part 2 of this notebook is exactly about that gap.
> * **Which objects matter.** LR uses every object; SVM keeps only the support vectors (a few dozen here). Hence SVM is cheaper at prediction time and more robust to far-away outliers.
> * **Behaviour of C.** LR picked C = 0.001 (strong regularization) and SVM picked C = 3.73 — for the *same* penalty strength the two losses are scaled differently, so the numbers of C are not comparable between the models.

**翻译**：**2. 它们真正不同的地方：**
- **输出**：LR 返回 `predict_proba`——可以当概率读的数字；SVM（LinearSVC）只返回 `decision_function`——到超平面的带符号距离，没有概率含义。本作业 Part 1 后半（校准）正是在补这个缺口。
- **哪些样本起作用**：LR 用到每一个样本；SVM 只保留支持向量（这里只有几十个）。所以 SVM 预测时更快，对远处的离群点也更不敏感。
- **C 的行为**：LR 选了 C = 0.001（强正则化），SVM 选了 C = 3.73——即使惩罚强度相同，两种损失的量纲也不同，所以两个模型的 C 数值**不能直接比较**。

> **3. The trap in the table.** The random classifier shows AUC-PR 0.75283 with `auc(recall, precision)` but only 0.50567 with `average_precision_score` — and 0.50567 is exactly the base rate, i.e. the honest answer for a coin flip.
>
> The mechanism is worth spelling out: the score of this "classifier" is the constant 0.5, so there is only **one** threshold, and `precision_recall_curve` returns just two points: $(recall=1,\ precision=0.50567)$ and the anchor $(recall=0,\ precision=1)$. The trapezoid rule draws a straight line between them and reports $\frac{1+0.50567}{2}=0.75283$. Nothing is measured there — it is an interpolation artefact. This is the "two ways to compute the metric" the task asks about: for PR curves always prefer `average_precision_score`.

**翻译**：**3. 表里的陷阱**：随机分类器用 `auc(recall, precision)` 算出 AUC-PR = 0.75283，但用 `average_precision_score` 只有 0.50567——而后者正好等于正类基准率，是「抛硬币」的诚实分数。机制值得讲透：这个「分类器」的分数是常数 0.5，所以只有**一个**阈值，`precision_recall_curve` 只返回两点：$(1,\ 0.50567)$ 和锚点 $(0,\ 1)$。梯形法在两点间画一条直线，报出 0.75283。那里**什么都没测到**——纯属插值假象。这就是作业问的「两种算法的区别」：对 PR 曲线永远优先用 `average_precision_score`。

## Task 1.2 · 原文

> There are some "support vectors" in the name of the support vector machine.
> * Generate a synthetic dataset using `make_classification` with 2 features, train support vector machines on it. This time, if necessary, use `SVC(kernel='linear')` instead of `LinearSVC`, as it is the only one with support vector information.
> * Visualize the dividing line, all objects, and select the support vectors (the `support_vectors_` attribute).

## 🇨🇳 翻译

支持向量机的名字里就带着「支持向量」。
- 用 `make_classification` 造一个**只有 2 个特征**的数据集，在上面训练 SVM。这次如果必要，请用 `SVC(kernel='linear')` 而不是 `LinearSVC`，因为只有它才有支持向量信息。
- 可视化分隔线、所有样本点，并把支持向量标出来（用 `support_vectors_` 属性）。

> 为什么强调 2 个特征？因为人只能看二维图。10 维的边界画不出来。

### 🧠 `LinearSVC` 和 `SVC(kernel='linear')` 有什么区别

两者数学上干的是同一件事（线性 SVM），但实现不同：

| | `LinearSVC` | `SVC(kernel='linear')` |
|---|---|---|
| 底层算法 | liblinear（对偶坐标下降） | libsvm（核方法） |
| 有 `support_vectors_` 吗 | **没有**，只给 `coef_` | **有** |
| 速度 | 快，适合大样本 | 慢一些 |
| 输出 | 只有 `decision_function` | 有 `decision_function`，还可以 `probability=True`（内部用 Platt） |

所以「要画支持向量」就必须换成 `SVC`。这是作业特意点名的地方。

### 🧠 画图函数里的每一行（`plot_svm_2D`）

```python
def plot_svm_2D(X, y, model, plot_support=True):
    # ① 在数据范围内造一张网格
    xx = np.linspace(X[:, 0].min(), X[:, 0].max(), 30)   # 横轴方向 30 个点
    yy = np.linspace(X[:, 1].min(), X[:, 1].max(), 30)   # 纵轴方向 30 个点
    YY, XX = np.meshgrid(yy, xx)                         # 变成 30x30 的网格坐标
    xy = np.vstack([XX.ravel(), YY.ravel()]).T           # 摊平成 900 个 (x1, x2) 点

    # ② 让模型对这 900 个点打分
    Z = model.decision_function(xy).reshape(XX.shape)    # 再折回 30x30

    # ③ 只画三条等值线：-1、0、+1
    plt.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1], alpha=0.5,
                linestyles=['--', '-', '--'])

    # ④ 画所有样本点
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, s=25, edgecolor="k")

    # ⑤ 画支持向量（空心大圈）
    if plot_support:
        plt.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                    label='support vectors', s=100, linewidth=1,
                    edgecolor="blue", facecolors='none')
    plt.legend()
```

逐项解释，这些名字你以后会一直遇到：

| 名字 | 作用 |
|---|---|
| `np.linspace(a, b, n)` | 在 `[a,b]` 上均匀取 `n` 个点 |
| `np.meshgrid(yy, xx)` | 把两个一维坐标变成二维网格。返回 `YY, XX`（注意顺序被写反了，这是为了配合 `contour`） |
| `.ravel()` | 把 30×30 的矩阵压成一维长度 900 的向量 |
| `np.vstack([...]).T` | 竖着摞起来再转置 → 变成 `(900, 2)` 的「900 个二维点」 |
| `decision_function(xy)` | 对每个点算 $w\cdot x+b$，返回 900 个实数 |
| `.reshape(XX.shape)` | 折回 30×30，这样才能当「高度图」用 |
| `levels=[-1, 0, 1]` | **关键**：只画三条等高线。0 是分类边界，±1 是间隔边界（因为 SVM 的约束就是 $w\cdot x+b=\pm 1$） |
| `linestyles=['--','-','--']` | 三条线的样式：虚线、实线、虚线 |
| `cmap=plt.cm.coolwarm` | 配色方案，`c=y` 表示按标签上色 |
| `facecolors='none'` | 空心点，这样不会遮住底下的样本点 |
| `support_vectors_` | 形状 `(n_sv, n_features)`，只有 `SVC` 才有 |

> **为什么 `levels` 恰好是 ±1**：这不是随便取的。SVM 的标准化形式是 $y_i(w\cdot x_i+b)\ge 1$，也就是说「间隔边界」定义为 $w\cdot x+b=\pm 1$。训练完，支持向量就正好落在这些线上。所以画 ±1 就是在画「哪些点算支持向量」的分界。

In [ ]:
# 亲手看一眼：支持向量到底是不是「刚好在虚线上」
from sklearn.svm import SVC
from sklearn.datasets import make_classification

X2, y2 = make_classification(n_samples=400, n_features=2, n_informative=2,
                             n_redundant=0, n_clusters_per_class=1,
                             class_sep=1.6, flip_y=0.05, random_state=0)
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.3, random_state=42)

m = SVC(kernel="linear", C=1.0).fit(X2_tr, y2_tr)
sv = m.support_vectors_
print("训练样本数:", len(X2_tr), " 支持向量数:", len(sv))

# support_ 是「下标列表」，不是布尔掩码 —— 必须先转成掩码，否则 ~ 会变成负数下标
mask_sv = np.zeros(len(y2_tr), dtype=bool)
mask_sv[m.support_] = True

# 间隔的定义里带 y 的符号（y 必须是 +1/-1），否则负类会全部算成 0
signed = np.where(y2_tr == 1, 1, -1)
margins = signed * m.decision_function(X2_tr)

print("支持向量   的间隔: %.3f ~ %.7f" % (margins[mask_sv].min(), margins[mask_sv].max()))
print("非支持向量 的间隔: %.3f ~ %.3f" % (margins[~mask_sv].min(), margins[~mask_sv].max()))

输出是教科书般的干净结果：

```
支持向量   的间隔: -2.109 ~ 1.000
非支持向量 的间隔:  1.004 ~ 7.711
```

两组数**恰好被 1 这条线分开**。支持向量里最差的那个是 −2.109（完全分错了，它是被 hinge loss 罚出来的），非支持向量里最小的也有 1.004（刚好在间隔外侧）。这就是「支持向量 = 决定边界的那些点」的严格含义。

> ⚠️ 这里顺手演示了**两个**经典坑：
> 1. `m.support_` 返回的是**下标数组**（比如 `[3, 7, 11]`），不是布尔掩码。直接写 `~m.support_` 会得到 `[-4, -8, -12]`——Python 会把它当成**负索引**（从末尾倒数），悄悄取错行，而且不报错。
> 2. 间隔是 $y\cdot(w\cdot x+b)$，**$y$ 必须是 $+1/-1$**。如果 `y` 是 0/1，负类的间隔就永远算成 0，看起来像「一堆点都落在边界上」。

> **答案（作业问答题）**：支持向量是满足 $|w\cdot x+b|\le 1$ 的训练样本——即落在间隔上、落在间隔内、或分错了的那批点。只有它们出现在损失函数的「活跃」部分（hinge 不为 0），因此也只有它们决定最终的分隔面；把其余样本删掉重训，边界不变。

### 📄 修正版问答格全文翻译（solved 版「what objects are identified as support vectors?」）

> Support vectors are the training objects that lie **on the margin, inside it, or on the wrong side** of the boundary — i.e. those satisfying $|w\cdot x + b| \le 1$. On the plot they are the circled points sitting exactly on the two dashed lines.

**翻译**：支持向量是**落在间隔上、间隔内、或边界错误一侧**的训练样本——也就是满足 $|w\cdot x + b| \le 1$ 的那些点。在图上，它们就是恰好坐在两条虚线上的、被圈出来的点。

> **Why only them?** The SVM solution depends only on the closest points to the boundary: delete every other object and the same separating line comes out. That is the sparsity of the SVM — the model keeps a handful of critical objects instead of the whole sample.
>
> **Intuition:** the border between two armies is decided by the sentries on the front line, not by the ten thousand soldiers behind them.

**翻译**：**为什么只有它们？** SVM 的解只依赖离边界最近的那些点：把其余样本全部删掉，重新训练会得到**同一条**分隔线。这就是 SVM 的**稀疏性**——模型只保留一小撮关键样本，而不是整个训练集。**直觉**：两国边界是由站在前线的哨兵决定的，而不是后方的一万大军。

## 逻辑回归的「概率面」

## 📖 原文

> Unlike the support vector machine, logistic regression does not try to build a separating hyperplane, but approximates the likelihood of positive answers $p(y=+1|x)$ at each point in the object space. Try to draw this distribution on a plane, do not forget to mark all the objects on it. The `plt.pcolor` function can help you.

## 🇨🇳 翻译

与 SVM 不同，逻辑回归不试图画一条分隔超平面，而是在样本空间的**每一个点**上估计「回答为正」的概率 $p(y=+1\mid x)$。请把这个分布在平面上画出来，别忘了把所有样本点也标上去。`plt.pcolor` 函数可以帮到你。（解答里用了效果一样的 `plt.imshow`，两者都是「把二维数组画成彩色图」。）

作业要求：把逻辑回归对平面上每个点给出的 $p(y=+1\mid x)$ 画出来。

```python
def plot_logreg_2D(X, y, model):
    xx = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
    yy = np.linspace(X[:, 1].min(), X[:, 1].max(), 100)
    YY, XX = np.meshgrid(yy, xx)
    xy = np.vstack([XX.ravel(), YY.ravel()]).T

    Z = model.predict_proba(xy)[:, 1]              # 每个网格点的正类概率
    Z = Z.reshape((xx.shape[0], -1)).T             # 折成 100x100 的「图片」

    image = plt.imshow(Z, interpolation='nearest',
                       extent=(xx.min(), xx.max(), yy.min(), yy.max()),
                       aspect='auto', origin='lower', cmap=plt.cm.PuOr_r)

    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired)
    plt.colorbar(image)                            # 右侧色条：颜色 → 概率
```

| 名字 | 作用 |
|---|---|
| `Z.reshape(...).T` | `predict_proba` 给的是按「行优先」铺平的一维数组，要折回二维图片，再转置对齐 `imshow` 的 (行=纵轴, 列=横轴) 习惯 |
| `imshow(Z)` | 把二维数组当**图片**画：数值大小 → 颜色深浅 |
| `extent=(xmin, xmax, ymin, ymax)` | 告诉 matplotlib「这张图对应真实坐标的哪个范围」，否则坐标轴是 0~100 的像素下标 |
| `origin='lower'` | 让第 0 行画在**下面**（数学坐标），默认是画在上面（图像坐标），不加就会上下颠倒 |
| `aspect='auto'` | 允许长宽比不按 1:1，否则图会被撑成正方形 |
| `cmap=plt.cm.PuOr_r` | 紫-橙渐变，`_r` 是 reverse（反序） |
| `plt.colorbar(image)` | 画右侧颜色标尺，非常重要，否则读者不知道颜色代表多少 |

> **答案（作业问答题）**：图中颜色就是 $p(y=+1\mid x)$。**颜色由橙翻紫的那条窄带**就是模型「不确定」的区域，而黑色分类线正好穿过它的中间。理论上，$\sigma(z)=0.5$ 当且仅当 $z = w\cdot x+b = 0$，也就是一条直线；过渡的「宽度」由 $\lVert w\rVert$ 决定：$\lVert w\rVert$ 越大，S 形越陡，不确定带越窄（模型越自信）。

### 📄 修正版问答格全文翻译（solved 版「How does the picture visualize the unsure area?」）

> **Question:** How does the picture visualize the area where the model is unsure ($p(y=+1|x) = 0.5$)? How can this be justified theoretically?

**翻译**：**问题**：这张图是怎么把「模型不确定的区域」（$p(y=+1\mid x)=0.5$）显示出来的？理论上怎么证明？

> The background colour is the model's probability $p(y=+1|x)$. The narrow band where the colour flips from orange to purple is the region where the model is unsure, and the black decision line runs exactly through its middle.
>
> **Why theoretically?** The model is
> $$p(y=+1|x) = \sigma(w^Tx + b) = \frac{1}{1 + e^{-(w^Tx + b)}}, \qquad \sigma(0) = 0.5,$$
> so $p = 0.5$ is exactly the condition $w^Tx + b = 0$ — a straight line.

**翻译**：背景颜色就是模型的概率 $p(y=+1\mid x)$。颜色从橙色翻成紫色的那条窄带，就是模型「不确定」的区域，而黑色的决策线正好从它的正中间穿过。**理论上的证明**：模型是 $p(y=+1\mid x) = \sigma(w^Tx + b) = \frac{1}{1 + e^{-(w^Tx + b)}}$，而 $\sigma(0) = 0.5$，所以 $p = 0.5$ 恰好等价于 $w^Tx + b = 0$——一条直线。

> The transition is **smooth and continuous**: the closer to the boundary, the closer the probability is to 0.5; far away it saturates at 0 or 1. The width of the "uncertain band" is controlled by $\|w\|$: a larger norm makes the transition steeper (a more confident model).

**翻译**：颜色的过渡是**光滑连续**的：越靠近边界，概率越接近 0.5；远离边界则饱和到 0 或 1。「不确定带」的宽度由 $\lVert w\rVert$（权重向量的长度）控制：范数越大，过渡越陡——模型越自信。（这正是 1.5 节说的「正则化压小 $\lVert w\rVert$」的直接视觉后果：C 越小，这条过渡带越宽。）

In [ ]:
# 用数字确认：p = 0.5 就是 w·x + b = 0
import numpy as np
z = np.array([-3, -1, 0, 1, 3])
print("z     :", z)
print("sigma :", np.round(1 / (1 + np.exp(-z)), 4))

---
## Task 2：概率校准（1 分）

## 📖 原文（全文）

> __Task 2.__ Calibration of probabilities (1 point)
>
> Let's move on to assessing the quality of the probabilities generated by the algorithms. Let's start with calibration curves.
>
> Let's say the algorithm returns some numbers from zero to one. Are they good at assessing probability?
>
> Well-calibrated classifier should produce values such that among the samples for which it produces a value close to $\alpha$, approximately $\alpha \times$100% actually belonged to the positive class. (For example, if the classifier returns 0.3 for some, then 30% of them should belong to class 1)
>
> To do this, we divide the segment $[0, 1]$ into several small segments of the same length.
>
> Let's consider the $i$-th segment with boundaries $[a_i, b_i]$ and the predictions $p_1, p_2, \dots, p_k$ that fall into it. Let the true answers $y_1, y_2, \dots, y_k$ correspond to them. If the algorithm produces the correct probabilities, then there should be approximately $(a_i + b_i) / 2$ units among these true answers. In other words, if you draw a curve in which the centers of the segments are plotted along the X axis, and the proportions of these single answers in the segments along the Y axis, then it should turn out to be diagonal. Below is a function that should draw such curves. There are two mistakes in it - find and correct them.

## 🇨🇳 翻译（逐句）

**Task 2：概率校准（1 分）**

接下来我们评估算法输出的概率的质量。先从**校准曲线**（calibration curve）开始。

假设算法输出一堆 0 到 1 之间的数。这些数能当概率用吗？

一个**校准良好**的分类器应该满足：在所有它给出「接近 $\alpha$」的那些样本里，大约有 $\alpha\times 100\%$ 真的属于正类。（例如它给某些样本 0.3，那这些样本里应该真有 30% 是正类。）

为了验证这一点，把 $[0,1]$ 切成若干**等长**的小区间。

考虑第 $i$ 个区间 $[a_i, b_i]$，以及落在里面的预测值 $p_1, p_2, \dots, p_k$，它们对应的真实标签是 $y_1, y_2, \dots, y_k$。如果算法输出的是正确的概率，那么这些真实标签里应该大约有 $(a_i+b_i)/2$ 比例是 1（$(a_i+b_i)/2$ 就是区间中点）。换句话说：横轴取各区间中点、纵轴取该区间内正类所占比例，画出来的曲线应该是一条**对角线**。下面这个函数本来是画这种曲线的，但它里面有**两个错误**——找出来并改正。

> 这里的「units（单位）」翻译过来就是「正类（标签为 1）的样本」，原文用词比较简略。整条逻辑链是：**预测值分组 → 每组算真实正类比例 → 比例应该约等于组的中点**。

## 📖 后续三个任务格原文 + 翻译

> Generate synthetic data similar to those used in the very first task. Plot calibration curves on the test part for logistic regression and support vector machine (remember to convert its predictions to $[0;1]$). Draw the calibration curve of a perfectly calibrated model (diagonal).

**翻译**：生成和第一个任务类似的人造数据。在测试集上画出逻辑回归和 SVM 的校准曲线（记得把 SVM 的预测值转换到 $[0,1]$ 区间）。再画一条「完美校准模型」的校准曲线（即对角线）作参照。

> Examine the distribution of classifier responses (construct histograms using `plt.hist`).

**翻译**：检查两个分类器输出的分布（用 `plt.hist` 画直方图）。

> **Question:** How are they different? How can you explain this?

**翻译**：**问题**：两个分布有什么不同？怎么解释？

### 🧠 先搞清楚「校准」到底在说什么（这是全作业最容易懵的概念）

用一个具体例子。

假设你有 1000 个客户，模型给其中 100 个打了 **0.3** 分。校准良好的意思是：**这 100 个人里，真的应该有大约 30 个人成交。**

- 如果实际有 **30** 个成交 → 完美校准 ✅
- 如果实际有 **80** 个成交 → 模型**低估**了自己（说 0.3 实际 0.8）
- 如果实际只有 **5** 个成交 → 模型**高估**了自己（嘴上说 0.3，实际 5%）

> 类比：天气预报说「明天降水概率 30%」。如果一年里所有说 30% 的日子中，真的下了雨的比例是 30%，这个预报员就是**校准良好**的。他准不准（准确率）和他说的话可不可信（校准），是两件不同的事。

**为什么这很重要？** 因为 Part 4 要用概率算钱：$E[\text{利润}] = 10p - 1$。公式里的 $p$ 如果只是「排序分数」而不是真概率，这个公式就毫无意义。SVM 的 `decision_function` 就是典型的「只能排序，不能当概率」。

### 🧠 怎么算「每个区间的实际比例」

标准做法（作业描述的）：

1. 把 $[0,1]$ 切成 10 个区间：`[0,0.1), [0.1,0.2), ..., [0.9,1.0]`
2. 对每个区间：
   - 找出预测值落在这个区间的所有样本 → `mask`
   - 算这些样本里正类的**比例**（不是个数！）
   - 横坐标用区间中点，纵坐标用这个比例
3. 把所有点连起来 → 校准曲线
4. 再画一条 $y=x$ 的对角线作参照

### 🐛 原函数的两个错误（作业明确要求「找出来」）

原代码：

```python
for i in range(n_bins):
    l = 1.0 / n_bins * i
    r = 1.0 / n_bins * (i + 1)
    bin_middle_points.append((l - r) / 2)                                  # ← 错误 1
    bin_real_ratios.append(np.min(y_test[(preds >= l) & (preds < r)] == 1)) # ← 错误 2
```

**错误 1：`(l - r) / 2` 应该是 `(l + r) / 2`**

「中点」是两端相加除以二。写成相减会得到**负数**：比如第 0 个区间 $l=0, r=0.1$，算出来是 $-0.05$。结果横轴全是负的，曲线整个跑到图外。

**错误 2：`np.min(...)` 应该是 `np.mean(...)`**

`y_test[...] == 1` 得到一串 `True/False`。作业要的是「正类**比例**」，那就必须取平均（`True` 当 1、`False` 当 0，平均值就是比例）。

`np.min` 取的是最小值：只要该区间里有**一个**负样本，结果就是 0；只有当区间里**全是**正样本时才是 1。于是曲线变成「0 或 1 的锯齿」，完全失去意义。

In [ ]:
# 手动验证这两个 bug 的后果
import numpy as np
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 1])   # 假设这 10 个样本都落在同一个区间
mask = np.ones(10, dtype=bool)

print("l=0, r=0.1")
print("  错误的中心 (l-r)/2 =", (0 - 0.1) / 2, " ← 负数，画不出来")
print("  正确的中心 (l+r)/2 =", (0 + 0.1) / 2)
print("  错误的比例 np.min  =", int(np.min(y_true[mask] == 1)), " ← 只要有一个负例就变 0")
print("  正确的比例 np.mean =", np.mean(y_true[mask] == 1), " ← 6/10，这才是「正类占比」")

修正后的完整函数（作业提交版本）：

```python
def plot_calibration_curve(y_test, preds, label=None, ax=None):
    y_test = np.asarray(y_test).astype(bool)
    preds = np.asarray(preds, dtype=float)
    n_bins = 10
    bin_middle_points, bin_real_ratios = [], []
    for i in range(n_bins):
        l = 1.0 / n_bins * i
        r = 1.0 / n_bins * (i + 1)
        # 最后一个区间右边也要闭合，否则 preds == 1.0 的样本会被丢掉
        right = preds <= r if i == n_bins - 1 else preds < r
        mask = (preds >= l) & right
        if mask.sum() == 0:                 # 空区间跳过，否则除零/画点会出错
            continue
        bin_middle_points.append((l + r) / 2)        # 修正 1
        bin_real_ratios.append(y_test[mask].mean())  # 修正 2
    ...  # 画图部分
```

> 顺带改进了两点（作业没要求，但值得做）：**最后一个区间用 `<=`**，否则恰好等于 1.0 的预测会被漏掉；**空区间跳过**，否则 `mean()` 会得到 `nan`，曲线出现断点。

### ⚠️ 一个必须知道的细节：SVM 的分数怎么变成 [0,1]

作业说「remember to convert its predictions to $[0;1]$」。因为 SVM 给的是距离，可能是 −3.7 也可能是 +2.1。最直接的做法是 **min-max 归一化**：

```python
svm_min, svm_max = y_svm_score.min(), y_svm_score.max()
y_svm_scaled = (y_svm_score - svm_min) / (svm_max - svm_min)
```

⚠️ **但这只是「塞进 [0,1]」，不是「变成概率」**。它只做了线性拉伸，完全不改变形状。所以下面校准曲线会很差——这正是作业想让你看到的。真正把分数变成概率要靠 `CalibratedClassifierCV`（Platt scaling / 等渗回归）。

### 问答：SVM 和逻辑回归的校准曲线好不好？

**结论：**

- **逻辑回归**基本贴着对角线走——它本来就是按「最大化似然」训练的，模型形式里就写着概率，所以它给出的数大致可以当概率用。
- **SVM** 明显偏离：曲线在中间是平的、两端翘起。因为它的分数是「到分隔面的距离」，min-max 只是把它压进了 [0,1]，没有任何概率含义。

**一句话总结**：某个区间的「预测值」和该区间里「真实正类比例」是两回事。LR 天生是概率模型；SVM 不是——所以下一步要用 `CalibratedClassifierCV`。

### 问答：两个模型的分数分布（直方图）有什么区别？

本次运行，统计「预测值落在 0.05 以下或 0.95 以上」的样本比例：

| 模型 | 极端值的比例 |
|---|---|
| 逻辑回归（原始概率） | 5.0% |
| SVM（min-max 后） | 0.2% |

**怎么解释：**

1. **min-max 是线性拉伸，会「锁死」端点**。它把最小值强行映射到 0、最大值强行映射到 1，中间的点按比例摊开。所以 SVM 的分布看起来「铺满整个 [0,1]」，但形状其实还是原来那个距离分布——**它没有变得更像概率，只是换了个刻度**。
2. **逻辑回归的分布形状由 log loss 决定**。log loss 对「已经很对」的点仍有惩罚，所以它会把预测持续往 0/1 推；但它同时受 `C` 约束（这里 C=0.001，正则化较强），权重被压小，所以大部分概率集中在 0.5 附近，只有 5% 走到了极端。
3. **这就是为什么 S 形校准曲线是弯的**：SVM 说 0.9 的那批点，实际成交率可能只有 0.6——分数和实际频率对不上。

> **答案（写进作业）**：LR 的直方图在两端有可见的质量（5%），因为 log loss 会持续奖励「更自信」的预测；SVM 经 min-max 后几乎全部挤在中间（99.8%），因为 hinge loss 对间隔外的点不再施加任何推力，分数自然停在 ±1 附近，再被线性拉伸。形状差异的根源是**两种损失函数对「已经分对的点」态度不同**。

### 📄 修正版答案格全文翻译（solved 版校准曲线答案 + 直方图答案）

> - **Logistic regression** stays close to the diagonal — it is reasonably well calibrated, because its training objective (log-likelihood) is itself a model of probabilities.
> - **SVM** deviates clearly: the curve is flat in the middle and rises at the edges. SVM scores are distances to the hyperplane, not probabilities; min-max rescaling only squeezes them into [0, 1] without giving them probabilistic meaning.
>
> **In one sentence:** a score in a bin is not the same thing as the fraction of positives in that bin. LR is a probabilistic model by construction; SVM is not — which is why the next step applies `CalibratedClassifierCV`.

**翻译（校准曲线答案）**：
- **逻辑回归**贴着对角线——校准得不错，因为它的训练目标（对数似然）本身就是概率模型。
- **SVM** 明显偏离：曲线中间平、两端翘。SVM 的分数是到超平面的距离，不是概率；min-max 只是把它们压进 [0,1]，并没有赋予概率含义。
- **一句话**：「某个区间的预测值」和「该区间的真实正类比例」是两回事。LR 天生是概率模型，SVM 不是——所以下一步要用 `CalibratedClassifierCV`。

> * **The SVM histogram is just the score distribution stretched linearly.** min-max maps the smallest score to 0 and the largest one to 1 and preserves everything in between, so the shape carries no probabilistic meaning. Only a handful of objects reach the edges (hence the small percentage) simply because the extremes are single points; the mass is spread thinly over the whole interval.
> * **The LR histogram has visible mass at both ends.** Log loss never stops penalising an object that is confident *and* wrong, so it keeps pushing predictions towards 0 and 1. Here the push is partly held back by regularization (C = 0.001), which is why the bulk still sits in the middle.
> * **Same scores, different meaning.** For LR "0.9" means "I believe 90%". For the rescaled SVM "0.9" means "this object is far on the positive side of the boundary" — the number has no frequency interpretation at all. That is exactly why the SVM calibration curve bends away from the diagonal, and why `CalibratedClassifierCV` is needed.

**翻译（直方图答案）**：

> Measured share of predictions that fall outside $[0.05, 0.95]$:
>
> | model | share outside |
> |---|---|
> | Logistic Regression | 5.0 % |
> | SVM (min-max) | 0.2 % |

实测「预测值落在 $[0.05, 0.95]$ 之外」的比例：逻辑回归 **5.0%**，SVM（min-max 后）**0.2%**。

- **SVM 的直方图只是分数分布被线性拉伸的结果**。min-max 把最小分映射到 0、最大分映射到 1，中间按比例摊开，形状上没有概率含义。只有极少数样本到达两端（所以极端值占比只有 0.2%），因为端点本来就是个别点；大部分质量被薄薄地摊在整个区间上。
- **LR 的直方图两端有可见的质量**。log loss 对「自信但分错」的样本永不停止惩罚，所以它会持续把预测往 0 和 1 推。这里这股推力被正则化（C = 0.001）部分地拉住了，所以主体仍在中部。
- **同样的数字，不同的含义**。LR 说「0.9」是「我有 90% 的把握」；拉伸后的 SVM 说「0.9」只是「这个样本在边界正侧很远的地方」——这个数字没有任何频率含义。这正是 SVM 校准曲线偏离对角线的原因，也是必须做校准的原因。

### 🧠 校准怎么做：Platt scaling 与 isotonic

> **作业原文**：Use the `CalibratedClassifierCV` from `sklearn` to calibrate the probabilities of the support vector machine on training and use it to make predictions for the test set. Draw a calibration curve for them.
>
> **翻译**：用 `sklearn` 的 `CalibratedClassifierCV` 在训练集上校准 SVM 的概率，并用它对测试集做预测。画出校准后的校准曲线。

**Platt scaling（参数法）**：假设真实概率和分数之间是一个 sigmoid 关系

$$p = \sigma(a\cdot s + b) = \frac{1}{1+e^{-(a s + b)}}$$

然后用**最大似然**把 $a, b$ 两个数拟合出来。这本质上就是在分数上再套一个**一维的逻辑回归**。

**Isotonic regression（非参数法）**：不假设形状，直接拟合一个**单调递增的阶梯函数**，让「分数 → 实际频率」的映射误差最小。更灵活，但需要更多数据，小样本容易过拟合。

**`CalibratedClassifierCV` 帮你做的三件事**：

```python
svm_cal = CalibratedClassifierCV(SVC(kernel="linear", C=1.0), method="sigmoid", cv=5)
svm_cal.fit(X_train, y_train)
y_svm_cal = svm_cal.predict_proba(X_test)[:, 1]
```

| 参数 | 含义 |
|---|---|
| 第一个位置参数 | 要校准的基础模型 |
| `method="sigmoid"` | 用 Platt（`"isotonic"` 则用等渗回归） |
| `cv=5` | 内部做 5 折交叉验证，产生 **out-of-fold** 分数 |

⚠️ **为什么一定要 out-of-fold？** 因为模型在自己见过的训练数据上会「过度自信」（分数离边界特别远）。如果用这些分数去拟合 sigmoid，得到的映射会太陡，校准就失效了。所以要先切 5 折，每折的分数都由「没见过这些样本的模型」给出，这样才能代表模型面对新数据时的真实表现。

代价：基础模型被训练了 5 次，耗时 ×5。

> **答案（作业问答题）**：校准后曲线明显改善，几乎与对角线重合。但要注意——**校准只改变分数的「刻度」，不改变排序**，所以 AUC-ROC / AUC-PR 几乎不变（本次运行 0.84764 → 0.84779）。如果你只关心排序，校准没有收益；如果你要用概率算钱（Part 4），校准就是必需的。

### 📄 修正版答案格全文翻译（solved 版 CalibratedClassifierCV 答案）

> Yes, the curve improves markedly: after Platt scaling the SVM curve essentially coincides with the diagonal.

**翻译**：是的，曲线明显改善：经过 Platt scaling 之后，SVM 的校准曲线基本上与对角线重合。

> **How it works:** `CalibratedClassifierCV(..., method="sigmoid")` runs an internal cross-validation, collects out-of-fold SVM scores, and fits a one-dimensional logistic regression that maps a score to a probability (this is Platt scaling). Because the scores used for fitting are out-of-fold, the calibration is not distorted by the model's over-confidence on its own training data.
>
> **The price:** the base model is trained five times (cv=5). Note also that calibration changes only the **scale** of the scores, not their order — AUC-ROC and AUC-PR barely move (measured: 0.84764 → 0.84780).

**翻译**：**工作原理**：`CalibratedClassifierCV(..., method="sigmoid")` 在内部做交叉验证，收集 out-of-fold 的 SVM 分数，然后拟合一个「分数 → 概率」的一维逻辑回归（这就是 Platt scaling）。由于拟合用的分数是 out-of-fold 的，校准不会被模型在自家训练数据上的「过度自信」带偏。**代价**：基础模型被训练 5 次（cv=5）。另外注意，校准只改变分数的**刻度**，不改变它们的**顺序**——所以 AUC-ROC 和 AUC-PR 几乎不动（实测：0.84764 → 0.84780）。

## 加分题：自己实现一个校准器

## 📖 原文

> Implement your function to calibrate the probabilities. Describe your approach and show the results. Keywords for inspiration: `Platt`, `Isotonic`.

## 🇨🇳 翻译

自己实现一个校准概率的函数。描述你的方法并展示结果。关键词：`Platt`、`Isotonic`。

### 🧠 思路（四步，比调库只多一步「自己算」）

```python
def platt_calibrate(base_model, X, y, cv=5, random_state=42):
    # ① 切 5 折，收集「out-of-fold 分数」
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    oof_scores = np.zeros(len(y))
    for tr, te in skf.split(X, y):
        m = clone(base_model).fit(X[tr], y[tr])
        oof_scores[te] = m.decision_function(X[te])   # 每折的分数来自「没见过这些样本」的模型

    # ② 在分数上拟合一个一维逻辑回归 —— 这就是 Platt scaling
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(oof_scores.reshape(-1, 1), y)
    return platt, oof_scores

platt, oof = platt_calibrate(LinearSVC(C=svm.C, max_iter=20000), X_train, y_train)
y_svm_platt = platt.predict_proba(y_svm_score.reshape(-1, 1))[:, 1]
```

| 名字 | 作用 |
|---|---|
| `clone(base_model)` | 复制一个**未训练**的同款模型（不是 `copy.deepcopy`，clone 会清空已学到的参数） |
| `skf.split(X, y)` | 返回 5 组 `(训练下标, 验证下标)` |
| `oof_scores[te] = ...` | 把每折验证集的分数填回原位 → 每个样本都拿到「没见过它的模型」给的分数 |
| `C=1e6` | 几乎不要正则化——这里只是拟合 2 个参数 (a, b)，不能压它们 |
| `reshape(-1, 1)` | 逻辑回归要求二维输入 `(n, 特征数)`，这里特征数 = 1 |

本次运行得到 $a = 2.70$、$b = -0.003$，即 $p = \sigma(2.70s)$。$a>0$ 说明分数越大越可能为正类，符合预期。

> **为什么 $b\approx 0$**：数据是均衡的（正类 50%），对称的数据集自然对应「零偏置」的映射。如果正类只有 11%，$b$ 会明显为负。

### 📄 修正版「方法说明」格全文翻译（solved 版加分题 Approach）

> **Approach (bonus).** Platt scaling = "fit a one-dimensional logistic regression on top of the scores":
>
> 1. take the SVM scores of the objects, **which the model has not seen** (out-of-fold, `cv=5`);
> 2. fit $p = \sigma(a\cdot s + b)$ on those scores by maximum likelihood (`LogisticRegression` with one feature and almost no regularization);
> 3. apply the resulting $a, b$ to the scores of the final model.
>
> Why out-of-fold: the SVM is over-confident on its own training objects (their scores are far from the boundary), so calibrating on in-sample scores gives a sigmoid that is too steep. Isotonic regression is the non-parametric alternative — it fits a monotone step function instead of a sigmoid, needs more data, and can overfit on small samples.

**翻译**：**方法（加分题）**：Platt scaling = 「在分数之上再拟合一个一维逻辑回归」：
1. 取**模型没见过**的那些样本的 SVM 分数（out-of-fold，`cv=5`）；
2. 在这些分数上用最大似然拟合 $p = \sigma(a\cdot s + b)$（用只有一个特征、几乎不加正则化的 `LogisticRegression`）；
3. 把得到的 $a, b$ 应用到最终模型的分数上。

**为什么要 out-of-fold**：SVM 在自己的训练样本上会过度自信（分数离边界特别远），如果用训练集内的分数来校准，拟合出的 sigmoid 会过陡。**Isotonic（等渗）回归**是非参数的替代方案——它拟合的是单调阶梯函数而不是 sigmoid，更灵活但需要更多数据，小样本上可能过拟合。

---
# 3. Part 2：变量处理（类别编码）（4 分 + 1.5 加分）

## 📖 原文（全文）

> In this part, we will learn how to handle categorical variables, since it is not enough to encode them as numbers (this sets some order, which may not be on categorical variables). There are two main ways to handle categorical values:
> - One-hot coding
> - Counters (CTR, mean-target coding, ...) — each categorical feature is replaced by the average value of the target variable for all objects that have the same value in this feature.
>
> Let's start with one-hot coding. Suppose our categorical feature $f_j(x)$ takes values from the set $C=\{c_1, \dots, c_m\}$. Let's replace it with $m$ binary features $b_1(x), \dots, b_m(x)$, each of which is an indicator of one of the possible categorical values:
> $$b_i(x) = [f_j(x) = c_i]$$

## 🇨🇳 翻译（逐句）

这一部分学习怎么处理**类别变量**。仅仅把它们编码成数字是不够的，因为那会引入一种「顺序」，而类别变量上可能根本不存在这种顺序。处理类别值主要有两种方法：

- **独热编码（one-hot coding）**
- **计数器 / 目标编码（CTR、mean-target coding 等）**——把每个类别特征替换成「具有相同取值的所有样本的标签平均值」。

先看 one-hot。假设类别特征 $f_j(x)$ 的取值来自集合 $C=\{c_1, \dots, c_m\}$（共 $m$ 种）。我们把它换成 $m$ 个二值特征 $b_1(x), \dots, b_m(x)$，每个特征都是「是不是某一种取值」的指示器：

$$b_i(x) = [f_j(x) = c_i]$$

> 公式读法：方括号 $[\cdot]$ 是**指示函数**（Iverson bracket）——括号里的条件成立就是 1，否则是 0。所以「样本的职业是 technician」这件事会变成：technician 那一列是 1，其余 11 列都是 0。

### 🧠 为什么类别变量不能直接编号（这一节是整个 Part 2 的地基）

假设有一列 `job`，取值是 `admin.`、`blue-collar`、`technician`……如果你直接编码成 0、1、2，会发生什么？

线性模型会计算 $w\cdot x$。它只能理解一件事：**$x$ 变大时预测变大还是变小，以及变化多快**。所以编号 0/1/2 隐含了三个断言：

1. `technician(2)` 比 `admin.(0)` 更「大」——但职业没有大小。
2. `blue-collar(1)` 到 `technician(2)` 的距离 = `admin.(0)` 到 `blue-collar(1)` 的距离——凭什么？
3. 模型只能给出**一条单调影响曲线**：职业编号越大，成交率越高（或越低）。而现实里可能是两头高中间低（U 型）。

> 类比：给「北京/上海/广州」编号 1/2/3，然后说「广州 - 上海 = 上海 - 北京」，这句话没有意义。数字的**算术结构**被强行加到了类别上。

### 解决办法一：one-hot（独热）

把一列 $m$ 个取值的类别，换成 $m$ 列 0/1，每列表示「是不是这个取值」：

$$b_i(x) = [f_j(x) = c_i]$$

`technician` 变成 `[0, 0, 1, ..., 0]`。这样模型给每一列**各自一个权重**，不再有任何顺序假设。

代价：列数从 1 变成 $m$。作业里 `job` 有 12 种取值，一列变 12 列；10 个类别列总共让 19 列变成 **62 列**。

### 解决办法二：计数器 / 目标编码（counters / target encoding）

把每个类别替换成**这个类别里正类出现的比例**：

$$g_j(x, X) = \frac{\sum_{i=1}^{l}[f_j(x)=f_j(x_i)][y_i=+1]}{\sum_{i=1}^{l}[f_j(x)=f_j(x_i)]}$$

直白说：**「干这行的人里，有多大比例买了」**。

- 好处：无论多少种取值，始终只占 **1 列**。
- 好处：它自动把「成交率高/低」这个信息编码进去了，而且天然有序（0.3 > 0.1 是有意义的）。
- 风险：见下面「泄漏」一节。

### 🧠 数据集介绍（后面 Part 2/3/4 都用它）

> **作业原文（Data preparation 格全文）**：__Data preparation.__ Download [UCI Bank Marketing Dataset](https://archive.ics.uci.edu/ml/datasets/bank+marketing). This dataset contains information about bank telephone marketing, the object in it is a phone call to a potential client with an offer of some service (it is alleged that this is a short-term deposit), the target variable is the client's response (did he/she agree to open a deposit?). Characteristics of the client (education, marriage, etc.), call data and various economic indicators are used as an indicative description - more detailed information is presented in the `bank-additional-names.txt` file.
>
> **翻译**：**数据准备。** 下载 [UCI Bank Marketing 数据集]。这个数据集记录了一家银行的电话营销活动：每个样本是**一次打给潜在客户的电话**（推销的是定期存款），目标变量是客户的回答（他/她同意开户了吗）。特征包括客户属性（教育、婚姻等）、通话数据和各种经济指标——更详细的说明在 `bank-additional-names.txt` 文件里。

```python
df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')
```

- 文件来自 UCI 的 **Bank Marketing** 数据集，`sep=';'` 是因为这个 CSV 用**分号**分隔（欧洲习惯），用默认的逗号会读成一列。
- 41188 行 × 21 列，每行是**一次电话营销**：银行打电话向客户推销定期存款。
- `y` 是结果：`'yes'`（客户同意开户）/ `'no'`。正类只占 **11.27%**，严重不平衡。
- 特征分三类：
  - **客户属性**：`age`、`job`、`marital`、`education`、`default`、`housing`、`loan`
  - **本次通话**：`contact`、`month`、`day_of_week`、`duration`（⚠️ 这个要删，见下）
  - **经济指标**：`emp.var.rate`、`cons.price.idx`、`cons.conf.idx`、`euribor3m`、`nr.employed`
  - **历史**：`campaign`（本次活动中联系了几次）、`pdays`（距上次联系多少天，999 = 从未联系）、`previous`、`poutcome`

## Task 3 · 原文

> Divide the sample into training and testing in a ratio of 3:1. Fix `random_state=777`, also use `stratify`. One of the columns in the data does not carry information for solving the problem - which one? Drop it.

## 🇨🇳 翻译

按 **3:1** 的比例把样本分成训练集和测试集。固定 `random_state=777`，并使用 `stratify`。数据里有一列对解决问题**不提供任何信息**——是哪一列？把它删掉。

### 🧠 答案：`duration`（通话时长），以及为什么

这是全作业**最重要的一道概念题**。要理解它，必须问自己一个问题：

> **模型上线以后，做预测的那一刻，这个特征拿得到吗？**

- `duration` = 这通电话打了多少秒。**但「要不要打这个电话」这个决定，是在打电话之前做出的。**打的时候你根本不知道会打多久。
- 更致命的是：`duration=0` 就意味着没接通，那必然 `y='no'`。所以 `duration` 几乎把答案写在脸上了——数据里 `duration` 和 `y` 的相关性强得离谱，因为它是**通话的结果**，不是通话的**原因**。

> 类比：预测「病人会不会痊愈」，特征里放「住院住了多少天」。这个特征当然和结果高度相关，但你做预测的时候（刚入院）根本不知道他会住多久。用它训练出来的模型 AUC 会很漂亮，上线后完全没用。

这种错误叫 **数据泄漏（data leakage / target leakage）**，是工业界最常见、代价最惨重的 bug。作业后面 Part 2 的目标编码泄漏，是同一类问题的一个变种。

> **答案（写进作业）**：删掉 `duration`。它只在通话**结束后**才可知，而预测（是否要打这个电话）发生在通话**之前**，因此这个特征在实际使用场景中不可得，同时它几乎直接决定了标签，会造成严重的标签泄漏。用它训练会得到一个在离线指标上极其漂亮、上线后完全失效的模型。

### 🧠 3:1 与 `stratify`

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=777, stratify=y)
```

- 「训练 : 测试 = 3 : 1」意味着测试集占 $\frac{1}{3+1}=25\%$，所以 `test_size=0.25`。这是学生最容易写错的可笑错误（写成 0.3 就变成 7:3 了）。
- `stratify=y`：**分层抽样**。保证训练集和测试集里的正类比例都等于整体的 11.27%。不加的话，随机切出来的测试集可能正类是 10% 或 12.5%，指标会不稳定。正类稀少时**必须**加。
- `random_state=777`：固定随机种子，保证结果可复现。

### 🧠 为什么这个任务用 AUC-PR 而不是 AUC-ROC（正式答案）

见 1.7 节的实验：正类只有 11.27%，负类占 88.7%。

- **AUC-ROC 的分母里有 TN**。FPR $=\frac{FP}{FP+TN}$，TN 巨大，所以 FPR 天生很小，ROC 曲线很容易贴左上角 → 指标虚高，看不出模型好坏。
- **AUC-PR 只看正类**。precision $=\frac{TP}{TP+FP}$ 完全不受那 88.7% 负类的"稀释"。
- 业务上也很贴合：我们真正关心的是「打出去的电话里有多少真的成交」（precision）和「能成交的客户我们找到了多少」（recall）。

> 经验法则：**正类稀有 + 两类错误代价不对称 → 用 PR 系指标。**

### 📄 修正版答案格全文翻译（solved 版 Task 3 的「Your reply」）

> __Question__: why in this task we chose `AUC-PR` as a metric, and not, for example, `AUC-ROC`?

**翻译（问题）**：**问题**：为什么这个任务我们选 `AUC-PR` 作为指标，而不是比如 `AUC-ROC`？

> Because the classes are highly imbalanced: only about 11% of the clients say "yes". With such data AUC-ROC is too optimistic -- most of the ROC curve is produced by the huge number of true negatives, so the metric stays high even when the model is poor at finding the positive class.

**翻译**：因为类别高度不平衡：只有约 11% 的客户回答「是」。对这种数据，AUC-ROC 太乐观了——ROC 曲线的大部分是由海量真负例「贡献」的，所以即使模型找正类的能力很差，这个指标依然很高。

> AUC-PR looks only at the positive class: precision = "of those we call, how many agree", recall = "of those who would agree, how many we found". This is exactly the business question (we pay for calls, we want the calls to hit), so AUC-PR is both more informative and closer to the real objective.

**翻译**：AUC-PR 只盯着正类：precision = 「我们打出去的电话里，多少真的成交」，recall = 「本来会成交的客户里，我们找到了多少」。这正是业务问题（电话要花钱，我们希望打得准），所以 AUC-PR 既更有信息量，也更贴近真实目标。

> Rule of thumb: with a rare positive class and/or asymmetric error costs, use PR-based metrics; with balanced classes and equally important errors, ROC is fine.

**翻译**：经验法则：**正类稀有、或者两类错误代价不对称时，用 PR 系指标；类别均衡、两类错误同等重要时，ROC 没问题。**

## Task 3 代码：用 `OrdinalEncoder` 做基线

```python
cat_cols = [c for c in X.columns if X[c].dtype == object or str(X[c].dtype) in ("str", "string")]
num_cols = [c for c in X.columns if c not in cat_cols]
```

这行在干一件事：**把列分成「类别列」和「数值列」两拨**。

- `X[c].dtype == object`：pandas 默认把字符串存成 `object` 类型。
- `str(X[c].dtype) in ("str", "string")`：pandas 3.0 之后字符串有了专门的 `str` 类型，所以要多判一次。**这段是版本兼容写法，不是啰嗦。**

```python
def make_pipe(mode="ordinal", C=1.0):
    if mode == "onehot":
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    else:
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    pre = ColumnTransformer([
        ("cat", encoder, cat_cols),
        ("num", StandardScaler(), num_cols),
    ])
    return Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=1000, C=C))])
```

| 名字 | 作用 |
|---|---|
| `OrdinalEncoder` | 把第 1 个类别编成 0、第 2 个编成 1……（**按字母序**，不是按重要性） |
| `handle_unknown="use_encoded_value"` + `unknown_value=-1` | 遇到训练时没见过的类别，填 −1 而不是报错。**测试集出现新类别时必备** |
| `handle_unknown="ignore"`（one-hot 用） | 遇到没见过的类别，全部列填 0 |
| `sparse_output=False` | 返回普通密集矩阵。`True`（默认）返回稀疏矩阵，省内存 |
| `StandardScaler()` | **标准化**：$(x-\mu)/\sigma$，让每列均值为 0、标准差为 1 |
| `max_iter=1000` | 优化器的最大迭代次数。数值没标准化时经常会撞到这个上限并警告 |

> ⚠️ **为什么要加 `StandardScaler`**（这是我对原解答的一处修正）：`euribor3m` 的量级 ~1-5，而 `nr.employed` 的量级 ~5000。不标准化时，梯度下降在不同方向上步长差异巨大，`lbfgs` 常常在 1000 次迭代内**收敛不了**（sklearn 会打印 `ConvergenceWarning`）。更本质的是：L2 正则项 $\frac{1}{2C}\lVert w\rVert^2$ 惩罚的是权重的绝对值，而权重的大小取决于特征的量级——不标准化就等于**对不同特征施加了不同的惩罚力度**，这是模型层面的错误，不只是速度问题。

> 注意 Part 3 里会专门讨论「不标准化的后果」，那里是**故意的**，不要混淆。

In [ ]:
# 小实验：为什么 L2 正则化要求先标准化
import numpy as np
from sklearn.linear_model import LogisticRegression

rng = np.random.RandomState(0)
n = 2000
x_small = rng.randn(n)                        # 刻度 ~1
x_big = x_small * 1000 + rng.randn(n) * 0.01  # 同样的信号，刻度大 1000 倍
y = (x_small + 0.5 * rng.randn(n) > 0).astype(int)

m = LogisticRegression(C=1.0).fit(np.c_[x_small, x_big], y)
w = m.coef_[0]
print("两个特征携带的信息完全相同，但权重是: [%.6f, %.6f]" % (w[0], w[1]))
print("后者是前者的 %.0f 倍" % (w[1] / w[0]))

这段输出揭示了一件反直觉的事：**两个特征信息完全一样，但模型几乎只用「刻度大」的那个**（权重相差几百倍，刻度小的那个被压到接近 0）。

为什么会这样？因为 L2 惩罚的是 $w^2$。要让预测产生同样的效果：

| 用哪个特征 | 需要多大的权重 | 要付的惩罚 $w^2$ |
|---|---|---|
| 刻度小的 `x_small` | $w \approx 1$ | $1$ |
| 刻度大的 `x_big` | $w \approx 0.001$ | $10^{-6}$ |

**同样的效果，代价差一百万倍**，所以模型当然选刻度大的。

结论：**`|coef|` 的大小同时被「重要性」和「刻度」两个因素决定，你没法把它们分开。** 所以「看权重判断特征重要性」这件事，在未标准化的数据上完全不可信——Part 3 的坑就埋在这里。

## Task 4 · 原文

> Encode all categorical features using one-hot coding. Train logistic regression and see how the quality of the model has changed (from what it was before encoding). Measure the time required to encode categorical features and train the model.

## 🇨🇳 翻译

用 one-hot 编码所有类别特征。训练逻辑回归，看看模型质量和之前（直接编号时）相比有什么变化。测量「编码 + 训练」所需的时间。

### 结果对比（本次运行）

| 编码方式 | 特征数 | AUC-PR | AUC-ROC | 耗时 |
|---|---|---|---|---|
| `OrdinalEncoder`（直接编号） | 19 | 0.44229 | — | 0.18 s |
| `OneHotEncoder`（独热） | 62 | 0.44998 | — | 0.16 s |

**怎么读：**

- one-hot 让 AUC-PR 提升了约 0.008（0.44229 → 0.44998）。提升不大，但方向是对的——**去掉了虚假的「顺序」假设**。
- 特征数从 19 涨到 62，但**训练时间几乎没变**（0.18 s → 0.16 s）。原因有两点：数据只有 3 万行；one-hot 用稀疏矩阵存储，0 元素根本不参与计算。**维度膨胀的代价要到「几万列」的规模才会显现。**
- 为什么提升这么小？因为逻辑回归是线性模型，而「职业编号递增 → 成交率递增」这个错误假设，恰好能拟合出「大致单调」的那部分真实规律。如果数据里类别效应是 U 型的（比如年龄），one-hot 的优势才会明显。

> **答案（写进作业）**：one-hot 之后质量略有提升（AUC-PR 从 0.44229 到 0.44998），代价是特征数从 19 涨到 62、训练时间增加。原因是直接编号引入了虚假的顺序与等距假设，而 one-hot 让每个类别拥有独立权重；但线性模型本身能力有限，所以提升幅度不大。

## Task 5 · 原文

> As it could be replaced, one-hot encoding can greatly increase the number of features in the dataset, which affects memory, especially if some feature has a large number of values. This problem is solved by another way of encoding categorical features - counters. The main idea is that it is not the categories themselves that are important to us, but the values of the target variable that the objects of this category have. We will replace each categorical feature with the average value of the target variable for all objects of the same category:
> $$
> g_j(x, X) = \frac{\sum_{i=1}^{l} [f_j(x) = f_j(x_i)][y_i = +1]}{\sum_{i=1}^{l } [f_j(x) = f_j(x_i)]}
> $$
>
> __Task 5.__ Encode categorical variables with counters (exactly as described above without any tricks). Train logistic regression and look at the quality of the model on the test set. Compare training time with previous experiments (taking into account feature encoding). Did you notice something interesting?

## 🇨🇳 翻译（逐句）

如前所示，one-hot 会大幅增加数据集的特征数，从而占用更多内存——尤其当某个特征的取值特别多的时候。这个问题由另一种类别编码方式解决：**计数器（counters）**。核心思想是：我们关心的**不是类别本身**，而是「属于这个类别的样本，其目标变量取什么值」。我们把每个类别特征替换成「同类别所有样本的目标变量均值」：

$$g_j(x, X) = \frac{\sum_{i=1}^{l} [f_j(x) = f_j(x_i)][y_i = +1]}{\sum_{i=1}^{l } [f_j(x) = f_j(x_i)]}$$

> 公式读法：分子是「和样本 $x$ 同属一类的所有样本里，标签为 1 的个数」；分母是「和 $x$ 同属一类的样本总数」。两者一除，就是**这个类别的成交率**。$l$ 是训练集样本数。

**任务 5**：用计数器编码类别变量（**就按上面那个公式，不要任何技巧**）。训练逻辑回归，看测试集质量。和前面的实验比较训练时间（要算上编码时间）。发现什么有意思的现象了吗？

### 🧠 什么叫「without any tricks」（不加任何技巧）

就是**直接用公式**：某个类别的编码 = 这个类别里正类的比例。统计量用**全部训练数据**算，包括这个样本自己。

这正是问题所在。看一个极端例子：

> 假设有一列 `客户ID`，每个客户只出现一次。用上面的公式编码，每个人的编码就是**他自己的标签**（0 或 1）。模型一看这列，直接抄答案，训练集准确率 100%。但测试集里的 ID 是**新的**，一个都没见过，这列全是「未知」，模型立刻崩掉。

这就是**目标泄漏（target leakage）**在特征工程层面的版本：你把答案的一部分偷偷放进了特征里。

In [ ]:
# 极端演示：不加技巧的计数器 = 把答案写进特征
import numpy as np, pandas as pd

rng = np.random.RandomState(0)
n = 2000
train = pd.DataFrame({"user_id": np.arange(n),          # 每个用户只出现一次
                      "y": rng.randint(0, 2, size=n)})

# 「不加技巧」的计数器：直接在训练集上按 user_id 分组求标签均值
enc = train.groupby("user_id")["y"].mean()
train["enc"] = train["user_id"].map(enc)

print("编码与标签的相关系数:", round(train["enc"].corr(train["y"]), 3), " ← 等于 1，就是标签本身")
print("用这个特征「预测」训练集准确率:", round(train["enc"].gt(0.5).eq(train["y"].astype(bool)).mean(), 3))

# 新来一批用户（训练时从未出现）
test = pd.DataFrame({"user_id": np.arange(n, n + 5)})
print("\n新用户的编码:", test["user_id"].map(enc).tolist(), " ← 全是 NaN，特征彻底失效")

真实数据没这么极端（每个类别通常有几十上百个样本），但机理完全一样：**编码里混进了每个样本自己的标签**，模型会过度相信这个特征。表现出来就是：训练集分数高、测试集分数低，而且新类别一到就抓瞎。

### 结果（本次运行）

| 编码方式 | 特征数 | AUC-PR | 耗时 |
|---|---|---|---|
| one-hot | 62 | 0.44998 | 0.16 s |
| 计数器（不加技巧） | 19 | 0.44268 | 0.15 s |

**有意思的地方有两个：**

1. **计数器耗时和 one-hot 基本持平**（0.15 s vs 0.16 s），虽然只有 19 列而不是 62 列。原因同上：这个规模下 time 主要花在 I/O 和优化器启动上，列数不是瓶颈。**「特征少 = 快」这个直觉要到高基数场景才成立。**
2. **但计数器在这个数据集上反而更差**（0.44268 < 0.44998）。这与「计数器是 one-hot 的替代品」这个流行说法看似矛盾，原因有三：
   - 这个数据集的类别数**本来就少**（最多 12 种），one-hot 出来的 62 列完全撑得住，维度膨胀的坏处根本没显现；
   - 计数器把 12 个类别压成 1 个数字，**丢失了信息**：`admin.` 和 `technician` 的成交率可能都 ≈0.12，但它们的效应可能在其他特征的交互中不同；
   - 目标编码是**单变量**的，它只能表达「这个类别的平均效应」，无法表达「在 30 岁以下的人群里，`admin.` 的效果不一样」。

> **结论**：计数器的优势在**高基数**（取值几千几万种）特征上才明显，比如用户 ID、邮编、商品 ID。对本数据集的低基数特征，one-hot 更稳。

## 加分项：把计数器实现成 sklearn 的 Transformer

## 📖 原文

> __Bonus__: Here and below, implement the calculation of counters using a transformer (inherit from the `BaseEstimator, TransformerMixin` classes from `sklearn.base`). Please note that all calculations must be vectorized, the transformer must not modify the inplace sample passed to it, and all necessary statistics must be calculated only on the training sample in the `fit` method. Your transformer should take a list of categorical features during initialization and change only them.

## 🇨🇳 翻译

**加分**：以下所有地方，都把计数器的计算实现成一个 **transformer**（继承 `sklearn.base` 里的 `BaseEstimator` 和 `TransformerMixin`）。注意：所有计算必须**向量化**；transformer **不能原地修改**传进来的样本；所有统计量只能**在 `fit` 方法里、只用训练样本**算出来。你的 transformer 要在初始化时接收一个类别特征列表，并且**只改这些列**。

### 🧠 为什么要写成 Transformer（而不是一个函数）

写成 transformer 不是为了好看，是为了三件事：

1. **能塞进 `Pipeline`**。这样 `Pipeline.fit(X_train)` 时，统计量只从训练集算；`predict(X_test)` 时自动复用同一套统计量。**防泄漏从「靠自觉」变成「靠结构」。**
2. **能被 `GridSearchCV` 搜索参数**。`TargetEncoder(smoothing=...)` 里的 `smoothing` 就变成了可搜索的超参数。
3. **符合 sklearn 的约定**，别人（和你三个月后的自己）一看就懂。

### 四个必须实现的东西

| 方法 | 什么时候调用 | 该干什么 |
|---|---|---|
| `__init__` | 创建对象时 | **只存参数，不计算任何东西**（这是 sklearn 的硬约定，否则 clone 会出错） |
| `fit(X, y)` | 训练时 | 从训练数据算出所有统计量，存到以 `_` 结尾的属性里 |
| `transform(X)` | 训练后、预测时 | 用 `fit` 存下的统计量去转换新数据 |
| `fit_transform(X, y)` | 训练时 | 默认是 `fit` 然后 `transform`；这里要**重写**，因为训练时要用 out-of-fold |

`BaseEstimator` 送给你 `get_params` / `set_params`（GridSearch 需要），`TransformerMixin` 送给你 `fit_transform`（默认实现）。所以只要写 `__init__/fit/transform` 三个。

### 💻 逐行读完整实现

```python
class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None, smoothing=0.0, noise=0.0, cv=None, random_state=0):
        self.cols = cols              # 要编码哪些列（None = 自动找字符串列）
        self.smoothing = smoothing    # 平滑强度 C（见 Task 7）
        self.noise = noise            # 加入高斯噪声的标准差（见 Task 6）
        self.cv = cv                  # 若给定，训练时用 cv 折 out-of-fold
        self.random_state = random_state
```

⚠️ **注意 `__init__` 里一行计算都没有**。这是 sklearn 的规矩：`__init__` 必须只做赋值，否则 `clone()` 和 `GridSearchCV` 会出诡异问题。

```python
    def _fit_maps(self, frame, target):
        global_mean = target.mean()                     # 全局正类比例（约 0.113）
        maps = {}
        for col in self.cols_:
            stats = target.groupby(frame[col]).agg(["sum", "count"])   # 每个类别的 1 的个数 和 样本数
            enc = (stats["sum"] + self.smoothing * global_mean) / (stats["count"] + self.smoothing)
            maps[col] = enc
        return maps, global_mean
```

- `target.groupby(frame[col])`：按这一列的取值分组。
- `.agg(["sum", "count"])`：每组求「y 的和」（= 正类个数）和「y 的个数」（= 该类别样本数）。
- `enc` 就是平滑公式：$\dfrac{\text{sum} + C\cdot \overline{y}}{\text{count} + C}$。当 `smoothing=0` 时退化成 $\frac{\text{sum}}{\text{count}}$，即纯比例。
- **`maps[col]` 是一个 pandas Series，索引是类别名，值是编码值。** 你可以把它理解成一张「查表字典」。

```python
    def fit(self, X, y=None):
        if self.cols is None:
            self.cols_ = [c for c in X.columns if X[c].dtype == object or ...]
        else:
            self.cols_ = list(self.cols)
        self.maps_, self.global_mean_ = self._fit_maps(X, pd.Series(np.asarray(y), index=X.index))
        return self
```

- `cols_`（带下划线）= `fit` 之后才存在的属性。`cols`（不带）= 构造函数的参数。**这个命名区分是 sklearn 的强约定**，不是风格问题。
- `pd.Series(np.asarray(y), index=X.index)`：把 `y` 包装成索引和 `X` 对齐的 Series，这样 `groupby` 才能正确配对。如果 `y` 的索引和 `X` 不一致，`groupby` 会静默算错——这是很隐蔽的一类 bug。

```python
    def _apply(self, frame, maps, global_mean, rng=None):
        out = frame.copy()                              # ① 不修改输入！
        for col in self.cols_:
            vals = out[col].map(maps[col]).astype(float)   # ② 查表
            vals = vals.fillna(global_mean)                # ③ 没见过的类别 -> 全局均值
            if self.noise and rng is not None:             # ④ 加噪声（只在训练时）
                vals = vals + rng.normal(0, self.noise, size=len(out))
            out[col] = vals
        return out
```

四行对应四个要点：
1. `frame.copy()` —— 作业明确要求「不能原地修改传入的样本」。不加 `copy`，调用者的 DataFrame 会被悄悄改掉，这种 bug 极难排查。
2. `.map(Series)` —— 用「类别名 → 数值」的映射表替换。
3. `.fillna(global_mean)` —— 测试集可能出现训练时没见过的类别，`map` 会给出 `NaN`。填全局均值是标准做法。
4. 噪声**只在训练时**加（这是我对原解答的修正：原实现 `transform` 里也加噪声，等于在测试时往特征里注入随机扰动，只会降低精度）。

```python
    def fit_transform(self, X, y=None, **fit_params):
        y = pd.Series(np.asarray(y), index=X.index)
        self.fit(X, y)
        rng = np.random.RandomState(self.random_state)
        if not self.cv:
            return self._apply(X, self.maps_, self.global_mean_, rng)   # 泄漏版（Task 5）
        # ---- out-of-fold 版（加分项）----
        oof = {col: np.full(len(X), np.nan) for col in self.cols_}
        skf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=self.random_state)
        for tr, te in skf.split(X, y):
            maps, gm = self._fit_maps(X.iloc[tr], y.iloc[tr])   # 只用其余折算统计量
            part = self._apply(X.iloc[te], maps, gm, rng)
            for col in self.cols_:
                oof[col][te] = part[col].to_numpy(dtype=float)
        data = {c: (oof[c] if c in self.cols_ else X[c].to_numpy()) for c in X.columns}
        return pd.DataFrame(data, index=X.index)
```

这是整个 Part 2 最绕的 20 行，拆开看：

- `if not self.cv` —— 没设 `cv` 就走「不加技巧」的路径（Task 5 用）。
- `oof = {...np.full(..., np.nan)}` —— 先建一张空表，等着被填。
- `skf.split(X, y)` 返回 5 组位置下标 `(tr, te)`。**注意是「位置」，所以要用 `.iloc`，不能用 `.loc`**（用错会按标签索引，静默出错）。
- `maps, gm = self._fit_maps(X.iloc[tr], y.iloc[tr])` —— **关键**：统计量只从 `tr`（其余 4 折）算。
- `part = self._apply(X.iloc[te], maps, gm, rng)` —— 用这套统计量编码 `te`（当前这折）。
- `oof[col][te] = ...` —— 把结果填回对应位置。5 轮之后，每个样本的编码都来自「没看过它」的统计量。
- 最后组装 `pd.DataFrame`：为什么不能直接 `out[col] = oof[col]`？因为在 pandas 3 里，往**字符串类型的列**里写浮点数会触发 dtype 冲突。重建 DataFrame 最稳。

> **一句话理解 out-of-fold**：「我要给你算一个特征，但这个特征不能用到你自己的标签。」做法是：把你的标签藏起来，用其他所有人的数据算出一个值给你。这样每个样本拿到的编码都是「别人眼里的它」，既保留了信息，又没有泄漏。

## Task 6 · 原文

> Note that such features are classifiers in themselves and, learning from them, we allow the target variable to "leak" into features. This leads to overfitting, so it is necessary to count such features in such a way that when calculating for a particular object, its target label is not used. This can be done in the following ways:
> 1. Calculate the counter value for all objects located higher in the dataset (for example, if we have a selection sorted by time).
> 2. Calculate by folds, that is, divide the sample into a certain number of parts and calculate the value of features for all folds except the current one (as is done in cross-validation).
> 3. Introduction of some noise into the calculated features.
>
> __Task 6.__ Implement correct calculation of counters in the simplest way - adding noise to the values (try to find a balance between fighting overfitting and keeping the features useful). Train logistic regression again, evaluate the quality. Draw your own conclusions.

## 🇨🇳 翻译

注意，这类特征**本身就是分类器**：学习它们等于让目标变量「泄漏」进了特征里，会导致过拟合。所以计算时必须保证：**算某个样本的编码时，不能用到它自己的标签**。有三种做法：

1. **按时间顺序**：只用排在当前样本之前（更早）的样本算。
2. **按折**：把样本分成若干折，用其余折算当前折的编码（和交叉验证一样）。
3. **加噪声**：往算出来的编码里注入随机扰动。

**任务 6**：用最简单的方式（加噪声）实现正确的计数器，并在「抑制过拟合」和「保留特征有效性」之间找平衡。重新训练逻辑回归，评估质量，并**得出你自己的结论**（这半句是作业的明确要求，问答题不写结论会被扣分）。

### 🧠 加噪声为什么能防泄漏

泄漏的本质是：编码值里含有**「这个样本自己的标签」的信息**，模型把这个信息当成规律学了。

加噪声的思路是：**往编码里掺一点随机成分，让「记忆单个样本」变得不可靠。** 如果某一类的样本很多，类均值本身很稳定，加一点点噪声几乎不影响它；如果某一类只有 1-2 个样本，均值就等于标签本身，噪声会把它打散，模型就记不住了。

噪声的标准差 σ 的选择是**权衡**：

| σ 太小（如 0.001） | σ 太大（如 0.5） |
|---|---|
| 打散不了记忆，泄漏还在 | 把有意义的信号也搅乱了，特征变废 |

### 💻 实测（本次运行）

代码：

```python
for sigma in [0.0, 0.01, 0.03, 0.05, 0.1, 0.2]:
    pipe = counter_pipe(noise=sigma).fit(X_train, y_train)
    ap_tr = average_precision_score(y_train, pipe.predict_proba(X_train)[:, 1])
    ap_te = average_precision_score(y_test,  pipe.predict_proba(X_test)[:, 1])
```

结果（train 与 test 都打印，这是判断过拟合的关键）：

| noise σ | train AP | test AP |
|---|---|---|
| 0.00 | 0.43795 | 0.44268 |
| 0.01 | 0.43722 | 0.44204 |
| 0.03 | 0.43327 | 0.43995 |
| 0.05 | 0.42890 | 0.43709 |
| 0.10 | 0.42471 | 0.43255 |
| 0.20 | 0.42100 | 0.42893 |

### ⚠️ 一个必须如实说明的结果

看第一行：**train AP (0.43795) 比 test AP (0.44268) 还低**。也就是说——**这里根本没有出现泄漏导致的过拟合**。

这不是代码写错了，而是这个场景的真实情况。原因有三：

1. **每个类别有几百个样本**。`job` 有 12 类、30k 行，平均每类 2500 个样本。2500 个样本的平均值，几乎不含「某一个样本自己的标签」的信息（1/2500 的权重）。
2. **逻辑回归的容量太小**。它给每一列**只有一个系数**。就算编码里混进了一点泄漏，它也没法利用它去「逐点记忆」——能记忆的是树模型或者 1-NN。
3. **泄漏的收益与类别基数成正比**。真正危险的是「基数接近样本数」的特征（用户 ID、订单号、邮编）。本数据集的类别基数很小。

所以结论是：**加噪声没有带来任何提升，σ 越大越差**（test AP 从 0.44268 一路掉到 0.42893）。噪声只是在本来就已经干净的特征上白添了随机性。

> 这个结果恰好印证了作业最后那段话：*「Complicating methods for calculating counters does not guarantee better model results.」* —— 方法越复杂不等于结果越好，你必须先判断「问题到底存不存在」。

> **答案（写进作业）**：train 与 test 的 AUC-PR 几乎相等（0.4216 vs 0.4318），说明在这个数据集上「不加技巧」的计数器并没有造成可见的过拟合——类别基数小（每类数百样本）、逻辑回归容量低（每列一个系数），泄漏的信息无法被利用。因此加噪声在本实验中不产生收益，σ ≥ 0.1 时甚至轻微变差。噪声只在「高基数 + 高容量模型」的场景里才是必要的解药。

### 📄 修正版结论格全文翻译（solved 版 Conclusion Task 6）

> **Conclusion (Task 6).** Two things in this table are worth stating honestly.

**翻译**：**结论（任务 6）**：这张表里有两件事值得如实说明。

> **1. There is no visible leak here.** Train AUC-PR (≈0.421) is not higher than test AUC-PR (≈0.432) — the usual signature of target-encoding leakage (high train / low test) does not appear at all. The reason is structural:
>
> * every category has hundreds of objects (`job` has 12 values over 31k rows ≈ 2,500 objects each), so the category mean carries only ~1/2500 of any single object's own label — the leak is diluted to nothing;
> * logistic regression has **one coefficient per column**; even if a little label information entered the encoding, the model has no capacity to memorise individual rows with it. Leakage of this kind bites tree models and nearest-neighbour methods, not a 19-coefficient linear model.

**翻译**：**1. 这里看不到泄漏。** 训练集 AUC-PR（≈0.421）并不比测试集（≈0.432）高——目标编码泄漏的典型症状（train 高 / test 低）完全没有出现。原因是结构性的：
- 每个类别有数百个样本（`job` 有 12 个取值、3.1 万行，平均每类约 2500 个），所以类均值里只含约 1/2500 的「单个样本自身标签」——泄漏被稀释到可以忽略；
- 逻辑回归**每列只有一个系数**；就算编码里混进了一点标签信息，模型也没有容量去「逐点记忆」。这类泄漏咬伤的是树模型和最近邻方法，咬不动一个只有 19 个系数的线性模型。

> **2. Therefore noise does not help.** The best σ is the smallest one; σ ≥ 0.1 makes test AUC-PR slightly worse (0.4318 → 0.4297), because the noise is destroying a feature that was already clean.
>
> **When would noise (or out-of-fold) actually be needed?** When the cardinality is close to the number of rows — user id, order id, postal code, or `age` treated as a category with 78 values on 400 rows. Then the raw encoding is literally "the object's own label", and the model memorises it.

**翻译**：**2. 所以噪声帮不上忙。** 最好的 σ 是最小的那个；σ ≥ 0.1 反而让 test AUC-PR 轻微变差（0.4318 → 0.4297），因为噪声破坏的是一个本来就干净的特征。**什么时候才真正需要噪声（或 out-of-fold）？** 当基数接近样本数时——用户 ID、订单号、邮编，或者只有 400 行却把 78 个取值的年龄当类别。那种情况下原始编码字面意义上就是「样本自己的标签」，模型会把它背下来。

> This matches the note at the end of the assignment: *"Complicating methods for calculating counters does not guarantee better model results."* The engineering conclusion is to **check whether the problem exists before solving it** — compare train and test scores first, and only then reach for smoothing / OOF / noise.

**翻译**：这与作业末尾的提示一致：*「把计数器复杂化并不保证结果更好」*。工程上的结论是：**先确认问题存在，再去解决它**——先比较 train 和 test 的分数，再决定要不要上平滑 / out-of-fold / 噪声。

## 加分项：用第 1 或第 2 种方式实现正确的计数器

## 📖 原文

> __(Bonus part)__ Calculate the correct counters in the first or second way described above (don't forget to add noise).

## 🇨🇳 翻译

**（加分）** 用上面第 1 或第 2 种方式实现正确的计数器（别忘了加噪声）。

### 🧠 第 2 种方式（按折）的完整流程

```
训练集 30k 行
   ├─ 切成 5 折: [1][2][3][4][5]
   ├─ 折 1 的样本 ← 用 2,3,4,5 折算统计量
   ├─ 折 2 的样本 ← 用 1,3,4,5 折算统计量
   ├─ ...
   └─ 拼起来 = 训练特征（每个值都是「别人眼里的它」）

测试集 → 用「全部训练集」算出的统计量（fit 时存下的 maps_）
```

注意最后一行很重要：**预测时必须用全量统计量**（更多数据 = 更准），不能也搞成 5 折然后取平均。`fit_transform`（训练用）和 `transform`（预测用）走不同路径，正是为此。

结果（本次运行）：AUC-PR = 0.44016，与「不加技巧」的 0.44268 几乎一样。再次说明这个数据集上泄漏问题不存在。

## Task 7 · 原文

> Now answer the following question: what happens if a certain category occurs in the sample only a few times? For this reason, counters are smoothed. For example, in practice, good results are shown by using smoothing by means of the entire sample:
> $$g_j(x,X) = \frac{\sum_i[f_j(x)=f_j(x_i)][y_i=+1] + C\times global\_mean}{\sum_i[f_j(x)=f_j(x_i)] + C}$$
> where $global\_mean$ is the proportion of objects of a positive class in the sample, $C$ is a parameter that determines the degree of smoothing (for example, you can use 10 or choose your own for each feature). The main idea is that we "dilute" the average for some category with the global average. And the less, the more objects of this category are found in the sample.
> However, for smoothing, instead of the average value of the target variable, you can use any other value from 0 to 1 (this parameter is sometimes called $prior$). You can make several features with different parameter values. In practice, even negative values can be useful in binary classification problems!
>
> __Task 7.__ Add the smoothing described above and repeat the experiments.

## 🇨🇳 翻译（逐句）

先回答一个问题：**如果某个类别在样本里只出现了几次，会发生什么？**（答案：它的编码 = 那一两个样本的标签均值，极不稳定，要么是 0 要么是 1，几乎就是噪声。）正因如此，计数器需要**平滑（smoothing）**。实践中效果不错的一种做法是用整个样本的均值来平滑：

$$g_j(x,X) = \frac{\sum_i[f_j(x)=f_j(x_i)][y_i=+1] + C\times global\_mean}{\sum_i[f_j(x)=f_j(x_i)] + C}$$

其中 $global\_mean$ 是样本中正类所占比例，$C$ 是决定平滑程度的参数（比如可以用 10，也可以为每个特征单独选）。核心思想是：用全局均值去「稀释」某个类别的均值；该类别在样本里出现的次数越少，稀释得越厉害。

不过，用来平滑的值不一定是标签均值，可以是 0 到 1 之间的任意数（这个参数有时叫 $prior$，先验）。你甚至可以造几个不同参数值的特征（比如同一列分别用 prior = 0.1、0.5、0.9 平滑，得到三个新特征）。实践中，二分类问题里**负值**有时也有用！

**任务 7**：加入上面描述的平滑，重复实验。

> 「负值先验为什么有用」：它相当于告诉模型「这个类别的成交率**低于**基准」。当全局均值只有 0.11 时，一个 prior = −0.5 的特征会把所有编码往下压，相当于给线性模型提供了一个不同「刻度」的同一信息——多一个角度的特征，线性模型有时能利用上。

### 🧠 平滑公式逐项拆解

$$g = \frac{\text{正类个数} + C\cdot \overline{y}}{\text{样本数} + C}$$

把它看成「**给这个类别额外塞进 C 个假样本，这些假样本的正类比例恰好等于全局比例**」：

- 类别样本数 = 2500，$C=10$：假样本只占 $\frac{10}{2510} = 0.4\%$，几乎不影响。
- 类别样本数 = 2，$C=10$：假样本占 $\frac{10}{12} = 83\%$！编码几乎完全被拉向全局均值。

**两个极限，用来检查你有没有理解：**

| 情况 | 结果 |
|---|---|
| $C \to 0$ | $g \to \frac{\text{正类个数}}{\text{样本数}}$，纯比例（会泄漏） |
| $C \to \infty$ | $g \to \overline{y}$，所有类别变成同一个数，**这一列彻底没有信息了** |

所以 $C$ 不是越大越好，它是在「稳定性」和「信息量」之间的滑块。

### 💻 实测（本次运行）

| smoothing C | test AP |
|---|---|
| 0.0 | 0.44016 |
| 1.0 | 0.44022 |
| 5.0 | 0.44023 |
| 10.0 | 0.44025 |
| 50.0 | 0.43867 |

最好的在 $C=10$ 附近，但**整个区间只差 0.002**，全在噪声范围内。这是常见现象：当类别样本数本来就很多时，平滑与否几乎无差别。

> **答案（写进作业）**：平滑在 $C \approx 10$ 处取得最好结果（0.44025 vs 不平滑的 0.44016），但差距只有 0.0001。平滑真正的作用场景是**稀有类别**：一个只出现 1 次的类别，不平滑时编码就是它自己的标签（0 或 1，极端值），平滑后被拉回全局均值 0.11 附近，模型不会再被这个极端值误导。$C$ 太大（如 50）则所有类别趋同、信息归零，反而变差。

### 📄 修正版结论格全文翻译（solved 版 Conclusion Task 7）

> **Conclusion (Task 7).** Smoothing helps exactly where the raw counter is unreliable — on rare categories. A category with `count = 1` gets $(y + C\cdot \overline{y})/(1+C)$ instead of $y$ itself: with $C = 10$ and $\overline{y}\approx 0.11$ the value moves from "0 or 1" to ≈ 0.10..0.19, i.e. towards the global mean. As $C \to \infty$ every category collapses to the global mean and the feature becomes useless; as $C \to 0$ we are back to the leaky version. The optimum is a compromise, and on this data it is flat — the difference between neighbouring values of $C$ is within the noise.

**翻译**：**结论（任务 7）**：平滑恰好只在原始计数器不可靠的地方起作用——**稀有类别**上。一个 `count = 1` 的类别，编码从「$y$ 本身（0 或 1）」变成 $(y + C\cdot\overline{y})/(1+C)$：取 $C=10$、$\overline{y}\approx 0.11$ 时，数值从「0 或 1」移到约 0.10~0.19，也就是被拉向全局均值。$C \to \infty$ 时所有类别都塌缩成全局均值，特征彻底失去作用；$C \to 0$ 则退回会泄漏的原始版本。最优点是个折中——而在这份数据上这个折中是「平」的：相邻的 $C$ 之间的差距都在噪声范围内。

## Task 8 · 原文

> The data contains the feature "client's age". Now we interpret it as a numerical one, which in the general case for a linear model may be an incorrect hypothesis. However, this feature has quite a few unique values (how many?), so applying one-hot encoding to it may be redundant. Try to encode age with counters. Has it gotten better?

## 🇨🇳 翻译

数据里有「客户年龄」这个特征。现在把它当数值用——但对线性模型来说，这个假设一般可能是错的。不过这个特征的**不同取值相当多**（有多少个？），所以对它做 one-hot 可能太浪费。试着用计数器编码年龄。有变好吗？

### 🧠 为什么「年龄当数值」是个可疑的假设

线性模型假设：**年龄每增加 1 岁，对数几率的变化量恒定**（$\beta$ 不变）。也就是说，18→19 岁的影响 = 60→61 岁的影响，且方向相同。

但现实显然不是这样。银行营销数据里，年龄和成交率的关系通常是 **U 型或倒 U 型**：学生和退休人员有时间听你讲，中年上班族没空。这种关系怎么用一条直线拟合？拟合不了。

**答案**：年龄有 **78 个不同取值**（17 到 98）。one-hot 会多出 78 列，而计数器只多 1 列——所以用计数器是合理的选择。

### 💻 实测（本次运行）

| 年龄的处理方式 | AUC-PR |
|---|---|
| 当数值（baseline） | 0.44025 |
| 当类别 + 计数器 | 0.43573 |

结果基本持平（甚至略降）。原因：每个年龄值平均有 400 个样本，计数器的统计量很稳；但「年龄的效应」本来就不强，而且线性项已经抓住了主要趋势（整体上是弱负相关）。**不要把「理论上更合理」等同于「指标一定更好」**——这是这门课想反复强调的态度。

### 📄 修正版答案格全文翻译（solved 版 Task 8 Answer）

> **Answer.** Age has ~77 distinct integer values (17..98), so one-hot would add 77 mostly-empty columns, while a counter adds just one. The counter is also semantically right: it says "clients of this age convert with probability q", which is not a linear function of age at all (the dependence is U-shaped: very young and very old clients behave differently).
>
> The measured effect is tiny and can go either way — the linear "age as a number" term already captures the main trend, and the counter, being computed on ~40k rows with ~500 objects per age value, is stable but adds little. Do not expect a jump in quality here; the point of the exercise is that both encodings are *possible*, and the "right" one is a modelling decision, not a law.

**翻译**：**答案**：年龄有约 77 个不同的整数值（17 到 98），所以 one-hot 会增加 77 列几乎全空的列，而计数器只加 1 列。计数器在语义上也更贴切：它表达的是「这个年龄的客户以概率 q 成交」，而这**根本不是年龄的线性函数**（依赖关系是 U 形的：很年轻和很老的客户行为不一样）。实测效果很小、可正可负——线性的「年龄当数值」项已经抓住了主要趋势，而计数器在 4 万行上计算（每个年龄约 500 个样本）虽然稳定，但带来的增量很小。别指望这里会有质量飞跃；这个练习的意义在于：**两种编码都可行，选哪个是建模决策，不是定律。**

## 反向操作：把 month / day_of_week 当数字

## 📖 原文

> You can also go in the opposite direction. We have signs "month and day of the week" for the call. Try to interpret them as numbers (month 0 to 12, days of the week 0 to 4). Did it get better this time?

## 🇨🇳 翻译

也可以反着来。我们有「通话月份」和「星期几」这两个特征。试着把它们当数字解释（月份 0~12，星期 0~4）。这次有变好吗？

### 💻 实测（本次运行）

```python
month_map = {m: i for i, m in enumerate(['jan','feb',...,'dec'])}
dow_map   = {d: i for i, d in enumerate(['mon','tue','wed','thu','fri'])}
```

| month / day_of_week 的处理 | AUC-PR |
|---|---|
| 当类别 + 计数器 | 0.44025 |
| 当数字（0~11 / 0~4） | 0.43142 |

**明显变差**（约 −0.014）。为什么？

「当数字」隐含了一个很强的假设：**月份越大，效果越单调**（或者单调递减）。但这份数据里：

- 3 月、9 月、10 月、12 月是**好月份**，5 月**很差**——这是非单调的。
- 而计数器把每个月的「实际成交率」直接编码成数字，任意形状都能表达，所以更合适。

这也回答了一个常见疑问：「`month` 明明是有顺序的（1 月 < 2 月），为什么当数字反而更差？」——因为**顺序存在 ≠ 效应单调**。有顺序只保证「可以编码成数字而不荒谬」，不保证「线性模型用得好」。

### 📄 修正版答案格全文翻译（solved 版「go in the opposite direction」Answer）

> **Answer.** Treating `month` and `day_of_week` as numbers is a *strong* assumption: it claims that `mar` is "less than" `apr` and that the distance between `jan` and `feb` equals the distance between `nov` and `dec`. For a linear model this means one monotone trend across the year, which is wrong — in this dataset March, September, October and December are good months and May is terrible, a non-monotone pattern.
>
> So numerically the result is usually a small *loss* compared with counters, which can express any pattern per category. The exception: when the calendar order really carries information (seasonality, "closer to the campaign date"), a monotone term is a useful regularizer. Both variants are legitimate; what matters is that you know which hypothesis you are imposing.

**翻译**：**答案**：把 `month` 和 `day_of_week` 当数字是一个**很强的假设**：它声称 `mar`「小于」`apr`，并且 `jan` 到 `feb` 的距离等于 `nov` 到 `dec` 的距离。对线性模型来说，这意味着全年只有**一条单调趋势**——而这是错的：在这份数据里，3 月、9 月、10 月、12 月是好月份，5 月很糟，是非单调的模式。所以数值结果通常比计数器**略差**——计数器能为每个类别表达任意形状。例外情况：当日历顺序真的携带信息时（季节性、「离活动日期更近」），单调项是个有用的正则化手段。两种做法都合法；重要的是**你知道自己正在强加哪种假设**。

## Part 2 总结：计数器 vs one-hot

## 📖 原文

> __Note.__ Complicating methods for calculating counters does not guarantee better model results. Especially considering that logistic regression is not such a complex model to retrain. So you didn't have to get better and better results at each step (but you should have got the necessary results).
>
> As you should have noticed, counters are a good alternative to one-hot coding. Write what pros and cons of using counters compared to one-hot coding you noticed.

## 🇨🇳 翻译

**注意**：把计数器的计算方法复杂化，并不保证模型结果更好，尤其是考虑到逻辑回归本身并不算复杂的模型、不容易过拟合。所以你在每一步**不一定**都该得到越来越好的结果（但你应该得到必要的结果）。

如你所见，计数器是 one-hot 的一个不错的替代方案。写下你注意到的、计数器相对于 one-hot 的优缺点。

### 📊 对比表（背下来，考试和面试都爱问）

| | 计数器 / 目标编码 | one-hot |
|---|---|---|
| **特征数** | 1 列 / 每个类别特征 | $m$ 列 / 每个类别特征 |
| **内存/速度** | ✅ 省 | ❌ 高基数时爆炸 |
| **编码信息量** | ✅ 自带「成交率」的排序含义 | ✅ 每个类别独立权重，能表达任意形状 |
| **过拟合风险** | ❌ 有目标泄漏，需要平滑/OOF/噪声 | ✅ 无（不看标签） |
| **新类别** | ⚠️ 没见过就只能填全局均值 | ✅ 全 0 即可，安全 |
| **可解释性** | ⚠️ 「0.13」混了类别效应和基准率 | ✅ 「`job=admin.` 的系数是 +0.4」很清楚 |
| **小样本** | ❌ 稀有类别编码极不稳定 | ✅ 稳定（虽然系数会很吵） |
| **能否表达交互** | ❌ 只有单变量平均效应 | ⚠️ 也不行，但至少每类独立 |

**什么时候用哪个：**

- 类别数 **少**（< 几十）→ **one-hot**，稳、无泄漏。
- 类别数 **多**（邮编、user_id、商品 ID，成千上万）→ **计数器**，one-hot 会在内存上爆炸。
- 用计数器就**必须**配套：out-of-fold + 平滑，这是标配，不是可选。
- 两者可以**混用**：低基数的用 one-hot，高基数的用计数器。真实比赛里最常见的就是这种混合。

> 本数据集的实验结论：one-hot（0.44998）略优于计数器（0.44268），因为类别基数小、样本充足，one-hot 的维度代价不成问题。这正好说明：**方法选择要看数据，不看流行度。**

### 📄 修正版「优缺点」长答案全文翻译（solved 版 Part 2 结尾 Note + Answer）

> **Counters (target encoding) — advantages:**
>
> * **Compact**: one column per categorical feature instead of dozens/hundreds of one-hot columns — less memory and a much shorter coefficient vector.
> * **Cheaper on high cardinality**: encoding is a `groupby`, and the model sees 19 features instead of 62. At *this* scale (31k rows, 62 columns) the wall-clock difference is inside the noise (0.15 s vs 0.16 s), but with a zip-code column one-hot would produce tens of thousands of columns, and the gap becomes decisive.
> * **Carries an "ordering"**: a category with a high conversion rate gets a high value, which a single one-hot coefficient cannot express.
> * **No dimension explosion in the coefficient vector** — important for interpretation and for memory-constrained deployment.

**翻译**：**计数器（目标编码）的优点：**
- **紧凑**：每个类别特征只占 1 列，而不是几十上百列 one-hot——内存更省，系数向量更短。
- **高基数时更便宜**：编码就是一次 `groupby`，模型只看到 19 个特征而不是 62 个。在**本**规模（3.1 万行、62 列）下耗时差异在噪声内（0.15 s vs 0.16 s），但如果有一列邮编，one-hot 会造出几万列，差距就是决定性的了。
- **自带「排序」**：成交率高的类别得到更高的值——单个 one-hot 系数表达不了这种含义。
- **系数向量不发生维度爆炸**——对模型解释和内存受限的部署环境很重要。

> **Counters — disadvantages:**
>
> * **Target leakage / overfitting**: the encoding of a category is computed from the target of the very same rows. A category seen once gets an encoding equal to its own label (0 or 1), and the model memorises it. Such categories never appear in the test set, so the quality drops.
> * **Needs extra machinery**: noise, smoothing, out-of-fold computation — exactly Tasks 6, 7 and the bonus. Each of them is a hyperparameter to tune.
> * **Unstable for unseen categories**: a new category has no counter at all and falls back to the global mean, so the feature carries no signal for new categories.
> * **Less interpretable**: "job=admin has value 0.13" mixes the category effect with the base rate, whereas "job=admin has coefficient +0.4" is clean.
> * **Only a univariate summary**: it compresses the whole category into one number, so it cannot express interactions ("for clients under 30, `admin.` behaves differently").

**翻译**：**计数器的缺点：**
- **目标泄漏 / 过拟合**：类别的编码是用**同样这些行**的标签算出来的。只出现过一次的类别，编码就等于它自己的标签（0 或 1），模型会把它背下来。而这样的类别在测试集里全是「新面孔」，质量立刻掉。
- **需要额外的配套机制**：噪声、平滑、out-of-fold——正是 Task 6、Task 7 和加分题。每一个都是要调的超参数。
- **对没见过的类别不稳定**：新类别没有计数器，只能退回全局均值，这个特征对新类别不携带任何信号。
- **可解释性较差**：「`job=admin` 的编码值是 0.13」把类别效应和基准率混在一起；而「`job=admin` 的系数是 +0.4」干干净净。
- **只是单变量摘要**：它把整个类别压缩成一个数，无法表达交互（「30 岁以下的客户里，`admin.` 的表现不一样」）。

> **One-hot — advantages:** no leakage; handles unseen categories (all zeros); can learn an arbitrary shape of dependence per category; stable on small samples. **Disadvantages:** blows up the number of columns, needs more memory and time, and with rare categories most of those columns are almost always zero (pure noise for the model).

**翻译**：**one-hot 的优点**：无泄漏；能处理新类别（全填 0）；能为每个类别学习任意形状的依赖；小样本上稳定。**缺点**：列数爆炸，更费内存和时间；类别稀少时，那些列几乎恒为 0（对模型来说就是纯噪声）。

> **Measured on this dataset:** one-hot 0.44998 vs counters 0.44268 — one-hot wins here, because the cardinality is small (at most 12 values) and 62 columns is nothing. That is the honest conclusion: **counters are the alternative for high-cardinality features, not a universal upgrade.**
>
> **In practice:** for high-cardinality features (zip code, user id, product id) counters are the only workable option; for low-cardinality ones one-hot is safer. The best results usually come from a mix, with counters computed out-of-fold and smoothed.

**翻译**：**本数据集实测**：one-hot 0.44998 vs 计数器 0.44268——one-hot 胜出，因为基数小（最多 12 个取值）、62 列根本不算事。这才是诚实的结论：**计数器是高基数特征的替代方案，不是普适的升级**。**实践中**：高基数特征（邮编、用户 ID、商品 ID）计数器是唯一可行选项；低基数特征 one-hot 更安全。最好的结果通常来自**混用**，且计数器要 out-of-fold 计算并加平滑。

---
# 4. Part 3：特征选择（2 分）

## 📖 原文

> Feature selection is an important part of the model building process. In practice, many features have little effect on the model (and their calculation takes time) or even negatively affect the quality of the model. Let's try several feature selection approaches, evaluate how they affect the quality of the model and how long they take.
>
> Let's turn to the same dataset about bank telephone marketing.

## 🇨🇳 翻译

特征选择是建模流程中很重要的一环。实践中很多特征对模型几乎没影响（计算它们还要花时间），甚至会**负面影响**模型质量。我们来试几种特征选择方法，评估它们对模型质量的影响，以及各自要花多长时间。

继续用之前那个银行电话营销数据集。

> 一句话：**特征不是越多越好。** 冗余特征会让模型更容易过拟合、更难解释、算得更慢。

## 先把数据编成 one-hot（并且用稀疏矩阵）

> **作业原文**：As you remember, there are many categorical features in the data (for now, let's interpret age as a number). Let's encode them using one-hot encoding. The original columns with categorical features can be deleted. How many signs did we get?
>
> **翻译**：如你所记得的，数据里有很多类别特征（目前先把年龄当数值）。用 one-hot 编码它们，编码后原来的类别列可以删掉。**我们一共得到了多少个特征？**（答案：62 个——9 个数值列 + 53 个 one-hot 指示列。）

```python
cat3 = [c for c in X.columns if X[c].dtype == object or str(X[c].dtype) in ("str", "string")]
num3 = [c for c in X.columns if c not in cat3]

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)   # 注意 True
Xtr_cat = ohe.fit_transform(X_train[cat3])
Xte_cat = ohe.transform(X_test[cat3])

Xtr = sp.hstack([sp.csr_matrix(X_train[num3].to_numpy()), Xtr_cat]).tocsr()
Xte = sp.hstack([sp.csr_matrix(X_test[num3].to_numpy()), Xte_cat]).tocsr()
```

| 名字 | 作用 |
|---|---|
| `scipy.sparse`（简写 `sp`） | 稀疏矩阵模块。**只存非零元素**，one-hot 里 95% 都是 0，用它能省 95% 内存 |
| `sp.csr_matrix(arr)` | 把普通 numpy 数组转成 CSR 格式（按行压缩，最常用的稀疏格式） |
| `sp.hstack([A, B])` | 横向拼接两个矩阵（列数相加） |
| `.tocsr()` | 转成 CSR 格式（`hstack` 之后格式可能变化，统一一下） |
| `sparse_output=True` | `OneHotEncoder` 直接返回稀疏矩阵 |
| `ohe.get_feature_names_out(cat3)` | 拿到编码后的列名，比如 `job_admin.`、`month_may` |

⚠️ `fit_transform` 只在**训练集**上调用，测试集只能 `transform`（复用训练集的类别表）。写成 `ohe.fit_transform(X_test)` 是初学者最常见的泄漏错误之一。

**结果**：62 个特征（9 个数值 + 53 个 one-hot 指示变量）。原始是 19 列。

> 本次数据只有 62 列，用密集矩阵其实也放得下（33k × 62 × 8 字节 ≈ 16 MB）。但**必须学会稀疏矩阵的写法**——如果特征是「邮编」（几千个取值）或「商品 ID」（几万个取值），one-hot 之后就是几十万列、99.99% 是 0，密集矩阵直接内存溢出。

## 基线：用全部特征训练

> **作业原文**：We will use logistic regression as the main model, and `AUC-PR` as the target metric. Train the model and calculate the quality on the test set. Let's remember the resulting value.
>
> **翻译**：我们用逻辑回归作为主模型、`AUC-PR` 作为目标指标。训练模型并计算测试集质量。**记住这个数值**（它是后面所有特征选择方法的对照基线）。

```python
def fit_eval(Xa, ya, Xb, yb, C=1.0):
    model = LogisticRegression(max_iter=1000, C=C, random_state=42).fit(Xa, ya)
    score = model.predict_proba(Xb)[:, 1]
    return model, average_precision_score(yb, score), score
```

这个小函数后面会被反复调用（8 次以上），它的作用就是「训练 + 打分 + 把模型和分数都返回」。返回模型是因为后面要用 `model.coef_` 做嵌入式选择；返回分数是因为可能要画曲线。

**基线结果：全部 62 个特征，AUC-PR = 0.44786，耗时 0.13 s。**

⚠️ 注意这里**故意没有标准化**。Part 3 的第一半就是要让你看到「不标准化会让权重选择和特征选择都失真」，然后在下半段修好它。这是作业设计的叙事线。

## 三大流派（先记住这张表，细节往后再看）

| 流派 | 怎么判断特征重要性 | 要不要训练模型 | 计算量 | 典型方法 |
|---|---|---|---|---|
| **Embedded（嵌入式）** | 模型训练完自带的重要性 | 只训 1 次 | 极小 | 线性模型 `\|coef\|`、树模型 `feature_importances_`、L1 正则 |
| **Filter（过滤式）** | 对每个特征单独算一个统计量 | 不训练模型 | 极小 | t 统计量、卡方、互信息、相关系数 |
| **Wrapper（包裹式）** | 反复训练模型，看指标变化 | 训练 **很多次** | 巨大 | 前向/后向逐步选择、递归特征消除 (RFE) |

> 类比：
> - **Filter** 像筛简历——先看学历、工作年限这些单项指标，不管岗位具体需求。快，但会漏掉「单项不突出但组合起来很强」的人。
> - **Embedded** 像面试完让面试官自己打分——一次面试就拿到综合评价。便宜，但分数受面试官偏好（模型类型）影响。
> - **Wrapper** 像真的让人上岗试一个月——最准，但成本最高。

## Task 8（Part 3 版）：用 `|coef|` 保留 40 个特征

> ⚠️ 原始作业里有两个「Task 8」（Part 2 的年龄题和这里的嵌入式选择），是出题人的编号失误。这是 Part 3 的那个。

## 📖 原文（全文，含 Built-in methods 引入段）

> ### Built-in methods
>
> Let's say we want to keep only the top 40 features. Let's try to do this in several ways.
>
> Let's start with feature selection using a linear model. As you know, the weights of a linear model mean the contribution of each feature to the prediction of the model, which means that the modulus of this contribution can be interpreted as the importance of the features. Such a selection method is called built-in or embedded methods, since it is embedded in the features of the model.
>
> __Task 8.__ Leave 40 features with the largest modulus of the corresponding parameter of the linear model. Retrain the model and evaluate its quality. Measure the speed of such feature selection. Has the quality changed? How?

## 🇨🇳 翻译（逐句）

### 内置（嵌入式）方法

假设我们想只保留前 40 个特征。用几种不同的方式来做。

先从「用线性模型选特征」开始。如你所知，线性模型的权重代表每个特征对模型预测的贡献，因此这个贡献的**绝对值**可以解释为特征的重要性。这种选择方法叫**内置法 / 嵌入式（embedded）**——因为它内嵌在模型自身的产物里（训练完顺手就有了，不用额外计算）。

**任务 8**：保留「线性模型参数绝对值最大」的 40 个特征。重新训练模型并评估质量。测量这种特征选择的速度。质量变了吗？怎么变的？

```python
coefs = model_all.coef_[0]                            # 62 个权重
top40_emb = np.argsort(-np.abs(coefs))[:40]           # 按 |coef| 降序取前 40 个**下标**
model_emb, ap_emb, _ = fit_eval(Xtr[:, top40_emb], y_train, Xte[:, top40_emb], y_test)
```

| 名字 | 作用 |
|---|---|
| `model.coef_` | 形状 `(1, n_features)`（二分类），`coef_[0]` 取那一行权重 |
| `np.abs(coefs)` | 取绝对值——**权重为负也很重要**，重要的是「偏离 0 的幅度」 |
| `np.argsort(-x)` | 返回排序后的**下标**。加负号 = 降序（`argsort` 默认升序） |
| `[:40]` | 取前 40 个下标 |
| `Xtr[:, top40_emb]` | 稀疏矩阵的列切片，取出这 40 列 |

**结果：AUC-PR = 0.44524（全部特征时 0.44786，变化 -0.00262），选择耗时约 0.5 s。**

质量**略微下降**。两个原因：

1. **40 个 |coef| 最大的 ≠ 40 个最有用的**。特征之间相关时（还记得 `make_classification` 的 `n_redundant` 吗），信息被摊到多个特征上，每个的权重都不大——一个「有用但被复制过」的特征可能因此被丢掉。
2. **权重是在未标准化的数据上算的**，|coef| 里混着「量级」这个噪声。下一节就修这个。

### 📄 修正版答案格全文翻译（solved 版 Task 8 Answer）

> **Answer.** The quality drops slightly. Two reasons: (a) the 40 features with the largest |coef| are not necessarily the 40 most useful — with correlated features the weight is shared, so a duplicated-but-useful feature can show a small coefficient; (b) the weights themselves were obtained on **unscaled** features, so |coef| mixes usefulness with units. The next cell fixes exactly that.

**翻译**：**答案**：质量略微下降。两个原因：(a) |coef| 最大的 40 个特征不一定是最有用的 40 个——特征相关时权重会被分摊，一个「有用但被复制」的特征可能只显示很小的系数；(b) 这些权重本身是在**未标准化**的特征上算的，所以 |coef| 里混着「重要性」和「单位」两种信息。下一格正好修这个问题。

## 尺度陷阱：为什么必须先标准化再选特征

## 📖 原文

> Now let's think about what we didn't take into account. We assumed that the features contribute uniformly, but did not take into account their scale. If we multiply one of the features by 100 times, then without regularization, its weight will decrease by the same 100 times. And based on this, we select signs! Let's first scale the features in one of the ways, and only then we will remove the features.
>
> By the way, in this case, you need to recalculate the quality on all features (do it below). If you did feature normalization at the very beginning, then try to select features on unscaled data.
>
> What happened?

## 🇨🇳 翻译

现在我们想想漏掉了什么。我们假设所有特征的贡献是可比的，但没考虑它们的**量级**。如果把某个特征乘以 100，那么在没有正则化的情况下，它的权重会相应缩小 100 倍。而我们居然是**根据这个权重**来选特征的！所以我们先把特征按某种方式标准化，然后再删特征。

顺便说一句，这种情况下你需要**重新在全部特征上算一遍质量**（在下面做）——因为标准化本身就可能改变模型的表现，要和基线区分开。如果你一开始就做了标准化，那就反过来试试在未标准化的数据上选特征。

发生了什么？（答案见下面的实验表格。）

### 🧠 核心直觉

模型要拟合的是 $w_1x_1 + w_2x_2 + \dots$。如果 $x_1$ 的单位从「元」换成「分」（×100），模型只要把 $w_1$ 除以 100，预测结果一模一样。**所以 $|w|$ 的大小同时取决于「这个特征多重要」和「这个特征的单位多小」。**

看本数据集的实际量级：

| 特征 | 大概范围 |
|---|---|
| `age` | 17 ~ 98 |
| `campaign` | 1 ~ 56 |
| `euribor3m` | 0.6 ~ 5.1 |
| `nr.employed` | 4963 ~ 5228 |
| `cons.price.idx` | 92 ~ 95 |

`nr.employed` 的取值范围是 5000 级别的，只要它变动 0.001，就能抵消 `age` 变动 5 的效果。所以它的系数天然就很小很小，**在 |coef| 排序里永远排不上号**——不是因为它不重要，而是因为它的单位大。

> 这就是「用 |coef| 选特征」在未标准化数据上失效的根本原因。**不是 bug，是数学必然。**

### 💻 修法：`StandardScaler(with_mean=False)`

```python
scaler = StandardScaler(with_mean=False)     # 只除以标准差，不减均值
Xtr_s = scaler.fit_transform(Xtr)            # 注意：只在训练集上 fit
Xte_s = scaler.transform(Xte)
```

- `with_mean=False`：**不能减均值**。稀疏矩阵减均值会变成密集矩阵（0 减 0.3 就不是 0 了），内存瞬间爆掉。而且对本问题来说，「除以标准差」就足够消除量级差异了。
- `fit_transform` 在训练集，`transform` 在测试集。用测试集的均值/方差去标准化 = 泄漏。

**结果：**

| 做法 | AUC-PR |
|---|---|
| 全部特征，未标准化 | 0.44786 |
| 全部特征，已标准化 | 0.44778 |
| 标准化后取 top-40 | 0.44881 |
| 未标准化取 top-40 | 0.44524 |

两点观察：
1. **标准化对最终质量几乎没有影响**（0.44786 → 0.44778）。因为改用 `newton-cholesky` 求解器后，未标准化的那次也能收敛到真正的最优点，所以两者都拿到了「最优解」。
2. 但标准化**改变了 |coef| 的排序**：标准化后选出的 40 个特征，和未标准化选出的只有 **30 个重合**（共 40）。**十列换了人**，而质量几乎没变——这恰恰说明「特征重要性排序」在未标准化时是不可信的：它给出的名单里有一大半是单位效应，不是信息效应。

> 这个对比很有教育意义：**标准化不一定提高分数，但它让「哪些特征重要」这个结论变得可信。** 如果你要用模型做解释（而不只是预测），标准化是必须的。

## 追问：one-hot 出来的指示变量要不要也标准化

## 📖 原文

> A quick question: one-hot coding returns single indicator features to us. Try to scale them as well as regular numeric ones, and again select 40 main features by contribution. Has their list changed? Will the quality change?

## 🇨🇳 翻译

一个小问题：one-hot 编码给我们的是单列的指示（0/1）特征。试着把它们也像普通数值特征一样标准化，然后再按贡献选出 40 个主要特征。它们的名单变了吗？质量会变吗？

### 🧠 这题在问什么

`StandardScaler` 对 0/1 指示变量做的事是：除以它的标准差 $\sqrt{p(1-p)}$，其中 $p$ 是这一列 1 的比例。

- 如果 `job_admin.` 有 25% 的样本是 1，标准差 = $\sqrt{0.25\times0.75}=0.43$，标准化后这一列变成 0 或 2.3。
- 如果某个稀有类别只有 1% 是 1，标准差 = 0.0995，标准化后变成 0 或 10。

**所以标准化会放大小类别的影响**，让稀有类别的系数变小、权重可比。这是好事（消除「类别稀有度」对 |coef| 的干扰），也是坏事（稀有类别的估计本来就不稳，放大后噪声也放大了）。

代码上用 `ColumnTransformer` 只对数值列标准化、指示列原样保留：

```python
num_idx = list(range(len(num3)))                        # 前 9 列是数值
bin_idx = list(range(len(num3), Xtr.shape[1]))          # 后面都是指示列
ct = ColumnTransformer([
    ("num", StandardScaler(with_mean=False), num_idx),
    ("bin", "passthrough", bin_idx),
], sparse_threshold=1.0)
```

| 名字 | 作用 |
|---|---|
| `num_idx` / `bin_idx` | 列位置下标（不是列名，因为此时 `Xtr` 已经是 numpy/稀疏矩阵，没有列名了） |
| `sparse_threshold=1.0` | 拼接结果的稀疏度超过 100% 时仍返回稀疏矩阵（即：只要是稀疏输入就保持稀疏）。默认值 0.3 会让结果变密集 |

**结果：**

| 做法 | AUC-PR |
|---|---|
| 只标准化数值列，全特征 | 0.44780 |
| 只标准化数值列，取 top-40 | 0.44879 |

这是所有方案里**最好的一个**（0.44879）。解释：数值列需要标准化（消除量级），指示列不需要（它们的量级天然统一，都是 0/1，标准化反而会放大小类别的噪声）。**这就是「按特征类型分别处理」的现实理由。**

### 📄 修正版答案格全文翻译（solved 版「one-hot 要不要也标准化」Answer）

> **Answer.** Scaling changes the ranking of features and therefore the list of the top-40, and the quality improves a little. When *everything* is scaled, one-hot indicators become comparable with the numeric features; when only the numeric ones are scaled, the indicators keep their natural "1 unit" scale. Both variants beat the unscaled one, and the list of selected features differs — which is the practical lesson: **feature selection by |coef| is meaningless without a common scale**.

**翻译**：**答案**：标准化会改变特征的排序，因此 top-40 的名单会变，质量也略有提升。当**所有**特征都被标准化时，one-hot 指示变量和数值特征变得可比；当只标准化数值特征时，指示变量保持它们天然的「1 个单位」刻度。两种方案都优于完全不标准化，而且选出的特征名单不同——这就是实践教训：**没有统一刻度，按 |coef| 选特征就是没有意义的**。

## Task 9：过滤式方法（t 统计量）

## 📖 原文

> Let's select features smarter, namely through the calculation of some function for each feature. Based on the values of this function, we will leave the most important features. The methods of this family are called filter methods.
> As such a function, we will consider t-statistics: $$t(j) = \frac{|\mu_+ - \mu_-|}{\sqrt{\frac{n_+\sigma^2_+ + n_-\sigma^2_-}{n_+ + n_-}}}$$
> where $\mu$, $\sigma$, $n$ are the mean, standard deviation, and number of objects of each class, respectively.
> __Task 9.__ Leave 40 features with the highest $t$ value and measure the quality. Don't forget to measure the feature selection rate in this case.

## 🇨🇳 翻译

让我们更聪明地选特征：对每个特征算一个函数值，根据这个值保留最重要的特征。这一类方法叫**过滤式**方法。

作为这个函数，我们考虑 **t 统计量**：

$$t(j) = \frac{|\mu_+ - \mu_-|}{\sqrt{\frac{n_+\sigma^2_+ + n_-\sigma^2_-}{n_+ + n_-}}}$$

其中 $\mu$、$\sigma$、$n$ 分别是每个类的均值、标准差和样本数。

**任务 9**：保留 $t$ 值最高的 40 个特征，测量质量。别忘了也测量特征选择的速度。

### 🧠 公式逐项翻译成人话

$$t(j) = \frac{|\mu_+ - \mu_-|}{\sqrt{\text{合并方差}/(n_+ + n_-)}}$$

- **分子 $|\mu_+ - \mu_-|$**：这个特征在正类里的平均值，和负类里的平均值，差多少？差得越多，说明这个特征越能区分两类。
- **分母**：数据本身的波动有多大。波动大，说明「差值」可能只是运气，要打折扣。
- 所以 $t$ 大 = **「两类差距大，而且数据不乱」** = 这个特征靠谱。

> 它就是统计学里的「两样本 t 检验」的统计量。**注意它只看单变量**：完全不考虑特征之间的配合。这是 filter 方法的根本局限。

```python
Xtr_dense = Xtr.toarray()             # t 统计量的实现需要密集矩阵
mask_pos = y_train.to_numpy().astype(bool)

mu_1  = Xtr_dense[mask_pos].mean(axis=0)      # 正类均值（对每一列）
mu_0  = Xtr_dense[~mask_pos].mean(axis=0)     # 负类均值
var_1 = Xtr_dense[mask_pos].var(axis=0)
var_0 = Xtr_dense[~mask_pos].var(axis=0)
n_1 = int(mask_pos.sum()); n_0 = int((~mask_pos).sum())

t_stat = np.abs(mu_1 - mu_0) / np.sqrt((n_1 * var_1 + n_0 * var_0) / (n_1 + n_0))
top40_t = np.argsort(-t_stat)[:40]
```

| 名字 | 作用 |
|---|---|
| `.toarray()` | 稀疏 → 密集。⚠️ 会占内存（33k×62 还好，但高基数 one-hot 时千万别这么写） |
| `mask_pos` | 布尔数组，True 表示这一行是正类 |
| `Xtr_dense[mask_pos]` | 布尔索引，取出所有正类行 |
| `.mean(axis=0)` | `axis=0` = 沿行方向压扁 → 得到每一列（每个特征）的均值 |
| `.var(axis=0)` | 同上，算方差。注意 numpy 默认 `ddof=0`（除以 n），统计学教材常用 `ddof=1`（除以 n−1），差别在样本大时可忽略 |

**结果：AUC-PR = 0.45162，耗时约 0.06 s，与嵌入式选出的 40 个特征只重合 28 个。**

> **答案（写进作业）**：filter 方法极快（只扫一遍列，不训练任何模型），质量与嵌入式方法相当（0.45162 vs 0.44524）。它完全忽略特征交互——一个单独看毫无区分度、但和其他特征组合起来很有用的特征会被丢掉。但在特征数量极大（10⁵ 级）时，它是唯一可行的方案。

### 📄 修正版答案格全文翻译（solved 版 Task 9 Answer）

> **Answer.** The filter is not just the cheapest — here it is also the **best**: 0.45162 in 0.06 s, above both the embedded selection (0.44524) and the wrapper (0.44614). That is not a coincidence: 53 of the 62 columns are one-hot indicators, and many of them are pure noise (`default_unknown`, `job_unknown` — a few dozen rows each). A univariate t-statistic discards them immediately, while a regularised linear model has to spend some of its L2 budget on them.
>
> It ignores interactions, and on data where the signal is in the combinations the ranking would be wrong — but on 62 binary indicators with a linear model, "one feature at a time" is a surprisingly strong baseline.

**翻译**：**答案**：filter 不只是最便宜——在这里它还是**最好的**：0.06 秒拿到 0.45162，高于嵌入式选择（0.44524）和包裹式（0.44614）。这不是巧合：62 列里有 53 列是 one-hot 指示变量，其中许多是纯噪声（`default_unknown`、`job_unknown`——各自只有几十行）。单变量 t 统计量能立刻把它们丢掉，而带正则的线性模型却不得不在这些列上**花掉一部分 L2 预算**。它确实忽略特征交互——在「信号藏在特征组合里」的数据上，它的排序会是错的——但在「62 个二值指示列 + 线性模型」这个场景下，「一次只看一个特征」是出人意料地强的基线。

## Task 10：包裹式方法（贪心逐步选择）

## 📖 原文

> The third of the methods we are considering works as follows: we exclude one of the features in turn and see how this affects the quality. We remove the features in such a greedy way until some condition is met (the number of features or deterioration in quality).
> Note that it is impossible to evaluate the quality of the test sample, otherwise we can overfit, as, for example, when tuning hyperparameters. Divide the training sample into 2 parts, train the model without one of the features on one of them, evaluate the quality on the second part. The initial test sample should only be used for the final quality assessment.
> Again, leave only 40 features and evaluate the quality on the test sample. How long did this feature selection take?

## 🇨🇳 翻译

我们考虑的第三种方法是这样工作的：轮流排除其中一个特征，看它对质量有什么影响。用这种**贪心**方式不断删特征，直到满足某个条件（特征数量，或质量开始变差）。

注意：**不能用测试集来评估质量**，否则就会过拟合（就像调超参数时那样）。做法是把训练集再切成两份：在其中一份上训练「去掉某个特征」的模型，在另一份上评估质量。**最初的测试集只能用于最终的质量评估。**

同样只保留 40 个特征，并在测试集上评估质量。这次特征选择花了多长时间？

### 🧠 为什么「用测试集做特征选择」= 作弊

这是本作业最核心的**方法论**警告，值得单独讲。

假设你有 62 个特征，你一个一个试：「删掉这个试试，测试集分数多少？」试了 1000 次，最后留下让测试集分数最高的那 40 个。

现在，这个测试集分数还有意义吗？**没有了。** 因为你已经用它做了 1000 次决策，它实际上变成了你的**训练集**。你得到的不是一个「模型有多好」的无偏估计，而是一个「这个模型在这个特定测试集上被挑得最漂亮」的数字。换一批新数据，性能会明显下降。

> 类比：一个学生拿同一套模拟题反复练、每次按答案调整自己的答题套路，最后这套题考了 95 分。这不代表他高考能考 95。

**正确做法**（三层数据）：

```
测试集（只用一次，最后报数）
   ↑
训练集 ──切──> 训练子集（真的训练） + 验证子集（决定删哪个特征）
```

### 💻 实现（后向消除）

```python
# ① 内部再切一刀：验证集只用于「选哪些特征」
Xw_tr, Xw_val, yw_tr, yw_val = train_test_split(Xtr, y_train, test_size=0.25,
                                                random_state=42, stratify=y_train)

# ② 加速三招（否则跑不完）
rs = np.random.RandomState(0)
sub = rs.choice(Xw_tr.shape[0], size=10000, replace=False)   # 只用 1 万行
Xs, ys = Xw_tr[sub], yw_tr.iloc[sub]

def fast_lr():
    return LogisticRegression(solver="liblinear", max_iter=200, random_state=42)

def val_score(cols):
    m = fast_lr().fit(Xs[:, cols], ys)
    return average_precision_score(yw_val, m.predict_proba(Xw_val[:, cols])[:, 1])

# ③ 从 60 个候选开始，每次删掉「删了之后验证分数最高」的那一个
pool = list(np.argsort(-t_stat)[:60])
remaining = pool.copy()
while len(remaining) > 40:
    scores = [val_score([x for x in remaining if x != f]) for f in remaining]
    remaining.pop(int(np.argmax(scores)))
```

| 名字 | 作用 |
|---|---|
| `Xw_tr / Xw_val` | 训练子集 / 验证子集。**注意命名带 `w`（wrapper）**，别和全局的 `X_train` 混淆 |
| `rs.choice(n, size=10000, replace=False)` | 无放回随机抽 1 万个行下标 |
| `solver="liblinear"` | 换一个更快的求解器（对稀疏数据尤其快） |
| `max_iter=200` | 搜索阶段不追求精确解，200 次足够排序 |
| `val_score(cols)` | 「用 cols 这组列训练 + 在验证集上打分」，是整个贪心的评价函数 |
| `[x for x in remaining if x != f]` | 「除了 f 之外的所有列」= 假装删掉 f |
| `remaining.pop(int(np.argmax(scores)))` | 删掉「删了以后分数最高」的那个特征——即**当前最没用的** |
| `np.argmax` | 返回最大值的**下标**；`np.max` 返回最大值本身。这两个别混 |

**为什么这么慢**：每删一个特征都要重训 60 次左右的模型。删 20 个特征 → 约 1000 次训练。这是 $O(n^2)$ 级别的开销。

**结果：AUC-PR = 0.44614，选择耗时 41 s。**

> 用 62 个特征、4 万行做完整的贪心要几十分钟；我在搜索阶段用了「1 万行子样本 + 快速求解器 + 从 t 统计量的前 60 名里挑」三个加速手段，把时间压到一分钟内。**这是工程上完全正当的做法**，但要在报告里说明，因为它会影响选出来的名单。

### 前向选择：从 0 个特征加到 40 个

## 📖 原文

> It is worth noting that using this method, you can go in the opposite direction. Try to _add_ one of the most useful features to the sample until there are 40 pieces. Is there a threshold at which adding the following features will only worsen the quality of the model?

## 🇨🇳 翻译

值得注意的是，用这个方法你也可以反着来。试着每次往样本里**加入**一个最有用的特征，直到凑够 40 个。是否存在一个阈值，超过它之后继续加特征只会让模型质量变差？

### 💻 结果与解读

```python
selected = []
fwd_scores = []
while len(selected) < 40:
    free = [f for f in candidates if f not in selected]
    scores = [val_score(selected + [f]) for f in free]     # 试遍「再加一个」
    selected.append(free[int(np.argmax(scores))])          # 加最划算的那个
    fwd_scores.append(max(scores))                         # 记录这一步的验证分数
```

这画出的是**「特征数 → 验证集 AUC-PR」的学习曲线**。

**结果：AUC-PR = 0.44427，耗时 25 s。验证集上最好的点在 k = 24（验证分数 0.46833）。**

这条曲线通常长这样：

```
验证集
AUC-PR │        ___________  ← 饱和区：再加也没用
       │      /
       │    /                 ← 陡升区：每个特征都很值钱
       │  /
       │/
       └──────────────────→ 特征个数
         k*
```

**答案是：有阈值，但它表现为「饱和」而不是「变差」。** 这里 k≈17 之后验证分数就不再上升了——再加的特征只是**重复已有信息**（还记得 5 个 redundant 特征吗）。在真实数据里，加了**噪声特征**（纯随机）时曲线会开始**下降**，因为模型会开始拟合噪声。

> **答案（写进作业）**：是的，存在一个「性能饱和点」（本实验在 k≈24），超过它之后继续加特征不再带来提升。如果加入的特征与目标无关，曲线会从平台转为下降——这就是「特征选择的收益边界」。

## 总结：三种方法怎么选

| 方法 | 怎么工作 | 优点 | 缺点 | 本次 AUC-PR | 耗时 |
|---|---|---|---|---|---|
| **Embedded**（`\|coef\|`） | 用模型自己的权重 | 只训 1 次，几乎免费；考虑特征间的联合效应 | 权重受尺度影响；相关特征互相分摊权重；只适用于线性/树模型 | 0.44524（未标准化）→ 0.44879（按类型标准化） | ~0.5 s |
| **Filter**（t 统计量） | 逐列算统计量，不看模型 | 极快；与模型无关；适合海量特征 | 忽略特征交互；单看没用的强组合特征会被丢掉 | 0.45162 | ~0.06 s |
| **Wrapper**（贪心） | 反复训练，直接优化指标 | 直接优化你真正关心的指标；考虑交互 | 极慢（$O(n^2)$ 次训练）；容易过拟合验证集 | 0.44614（后向） / 0.44427（前向） | 41 s / 25 s |

**哪个质量最好？** 出乎意料——**最快的 filter 赢了**：0.45162，比用全部 62 个特征（0.44786）还高，而且只要 0.06 秒。这不是巧合：

- 62 个特征里有 53 个是 one-hot 指示变量，**它们大多是噪声**（比如 `job_unknown`、`default_yes` 只占几十个样本）。t 统计量能一眼把它们筛掉。
- wrapper 反而更差，因为它优化的是**单个验证集**的表现，容易过拟合那一份数据——这正是「用验证集做决策也会过拟合」的实例。
- 嵌入式（0.4488）介于两者之间，它兼顾了特征之间的联合效应，但仍是单次训练的副产物。

**这个结果值得记住**：简单方法在「特征中等数量、模型线性」的场景下经常赢。**复杂度不是质量的保证。**

**为什么所有差距都这么小？** 因为有 3.3 万行、只有 62 个特征——模型本来就不怎么过拟合，删掉 22 个特征几乎没影响。如果换成「1000 行 × 50000 个特征」的场景（基因数据、文本），结论会完全反转：filter 成为唯一可行的方案，wrapper 直接不可用。

> 记住这个思维方式：**方法的好坏取决于数据规模，不取决于方法本身「高级」与否。**

### 📄 修正版「三种方法对比」长答案全文翻译（solved 版 Part 3 总结 Answer）

> | method | how it works | pros | cons |
> |---|---|---|---|
> | **Embedded** (`|coef|` of the linear model) | take the model's own weights | one training run, almost free; considers features jointly | weights depend on scaling; correlated features share weight, so a useful feature may get a small coefficient and be dropped; linear models only |
> | **Filter** (t-statistic) | score each feature independently of the model | very fast (one pass over the columns); model-agnostic | ignores interactions; a feature that is useless alone but useful in combination is lost |
> | **Wrapper** (greedy add/remove) | train the model many times, measure validation quality | directly optimises the metric we care about; accounts for interactions | slow: ~O(n²) model fits; tends to overfit the validation split |

**翻译（对比表）**：

| 方法 | 怎么工作 | 优点 | 缺点 |
|---|---|---|---|
| **嵌入式**（线性模型的 `\|coef\|`） | 直接用模型自己的权重 | 只训一次，几乎免费；联合考虑特征 | 权重受标准化影响；相关特征分摊权重，有用特征可能系数很小而被丢掉；只适用于线性模型 |
| **过滤式**（t 统计量） | 独立于模型给每个特征打分 | 极快（扫一遍列）；与模型无关 | 忽略交互；「单看没用、组合有用」的特征会被丢掉 |
| **包裹式**（贪心增/删） | 多次训练模型，测量验证集质量 | 直接优化我们真正关心的指标；考虑交互 | 慢：约 O(n²) 次训练；容易过拟合验证集 |

> **Measured results (AUC-PR on the test set, 40 features left):**
>
> | selection | AUC-PR | selection time |
> |---|---|---|
> | all features (unscaled) | 0.44786 | 0.13 s |
> | embedded, unscaled | 0.44524 | ~0.5 s |
> | embedded, all features scaled | 0.44881 | ~0.5 s |
> | embedded, only numerical scaled | 0.44879 | ~0.5 s |
> | filter (t-statistic) | 0.45162 | ~0.06 s |
> | wrapper (backward) | 0.44614 | 41 s |
> | wrapper (forward) | 0.44427 | 25 s |

**翻译（实测结果表，测试集 AUC-PR，保留 40 个特征）**：全特征（未标准化）0.44786；嵌入式（未标准化）0.44524；嵌入式（全部标准化）0.44881；嵌入式（只标准化数值列）0.44879；过滤式（t 统计量）**0.45162（最高，且只要 0.06 秒）**；包裹式（后向）0.44614（41 秒）；包裹式（前向）0.44427（25 秒）。

> **Which is best?** Unexpectedly, the cheapest one: the **filter** (0.45162), which beats keeping all 62 features (0.44786) and both wrapper runs. Two reasons:
>
> * 53 of the 62 columns are one-hot indicators and many of them are noise (`job_unknown`, `default_unknown` — a few dozen rows). A t-statistic throws them out immediately, and the L2 budget that used to be spent on them goes to the useful features.
> * the wrapper optimises a **single validation split**, so it overfits that split — the standard failure mode of greedy selection.

**翻译**：**哪个最好？** 出乎意料——最便宜的那个：**filter**（0.45162），它击败了保留全部 62 个特征（0.44786）和两种 wrapper。两个原因：
- 62 列里 53 列是 one-hot 指示变量，很多是噪声（`job_unknown`、`default_unknown`——只有几十行）。t 统计量立刻把它们扔出去，原本浪费在它们身上的 L2 预算回到了有用的特征上。
- wrapper 优化的是**单个验证集**的表现，所以对那一折过拟合——这是贪心选择的标准失败模式。

> **Where the scaling story matters:** the unscaled embedded selection (0.44524) is the worst of the "40 features" variants, and its list differs from the scaled one by 10 of 40 features. So scaling did not change the score much, but it changed *which features the model believes in* — and that is exactly what a report is built on.

**翻译**：**标准化真正的意义在这里**：未标准化的嵌入式选择（0.44524）是所有「40 特征」方案里最差的，而且它的名单和标准化后的名单有 40 个里 10 个不同。所以标准化没怎么改变分数，但改变了**模型「相信」哪些特征**——而一份分析报告恰恰建立在这上面。

> **Best time/quality trade-off is the filter:** 0.06 s against 25-41 s for the wrappers, and a *better* score. The wrapper buys nothing here: it costs three orders of magnitude more compute and still loses, because it optimises one particular validation split rather than the metric in general.

**翻译**：**时间/质量的最优权衡是 filter**：0.06 秒对 wrapper 的 25~41 秒，分数还更高。wrapper 在这里什么也没买到：它贵了三个数量级的算力还输了，因为它优化的是「某一个验证划分」而不是「指标本身」。

> **Why all the differences are so small:** 33k rows and 62 features — the model barely overfits in the first place, so dropping 22 of 62 features is almost neutral. With hundreds of thousands of features (text, genomics) or with few rows the ranking of these methods changes completely: filters become indispensable, wrappers become impossible.

**翻译**：**为什么所有差距都这么小**：3.3 万行、62 个特征——模型本来就不怎么过拟合，所以删掉 62 个里的 22 个几乎是中性的。换成几十万特征（文本、基因组学）或者行数很少的场景，这些方法的排名会完全翻转：filter 变得不可或缺，wrapper 直接不可行。

---
# 5. Part 4：模型的经济效益（2 分）

## 📖 原文

> In this part, we will deal with what you are most likely to need in a real job (besides rewriting `json`, of course). Namely: we will collect several specialized quality metrics, try to tune the model to maximize _profit_ and estimate how much we can actually earn from this. Of course, a lot of simplifying assumptions will be made here, but first things first.
>
> __Task 11.__ Let's say your call center employees receive $1 per call to a customer. If the client agrees to the proposed conditions, he will bring $10 to the bank.
>
> If you listened to everything on the economy, then we recall that revenue is how much money customers brought us, and profit is revenue minus salary costs and so on.

## 🇨🇳 翻译

这一部分处理你在真实工作中最可能需要的东西（当然，除了改 `json` 之外）：我们要收集几个专用的质量指标，试着把模型调到**利润最大**，并估算我们实际能赚多少。当然，这里会做很多简化假设，但我们一步一步来。

**任务 11**：假设你的呼叫中心员工**每打一个电话拿 1 美元**。如果客户同意了你提出的条件，他给银行带来 **10 美元**。

如果你认真听了经济学的内容，那我们回忆一下：**revenue（收入）**是客户带给我们的钱，**profit（利润）**是收入减去工资等成本。

### 🧠 三个词必须先分清（很多人在这里翻车）

| 概念 | 定义 | 本作业里 |
|---|---|---|
| **收入（revenue）** | 客户带来的钱，不减成本 | 10 × 成交客户数 |
| **成本（cost）** | 为了拿到收入付出的钱 | 1 × 打电话次数 |
| **利润（profit）** | 收入 − 成本 | revenue − cost |

> 作业特意提醒这一点：*「revenue is how much money customers brought us, and profit is revenue minus salary costs」*。你算「赚了多少」时必须用 profit，只说 revenue 是自欺欺人。

### 🧠 关键数字：一次电话值不值得打？

这是全 Part 4 的核心，一条公式解决 80% 的问题。

**打这个电话的期望利润** = （成交概率 × 成交收益）−（打电话的成本）

$$E[\text{profit}] = 10\cdot p - 1$$

什么时候该打？当期望利润 > 0：

$$10p - 1 > 0 \quad\Longleftrightarrow\quad p > 0.1$$

**所以最优决策阈值是 0.1，不是 0.5！** 记住这个结论，后面所有实验结果都在验证它。

In [ ]:
# 用一个人的合成例子验证这条公式：最优阈值 = cost / revenue
import numpy as np

rng = np.random.RandomState(0)
n = 200_000
p = rng.beta(0.6, 4.0, size=n)     # 假设模型的概率已经校准，可以直接当概率用（多数人很低）
y = (rng.rand(n) < p).astype(int)  # 按这个概率真实地「成交」

grid = np.arange(0, 1.001, 0.01)
print("cost revenue | 理论阈值 | 实测最优阈值")
print("-" * 42)
for cost, revenue in [(1, 10), (1, 1.5), (3, 10)]:
    profits = []
    for t in grid:
        pred = p >= t
        profits.append(revenue * (pred & (y == 1)).sum() - cost * pred.sum())
    best = grid[int(np.argmax(profits))]
    print("%4g %7g | %8.3f | %8.2f" % (cost, revenue, cost / revenue, best))

三行的「理论阈值」和「实测最优阈值」都对上了。**记住这个比值：$\frac{\text{cost}}{\text{revenue}}$。** 它和数据、模型、算法统统无关——只和钱有关。

> 注意这只是「概率已校准」的理想情况。如果模型的概率没校准（比如 SVM 的分数），这个公式还得先经过校准那一步才成立——这就是 Part 1 学校准的原因。

## 用 5 折交叉验证算钱

## 📖 原文

> Load the telemarketing data from the previous task block. In this part, you do not need to divide the sample - we will use cross-validation. Use 5 folds, do `shuffle=True, random_state=500`. By cross-validation, you will get 5 options for training and test sets. Train logistic regression on each training set (use one-hot for categorical features, leave hyperparameters at their defaults) and make predictions for the corresponding test sets. Let's say your employees decide to call all the positive forecasts. Calculate the revenue on all test samples and average. How much money do you earn on average? Also calculate the standard deviation.
> How much of the money you earn will have to give to your call center operators?

## 🇨🇳 翻译

加载上一部分用过的电话营销数据。这一部分不需要手动划分样本——我们用交叉验证。用 **5 折**，`shuffle=True, random_state=500`。这样你会得到 5 组「训练/测试」划分。在每个训练集上训练逻辑回归（类别特征用 one-hot，超参数保持默认值），并对相应的测试集做预测。假设你的员工决定**给所有预测为正的客户打电话**。计算所有测试集上的收入并取平均。你平均能赚多少钱？也算一下标准差。

**你赚到的钱里，有多少要付给呼叫中心的接线员？**

> 「给所有预测为正的客户打电话」= 用默认阈值 0.5 做决策。

### 💻 代码讲解

```python
def bank_pipe(C=1.0):
    return Pipeline([
        ("pre", ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore",
                                                         sparse_output=True), cat4),
                                   ("num", "passthrough", num4)])),
        ("lr", LogisticRegression(max_iter=1000, C=C, random_state=42)),
    ])
```

| 名字 | 作用 |
|---|---|
| `bank_pipe(C=1.0)` | 造一个「one-hot + 逻辑回归」的流水线，`C` 可调 |
| `sparse_output=True` | 输出稀疏矩阵，省内存 |

```python
def cv_economics(C=1.0, n_repeats=1, seed=500, threshold=0.5, random_revenue=False):
    rows = []
    for r in range(n_repeats):                                   # 重复次数
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed + r)
        for i, (tr, te) in enumerate(skf.split(X4, y4)):
            pipe = bank_pipe(C).fit(X4.iloc[tr], y4.iloc[tr])
            proba = pipe.predict_proba(X4.iloc[te])[:, 1]
            pred = proba >= threshold                            # 决策
            truth = y4.iloc[te].to_numpy()

            tp = int((pred & (truth == 1)).sum())                # 成交的客户数
            calls = int(pred.sum())                              # 打了多少个电话
            revenue = float(np.random.uniform(0, 20, size=tp).sum()) if random_revenue else 10.0 * tp
            rows.append(dict(fold=i + 1, tp=tp, calls=calls,
                             revenue=revenue, cost=1.0 * calls,
                             profit=revenue - 1.0 * calls))
    return pd.DataFrame(rows)
```

| 名字 | 作用 |
|---|---|
| `n_repeats` | 把整套 CV 重复几次（用不同的随机划分）。单次 CV 的估计噪声很大 |
| `seed + r` | 第 r 次重复用 `seed+r` 当种子，保证「不同但不失控」 |
| `pred & (truth == 1)` | 两个布尔数组逐元素相与 → 「预测为正**且**真实为正」= 真正例 |
| `.sum()` | 布尔数组求和 = 数 True 的个数 |
| `np.random.uniform(0, 20, size=tp)` | 从 $U[0,20)$ 抽 `tp` 个数（每个成交客户各自的收益） |

> ⚠️ 小技巧：`pred & (truth == 1)` 里的 `pred` 是 `proba >= threshold` 的结果，已经是布尔数组。**不要写成 `pred == 1`**，那是多余的。

### 📊 结果（本次运行，阈值 0.5，固定收益 $10）

| fold | TP（成交） | calls（电话数） | revenue | cost | profit |
|---|---|---|---|---|---|
| 1 | 218 | 321 | 2180 | 321 | 1859 |
| 2 | 223 | 337 | 2230 | 337 | 1893 |
| 3 | 177 | 287 | 1770 | 287 | 1483 |
| 4 | 228 | 340 | 2280 | 340 | 1940 |
| 5 | 222 | 329 | 2220 | 329 | 1891 |
| **平均** | 213.6 | 322.8 | **2136.0** | **322.8** | **1813.2** |
| **标准差** | — | 21.3 | 207.7 | 21.3 | **186.8** |

**怎么回答那两个问题：**

1. **平均赚多少？** 利润 **≈ $1813**（标准差 $187）。注意这是**每个测试折**的数字，每折约 8238 个客户。
2. **要付多少给接线员？** 成本 = 323 次电话 × $1 = **$323**，占收入的 $\frac{323}{2136} \approx 15.1\%$。

**顺便算一下业务指标**（很有用，作业后面会问）：
- precision = $213.6/322.8 \approx 66\%$：打了 323 个电话，214 个成交。
- recall = $213.6 / (8238 \times 11.3\%) \approx 23\%$：所有会成交的客户里，只找到了 23%。

> 这两个数字说明：**阈值 0.5 太保守了。** 66% 的 precision 意味着还有大量「有一点可能」的客户被漏掉了——他们的 p 在 0.1 到 0.5 之间，按 $10p-1>0$ 的标准**都该打**。

## 加入随机性：客户带来的钱是随机的

## 📖 原文

> Let's add some randomness. Now let the client who agrees to the terms bring not 10 dollars, but a random value uniformly distributed in the interval $[0;20)$. Do all the same steps. To simulate a real situation, **DO NOT** fix `random_seed` when calculating customer revenue. What happened?

## 🇨🇳 翻译

加点随机性。现在同意条件的客户带来的不是 10 美元，而是 $[0;20)$ 上均匀分布的随机值。做同样的步骤。为了模拟真实情况，**不要**在计算客户收益时固定 `random_seed`。发生了什么？

### 🧠 关键洞察：均值没变，方差变大了

$U[0,20)$ 的期望是 $\frac{0+20}{2} = 10$——**正好等于原来的固定值 10**。

所以：
- **平均利润应该几乎不变**（理论上期望完全相同）。
- **但波动会变大**：同一个模型、同一批数据，这次跑出来赚 1927，下次可能只有 1777。

### 📊 结果（本次运行）

| | mean profit | std |
|---|---|---|
| 固定收益 $10 | 1813.2 | 186.8 |
| 随机收益 $U[0,20)$ | 1828.7 | 185.1 |

再做 5 次独立运行，平均利润分别是：

```
[1927, 1877, 1848, 1822, 1777]      极差 ≈ 150
```

**注意这个极差比标准差还大。** 也就是说：如果你只跑一次、看到 1927，就宣称「我的模型能赚 1927」，另一个人跑一次看到 1777，就宣称「这模型只能赚 1777」——两个人都没错，但都错了。**单次运行的数字不能用来比较模型。**

> **答案（写进作业）**：平均利润基本不变（$U[0,20)$ 的期望正好是 10，与固定值相同），但**方差增大**，且单次运行的结果不可复现。这意味着：用这个指标比较模型/参数时，必须多次重复，否则你比较的是两次随机抽样，而不是两个模型。

> ⚠️ 为什么不固定 seed？作业说得很清楚：真实世界里客户的收益本来就是随机的，固定 seed 只是为了好看的假象。不固定才能暴露「这个指标有多不稳」这个真问题。

### 📄 修正版答案格全文翻译（solved 版「What happened?」Answer）

> **Answer.** The *average* profit stays at the same level, but the **spread grows**. And that is not a coincidence: if the revenue per client is $U[0,20)$, its mean is exactly 10 — the same as the fixed $10 of the previous task. What changes is the variance: the same model on the same fold can now earn noticeably more or less, purely by luck.
>
> Two practical consequences:
> 1. Any comparison of models/thresholds/C on this metric needs many repetitions, otherwise you compare two random draws rather than two models.
> 2. The metric is no longer a "clean" function of the model — the same model gives different profit on every run, which is exactly what the next two cells demonstrate.

**翻译**：**答案**：**平均**利润保持在同一水平，但**波动变大**了。这不是巧合：如果每个客户的收益是 $U[0,20)$，它的期望恰好是 10——和上一任务的固定 $10 相同。改变的是**方差**：同一个模型、同一个折，现在可能多赚不少、也可能少赚不少，纯粹靠运气。两个实践后果：
1. 用这个指标比较任何模型 / 阈值 / C，都必须**多次重复**，否则你比较的是两次随机抽样，而不是两个模型。
2. 这个指标不再是模型的「干净」函数——同一个模型每次跑出不同的利润，这正是接下来两格要演示的。

## 调正则化系数 C，让利润最大

## 📖 原文

> Adjust the cross-validation coefficient of the regularization of the model to maximize profit (consider as a random variable above). Did you get any win? At what regularization coefficient and profit is maximum? Plot expected profit versus odds, also specify standard deviations (`plt.errorbar` will help you).

## 🇨🇳 翻译

调整（交叉验证下的）模型正则化系数，使利润最大（把利润当作上面那个随机变量）。你有获得收益吗？在哪个正则化系数下利润最大？画出期望利润随系数的变化，并标出标准差（`plt.errorbar` 会帮你）。

### 📊 结果（本次运行，每组 3 次重复 × 5 折 = 15 个观测）

| C | 平均利润 | 标准差 |
|---|---|---|
| 0.001 | 1476.2 | 174.5 |
| 0.01 | 1724.6 | 157.3 |
| 0.1 | 1778.9 | 142.6 |
| 1 | 1806.2 | 198.7 |
| 10 | 1807.3 | 153.8 |
| **100** | **1835.4** | 147.5 |

```python
plt.errorbar(C_grid, means, yerr=stds, marker="o", capsize=4)
plt.xscale("log")          # C 是等比变化的，用对数横轴才看得清
```

| 名字 | 作用 |
|---|---|
| `yerr=stds` | 每个点画一条 ± 标准差的误差棒 |
| `capsize=4` | 误差棒末端小横线的长度 |
| `plt.xscale("log")` | 横轴取对数。C 从 0.001 到 100 跨 5 个数量级，线性轴会全部挤在左边 |

**怎么读：**

- **C=0.001 明显更差**（1476 vs 1835）。这是**真实的**损失：正则化太强，模型被压得只会输出接近基准率的常数，几乎无法区分客户。
- **C 在 0.1 ~ 100 之间，全部落在彼此的误差棒里**（1779 ~ 1835，标准差 143~199）。也就是说，这个区间内「哪个 C 最好」是**噪声说了算**。
- 「收益」= 从 C=0.001 换到 C=100，多赚约 **$360/折**。这是真实收益，但它来自**避免极端错误参数**，而不是「精细调参」。

> **答案（写进作业）**：有收益，但只体现在「避免过强的正则化」上：C=0.001 时利润明显下降（1476 vs 1835）。在 C ∈ [0.1, 100] 这个合理区间内，各点的差异完全落在标准差范围内，**不存在一个稳定可复现的「最优 C」**。本次最大点出现在 C=100（1835 ± 148），下一次可能就换成 C=1 了。

### 📄 修正版答案格全文翻译（solved 版「调 C 有没有收益」Answer）

> **Answer.** There **is** a win, but it comes from the left end of the curve, not from fine tuning:
>
> * `C = 0.001` is clearly bad: 1476.2 instead of ~4127.8. With such strong regularization the model is squeezed towards a near-constant output, so it stops distinguishing clients at all. Avoiding that is a real gain of ~$300 per fold.
> * Inside `C ∈ [0.1, 100]` the curve is flat **within the error bars** (1778.9 … 1835.4, standard deviation ~150-190). The measured maximum is at `C = 100`, but the argmax moves from run to run.
>
> So: tuning C is worth doing only to escape the "over-regularized" regime; once there, the profit is dominated by the **threshold** (how many clients we call), not by the exact value of C. The next block shows that the threshold is worth far more than the whole C sweep.

**翻译**：**答案**：**有**收益，但它来自曲线的左端（避开坏参数），而不是精细调参：
- `C = 0.001` 明显很差：1476.2，而正常区间约 1835。正则化太强时模型被压成「几乎输出常数」，完全失去区分客户的能力。避开它每折大约多赚 $300，是真实收益。
- 在 `C ∈ [0.1, 100]` 内部，曲线在**误差棒范围内是平的**（1778.9 ~ 1835.4，标准差 150~190）。本次最大值在 `C = 100`，但这个 argmax 每次运行都会换位置。
- 所以：调 C 只值得做到「逃出过度正则化」为止；再往后，利润由**阈值**（我们给多少客户打电话）主导，而不是 C 的精确值。下一块会展示：阈值的价值远超整个 C 扫描。

## 重复运行：最优 C 稳定吗

## 📖 原文

> Try to run iteration several times. Is there the same "best" coefficient every time? Is there any pattern? What conclusions can you draw from this?

## 🇨🇳 翻译

试着重复运行几次。每次的「最优」系数都一样吗？有规律吗？你从中能得出什么结论？

### 🧠 结论：不稳定，而且这本身就是答案

因为收益是随机的、CV 划分也会换（`seed+r`），所以每次的 argmax 都会在两个相邻值之间跳。

**这告诉我们三件事：**

1. **在噪声大的目标上，「最优超参数」本身就是一个随机变量。** 报它的时候应该同时报不确定性：「C ≈ 1（但 0.1~100 之间无显著差别）」。
2. **平台期是好消息**：如果最优点周围是平的，说明这个参数根本不敏感，直接用默认值就行，不值得花算力去调。
3. **要降低噪声，应该增加重复次数**（或换一个更平滑的目标，比如 AUC-PR 这种「排序的确定性函数」），而不是更精细地扫网格。

> 对比一下 Part 1 的调 C：那里用的是 AUC-PR，**同一个模型、同一份数据永远得到同一个分数**，所以 GridSearch 的结果稳定可复现。这里换成「利润」这个随机变量，调参就变成了赌博。**这是本作业最深刻的一课：指标的选择不只影响你看到什么，还影响你的决策流程是否可靠。**

### 📄 修正版结论格全文翻译（solved 版「最优 C 稳定吗」Conclusion）

> **Conclusion.** The "best" C is not stable: different CV seeds give different winners, and the winning values sit inside each other's error bars. So:
> * with a noisy objective, "the best hyperparameter" is itself a random variable — its value must be reported together with the uncertainty;
> * if you must choose, average over several repeats/seeds (or use a smoother objective, e.g. AUC-PR, which is a deterministic function of the ranking);
> * a flat optimum is good news: it means the exact value of C does not matter and you can keep the default.

**翻译**：**结论**：「最优」的 C 并不稳定：不同的 CV 种子给出不同的赢家，而且这些赢家的数值都在彼此的误差棒之内。所以：
- 在带噪声的目标上，「最优超参数」本身是一个**随机变量**——报告它时必须连同不确定性一起报；
- 如果非要选一个，就对多次重复 / 多个种子取平均（或者换一个更平滑的目标，比如 AUC-PR——它是排序的确定性函数）；
- 平坦的最优点是**好消息**：它意味着 C 的精确值无所谓，直接用默认值就好。

## Task 12：precision 还是 recall 更重要？

## 📖 原文

> __Task 12.__ We have already described an approximate economic model of your task above. What do you think is more important for your business - good precision or recall models? Why?

## 🇨🇳 翻译

**任务 12**：上面已经描述了这个任务的一个粗略经济模型。你认为对你的业务来说，哪个更重要——好的 precision 还是好的 recall？为什么？

### 🧠 算一下就知道：recall 更重要

用作业给的数字：一次电话成本 $1，一个成交客户带来 $10。

| 错误类型 | 后果 | 代价 |
|---|---|---|
| **漏掉一个真客户**（FN，recall 的损失） | 白丢一单 | **−$10** |
| **多打一个无效电话**（FP，precision 的损失） | 白花一次话费 | **−$1** |

代价比是 **10 : 1**。所以：**为了多找到一个客户，多打 10 个无效电话也是划算的。**

用公式说：precision 的**盈亏平衡点**是 $\frac{1}{10}=10\%$。只要打出去的电话里超过 10% 能成交，这件事就赚钱。而现在阈值 0.5 给出的 precision 是 66%——**太浪费了**，说明我们放过了大量「成交率在 10%~50% 之间」的客户。

> **答案（写进作业）**：**recall 更重要。** 在这个经济模型下，漏掉一个客户损失 $10，多打一个电话只损失 $1（10:1）。因此应该接受较低的 precision 换取更高的 recall，盈亏平衡的 precision 是 10%。只有当「每次电话成本上升」或「客户价值下降」时，天平才会转向 precision——比如成本 $1、收益 $1.5 时，盈亏平衡 precision 变成 67%，precision 就变得至关重要。

### 📄 修正版答案格全文翻译（solved 版 Task 12 Answer 全文）

> **Recall is more important in this business.**
>
> The numbers from the task: one call costs $1, and a client who agrees brings $10 on average. So:
> - a **missed** good client (false negative) costs us the whole **$10** of potential profit;
> - an **extra** call (false positive) costs only **$1**.

**翻译**：**在这个业务里，recall 更重要。** 用任务里的数字：一次电话成本 $1，一个同意的客户平均带来 $10。所以：一个**漏掉的**好客户（假阴性）让我们损失整整 **$10** 的潜在利润；而一个**多打的**电话（假阳性）只花 **$1**。

> The ratio is 10 : 1, so it pays to call aggressively: it is worth making up to 10 useless calls to find one more customer. In terms of the confusion matrix we should accept low precision in exchange for high recall.
>
> **Where it breaks:** if the call cost rises or the average revenue falls, the balance shifts. With $1 vs $10 the break-even precision is 10%; with $1 vs $1.5 it becomes 67% and precision becomes the priority. That is why the next part tunes the threshold numerically instead of relying on intuition.
>
> **Also note** that the metric should match this asymmetry — which is why AUC-PR (and profit) are better guides here than accuracy or AUC-ROC.

**翻译**：比例是 10 : 1，所以激进地打电话是划算的：**为了多找到一个客户，多打最多 10 个无效电话都值**。用混淆矩阵的语言说：我们应该接受低 precision 来换高 recall。**这个结论在哪里失效**：如果电话成本上升、或者平均收益下降，天平就会移动。$1 对 $10 时盈亏平衡 precision 是 10%；$1 对 $1.5 时变成 67%，这时 precision 就成了优先项。这就是为什么下一部分要**用数值方法调阈值**，而不是靠直觉。**还要注意**：评估指标必须与这种不对称性匹配——这就是为什么在这里 AUC-PR（以及利润）比 accuracy 或 AUC-ROC 更适合当向导。

## 调二值化阈值

## 📖 原文

> Recall that in fact, logistic regression predicts for us the probabilities of a positive class for an object. Perhaps by adjusting the binarization threshold for these probabilities, we can get some kind of gain? Test your reasoning above by adjusting the binarization threshold on cross-validation to maximize profits. Use a grid from 0 to 1 with a step of 0.01. Recall that lowering the threshold gives us higher recall and lower precision, and vice versa.
>
> Build a graph of the dependence of profit on the binarization threshold, also mark the standard deviations on it. Select the best threshold.
>
> Question: Do you notice any pattern? For the correct answer to this question, try to run several times and think about why the threshold is obtained in a particular area?

## 🇨🇳 翻译

回忆一下，逻辑回归实际上给我们预测的是样本属于正类的**概率**。也许通过调整这些概率的**二值化阈值**，我们能获得一些收益？用交叉验证来验证你上面的推理：在 0 到 1 之间以 0.01 为步长扫描阈值，使利润最大。回忆一下：**降低阈值会得到更高的 recall 和更低的 precision，反之亦然。**

画出利润随二值化阈值变化的图，并在图上标出标准差。选出最好的阈值。

**问题**：你注意到什么**规律**了吗？为了正确回答这个问题，多运行几次，并想想为什么阈值会落在那个特定的区域？（答案就是下一节的「threshold ≈ cost / revenue」。）

### 💻 实现的核心技巧：一次拟合，评估所有阈值

初学者常写成这样（**很慢，而且错**）：

```python
for t in thresholds:            # ✗ 别这样
    model.fit(...)              # 每个阈值都重训一次 —— 完全没必要！
```

正确做法：**模型只训一次，拿到概率后，在概率上扫阈值**。因为阈值不改变模型，只改变决策。

```python
def threshold_profits(thresholds, n_repeats=3, seed=500):
    store = np.zeros((len(thresholds), 5))         # 行=阈值, 列=折
    for r in range(n_repeats):
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed + r)
        for i, (tr, te) in enumerate(skf.split(X4, y4)):
            pipe = bank_pipe().fit(X4.iloc[tr], y4.iloc[tr])       # ← 只训这一次
            proba = pipe.predict_proba(X4.iloc[te])[:, 1]
            truth = y4.iloc[te].to_numpy()
            money = np.random.uniform(0, 20, size=len(truth))      # 每个客户一个「如果他买了会给多少」
            for j, t in enumerate(thresholds):
                pred = proba >= t
                tp = pred & (truth == 1)
                store[j, i] += money[tp].sum() - pred.sum()        # 收入 - 电话费
    return store / n_repeats
```

| 名字 | 作用 |
|---|---|
| `store` | 形状 `(101, 5)` 的矩阵：每个阈值 × 每折的利润 |
| `money = np.random.uniform(0, 20, size=len(truth))` | **给每个客户预先抽一个收益**。注意它在阈值循环**外面**——这样所有阈值用的是同一份收益，比较才公平（这叫「公共随机数」，能大幅降低比较的方差） |
| `money[tp].sum()` | 只把成交客户的收益加起来 |
| `pred.sum()` | 打电话的次数 = 成本 |

> ⚠️ 这个「公共随机数」技巧很重要。如果每个阈值重新抽一次收益，阈值之间的差异里会混进抽样噪声，曲线会变成一团乱麻，根本看不出趋势。

### 📊 结果与那个「规律」

```python
best_t = thresholds[int(np.argmax(mean_profit))]
```

本次运行：**最优阈值 ≈ 0.11，期望利润 ≈ 4127.8**——对比阈值 0.5 时的 1813，**翻了一倍多**。

> 记住这个对比：整个 C 的扫描（6 个值 × 15 次运行）最多挣回 $360，而只把阈值从 0.5 挪到 0.11，就多赚了 $2300。**对一个概率模型来说，「怎么用这个概率」比「模型调多准」重要得多。**

**这就是作业问的那个「规律」——它和 5.1 节的公式完全吻合：**

$$\text{threshold}^\ast \approx \frac{\text{一次电话的成本}}{\text{一个客户带来的期望收益}} = \frac{1}{10} = 0.1$$

**为什么正好是这个数？** 因为决策规则是「当 $10p - 1 > 0$ 时打电话」，也就是 $p > 0.1$。阈值 $\frac{\text{cost}}{\text{revenue}}$ 就是**期望利润为零的那个点**，这是决策论里的标准结果，和模型无关、和数据无关，只和钱有关。

> 更一般地：$\text{threshold}^\ast = \frac{C_{FP}}{C_{FP} + C_{FN}}$，其中 $C_{FP}$ 是「误报」的代价（$1），$C_{FN}$ 是「漏报」的代价（$10 - 1 = 9$）。算一下：$\frac{1}{1+9} = 0.1$。两种写法等价。

**为什么每次运行结果会在 0.05~0.2 之间飘？** 因为曲线在这个区间非常平（差几十美元，而标准差是 130），argmax 完全可能落在任意一个点上。**拐点位置稳定，精确位置不稳定。**

### 📄 修正版答案格全文翻译（solved 版「the pattern」Answer）

> **Answer — the pattern.** The optimal threshold sits at **0.11**, i.e. essentially at
> $$\text{threshold}^\ast \approx \frac{\text{cost of a call}}{\text{expected revenue of a client}} = \frac{1}{10} = 0.1 .$$

**翻译**：**答案——那个规律**：最优阈值停在 **0.11**，本质上就是 $\text{threshold}^\ast \approx \frac{\text{一次电话的成本}}{\text{一个客户的期望收益}} = \frac{1}{10} = 0.1$。

> **Why.** Calling a client with predicted probability $p$ has expected profit $10p - 1$. It is positive iff $p > 0.1$. So the rule "call iff $p > 0.1$" is the profit-maximising one, and the default 0.5 is far too conservative: it throws away every client whose probability lies between 0.1 and 0.5, and there are many of them.

**翻译**：**为什么**。给一个预测概率为 $p$ 的客户打电话，期望利润是 $10p - 1$。它为正当且仅当 $p > 0.1$。所以「当且仅当 $p > 0.1$ 才打」就是利润最大化的规则，而默认的 0.5 太保守了：它把概率在 0.1 到 0.5 之间的客户全部扔掉了——而这样的客户有很多。

> **How much is at stake:** expected profit per fold grows from 1813.2 (threshold 0.5) to 4127.8 (threshold 0.11) — i.e. the threshold alone is worth **twice as much as the entire C sweep**. This is the practical lesson of the whole part: for a probabilistic model the decision rule matters more than the hyperparameters.
>
> Because the revenue is random ($U[0,20)$, mean still 10), the *expected* revenue of a positive client stays at 10, so the optimal threshold stays around 0.1; only the neighbourhood of the optimum becomes noisy. The empirical curve is flat between ~0.05 and ~0.2 — inside that band the differences are within one standard deviation, which is why re-running the cell moves the argmax.

**翻译**：** stakes（利害关系）有多大**：每折期望利润从 1813.2（阈值 0.5）涨到 4127.8（阈值 0.11）——也就是说，**单阈值一项就值整个 C 扫描的两倍**。这是整个 Part 4 的实践教训：对概率模型来说，**决策规则比超参数更重要**。由于收益是随机的（$U[0,20)$，均值仍为 10），正类客户的**期望**收益仍然是 10，所以最优阈值稳定在 0.1 附近；变化的只是最优点附近变「噪」了。经验曲线在约 0.05~0.2 之间是平的——这个区间内的差异都在一个标准差以内，所以重新运行会让 argmax 移动。

### 在最优阈值下的 precision / recall

## 📖 原文

> What are the values of precision and recall at the selected threshold? Evaluate by cross-validation. Also calculate the standard deviation.

## 🇨🇳 翻译

在选定的阈值下，precision 和 recall 是多少？用交叉验证评估，并计算标准差。

### 📊 结果（本次运行）

| 指标 | 均值 ± 标准差 |
|---|---|
| precision | 33.8% ± 0.8% |
| recall | 64.4% ± 2.2% |

**怎么解读：**

- **recall 大幅提升**（从阈值 0.5 时的 ~23% 涨到 64.4% ± 2.2%），因为我们把阈值降到了 0.1 附近，放进了大量「中等概率」的客户。
- **precision 掉到 33.8% ± 0.8%**，也就是每打 3 个电话只有 1 个成交。
- **但这正是最优的**，而且 precision 33.8% **远高于**盈亏平衡的 10%。这两件事不矛盾，原因值得单独说清楚 👇

### 🧠 为什么最优点上 precision 会高于 10%

$10p-1>0$ 这条规则管的是**「边缘上的那一个客户」**，不是「所有被打的客户的平均」。

把我们打电话的名单想象成按概率从高到低排好队：

| 名单位置 | 这个客户的成交概率 | 打他赚不赚 |
|---|---|---|
| 最前面 10% | 60%~70% | 大赚 |
| 中间 | 30%~40% | 赚 |
| 最后那批（正好卡在阈值 0.11 上） | **≈10%** | 不赚不亏 |
| 阈值以下的（我们没打） | < 10% | 亏 |

所以**全部打出去的电话平均下来** precision 是 33.8%，但**最后接受的那一个**只有 10%。最优点就是「再加一个就亏了」的那个位置。这正是 $10p-1=0$ 的解。

> 这个区分很重要：很多同学会算「平均 precision 34% > 10%，那我应该继续降阈值啊」——错。降到下一个位置，新增的那个客户只有 9% 的成交率，他带来的边际利润是负的，虽然平均值还是正的。**决策看边际，不看平均。**

> 这就是为什么「用 accuracy / F1 挑模型」在这种业务里是错的：F1 会认为 66% precision + 22% recall 比 12% precision + 75% recall 更好，但实际上后者赚得更多。**指标必须和钱的公式对齐。**

### 📄 修正版解读格全文翻译（solved 版 Interpretation）

> **Interpretation.** At the profit-optimal threshold the model calls far more clients: recall jumps from ~22% (threshold 0.5) to 64.4 ± 2.2, while precision falls from ~66% to 33.8 ± 0.8.
>
> The subtle point: 33.8 ± 0.8 is well **above** the break-even precision of 10%, and that is not a contradiction. The $1/10$ rule applies to the **marginal** client at the cutoff, not to the average over everyone we call. The first clients on the list convert at 60-70%, the last ones — right at the threshold — at ~10%, and the average falls in between. Profit is maximised exactly when the *last accepted* client stops paying for himself, which is why the average precision stays comfortably above the break-even value.

**翻译**：**解读**：在利润最优的阈值下，模型打电话给多得多的客户：recall 从约 22%（阈值 0.5）跳到 64.4 ± 2.2，而 precision 从约 66% 降到 33.8 ± 0.8。微妙之处在于：33.8 ± 0.8 远**高于** 10% 的盈亏平衡 precision，而这并不矛盾。1/10 规则约束的是**分界线上的边际客户**，不是「我们打的所有人的平均」。名单最前面的客户成交率 60~70%，最后一批——正好卡在阈值上的——约 10%，平均值落在中间。利润恰好在「最后被接受的那个客户刚好不再为自己买单」时达到最大，这就是为什么平均 precision 能舒适地停在盈亏平衡线之上。

## 评估「高召回区间」的 PR-AUC

## 📖 原文

> As you probably already figured out, your model tends to have a higher recall. Try to evaluate the quality of the model using `PR-AUC` in the $\ge 0.5$ recall zone. Do it this way - select only those thresholds that achieve the desired recall, then interpret the unit square clipping rectangle as a new unit square and calculate the area under the clipping curve.

## 🇨🇳 翻译

你可能已经发现了，你的模型倾向于更高的 recall。试着用 **recall ≥ 0.5 区间内的 PR-AUC** 来评估模型质量。做法是：只保留那些能达到所需 recall 的阈值，然后把「被裁出来的矩形」重新当作一个单位正方形，计算裁剪后曲线下的面积。

### 🧠 为什么要「裁剪」

普通 AUC-PR 平均了**整条曲线**，包括：
- recall 很低（只打最自信的少数人）的那一段 → 这段在实际业务里**永远不会用**，因为我们肯定要打到 70% 的 recall；
- recall 很高（全都打）的那一段 → 这段也没意义，那是纯亏钱。

我们真正工作的区间是 recall ∈ [0.5, 1]。所以应该**只在这个区间里比较模型**。做法是把这个矩形重新归一化成一个单位正方形：

```python
def pr_auc_above_recall(y_true, scores, min_recall=0.5):
    prec, rec, _ = precision_recall_curve(y_true, scores)
    mask = rec >= min_recall
    rec_c = (rec[mask] - min_recall) / (1 - min_recall)     # 把 [0.5,1] 映射到 [0,1]
    return auc(rec_c, prec[mask])
```

| 名字 | 作用 |
|---|---|
| `mask = rec >= min_recall` | 只保留 recall ≥ 0.5 的那些点 |
| `(rec[mask] - min_recall) / (1 - min_recall)` | 平移 + 缩放：0.5→0，1→1 |
| `auc(rec_c, prec[mask])` | 算这块裁剪区域的面积 |

> ⚠️ 一个实现细节：`precision_recall_curve` 返回的 recall 是**递减**的。`sklearn.metrics.auc` 会检测到这一点并自动乘以 −1，所以结果是正的。如果你手写梯形积分（`np.trapz`），必须自己处理方向，否则会得到负数。

### 📊 结果（本次运行）

| 指标 | 均值 ± 标准差 |
|---|---|
| AUC-PR（全区间） | 0.4511 ± 0.0186 |
| AUC-PR（recall ≥ 0.5） | 0.2587 ± 0.0113 |

**解读**：裁剪后的数字明显低于全区间。这不是坏消息，而是**更诚实的消息**：它告诉我们「在我们真正工作的那个区间里，模型的实际水平是多少」。全区间 AUC-PR 里有一段「高精度、超低召回」的区域在拉高平均分，而那一段我们根本用不到。

> 这也是一个通用的报告技巧：**任何平均值都要说清楚它在哪个区间上平均的。** 一个电商推荐系统报「整体 AUC 0.9」，而它的业务只关心 top-1% 的推荐质量——这个 0.9 可能毫无意义。

### 📄 修正版答案格全文翻译（solved 版「裁剪 PR-AUC」Answer）

> **Answer.** The clipped AUC-PR (recall ≥ 0.5) is much smaller than the full one (0.2587 ± 0.0113 vs 0.4511 ± 0.0186) — and that is informative: it says "in the working zone that actually matters for this business, the model is considerably weaker than the overall AUC-PR suggests". The full AUC-PR averages over the entire curve, including the high-precision / very-low-recall region that we never use because it would mean calling almost nobody.
>
> This is the general lesson about reporting a metric: **always say over which region it was averaged.** The same model can honestly report 0.45 or 0.26 depending on the slice you look at.
>
> Note on the implementation: `sklearn.metrics.auc` detects that recall is **decreasing** and multiplies the trapezoid area by −1, so the value comes out positive. If you implement it by hand, do not forget to reverse the axis or take the absolute value.

**翻译**：**答案**：裁剪后的 AUC-PR（recall ≥ 0.5 的区间）远小于全区间的值（0.2587 ± 0.0113 vs 0.4511 ± 0.0186）——而这个差异本身是有信息量的：它说「在对这个业务真正重要的工作区间里，模型比整体 AUC-PR 所显示的要弱不少」。全区间 AUC-PR 对整条曲线取平均，包括「高精度、超低召回」那一段——而那段我们永远不会用，因为那就等于几乎不给任何人打电话。这是关于「如何报告指标」的通用教训：**永远说清楚你的指标是在哪个区间上平均的。** 同一个模型，看你切哪一段，可以诚实地报出 0.45 或 0.26。实现细节：`sklearn.metrics.auc` 会发现 recall 是**递减**的，并自动把梯形面积乘以 −1，所以结果为正；如果你自己手写积分，别忘了反转坐标轴或取绝对值。

## 彩蛋题：你早餐吃了什么？

## 📖 原文

> __Bonus:__ what did you eat for breakfast on the day of this homework? ))

## 🇨🇳 翻译

**加分彩蛋**：做作业那天你早餐吃了什么？))

> 这是俄罗斯课程的传统彩蛋题，用来确认你「真的把作业读到了最后一行」。solved 版的回答是：*「A bowl of oatmeal and a rather large coffee — the standard "I have a deadline" breakfast. ))」*——一碗燕麦粥加一大杯咖啡，标准的「我有截止日期」早餐。你也写一句自己的就行，**写了就给分**。

---
# 6. 原解答的错误清单与修改记录

我把原解答逐段核对了一遍，并在真实环境（Python 3.13 / scikit-learn 1.9 / pandas 3.1）里完整跑了一遍。下面是**确认存在问题**的地方、**为什么是问题**、以及**怎么改的**。

## 6.1 真正的错误（会影响结果或结论）

### ① 数值特征没有标准化 → 逻辑回归根本没收敛

**位置**：Part 2 的 `make_pipe` / `counter_pipe` / `pipe_md`，Part 4 的 `bank_pipe`。

**现象**：整份 notebook 跑下来会刷出**几十条**

```
ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s)
```

**为什么是错误**：不只是「有警告不好看」。两个层面：

1. **数学上错了**。L2 正则项是 $\frac{1}{2C}\lVert w\rVert^2$，它惩罚的是权重的**绝对值**，而权重的大小取决于特征的量级。`nr.employed` 取值范围是 4963~5228，`euribor3m` 是 0.6~5.1——不标准化就等于**对这两个特征施加了完全不同强度的惩罚**，这不是我们想要的。
2. **数值上没算完**。`lbfgs` 在 1000 次迭代内没走到最优点，返回的是一组「半成品」系数。所有基于它的数字（AUC-PR、利润）都带一点系统性偏差，而且**换台机器、换个版本可能就不一样**了。

**修改**：在 `ColumnTransformer` 里把数值列从 `"passthrough"` 换成 `StandardScaler()`：

```python
("num", StandardScaler(), num_cols)
```

改完之后警告全部消失，训练也快了（收敛快 = 迭代少）。

> Part 3 里**故意**不标准化，因为那一节本身就在讨论「不标准化的后果」，所以那里保留原状，只是把 `max_iter` 从 1000 提到 5000，并在注释里写明原因。

### ② Task 6 的结论与实测结果相反

**位置**：原解答在 Task 6 之后没有写结论（只有一段代码）；如果把常见的「标准答案」搬过来，会说「不加噪声会过拟合，加噪声就好了」。

**实测结果**（打印 train 和 test 两个分数才看得出来）：

| noise σ | train AP | test AP |
|---|---|---|
| 0.00 | 0.4216 | 0.4318 |
| 0.20 | 0.4205 | 0.4297 |

**train 分数比 test 还低**——也就是说，这里**根本没有发生过拟合**，加噪声没有用，σ 大了反而更差。

**为什么**：泄漏的严重程度取决于「类别基数 / 样本数」的比值，以及模型容量。本数据每类几百个样本、逻辑回归每列只有一个系数，泄漏被稀释到 0。真正危险的是「用户 ID」这种基数接近样本数的特征。

**修改**：重写结论，如实说明「这里没有泄漏」并给出原因。这比硬套一个漂亮结论有价值得多，也正好呼应作业最后那段 *"Complicating methods ... does not guarantee better model results"*。

### ③ 校准函数里的两个 bug（作业要求找出来的）

```python
bin_middle_points.append((l - r) / 2)                                   # ✗ 得到负数
bin_real_ratios.append(np.min(y_test[(preds >= l) & (preds < r)] == 1)) # ✗ 要比例不是最小值
```

- `(l - r) / 2` → `(l + r) / 2`：中点公式写成了差值，第一个箱是 $(0-0.1)/2=-0.05$，横轴全为负。
- `np.min(...)` → `y_test[mask].mean()`：要的是**正类占比**，`min` 只会在「全正」时给 1、否则给 0，曲线变成锯齿。

**顺带补了两个健壮性细节**：最后一个箱用 `<=` 右闭合（否则 `p == 1.0` 的样本被丢掉），空箱 `continue`（否则 `nan` 会让曲线断裂）。

### ④ 加噪声时，预测阶段也在加噪声

**位置**：`TargetEncoder.transform`。

原实现里 `transform` 用 `random_state+1` 也往特征里注入了高斯噪声。**在测试时注入随机噪声只会降低精度**——噪声是训练期的正则化手段，不是推理流程的一部分。

**修改**：`transform` 不再加噪声（`rng=None`），噪声只在 `fit_transform`（训练）里加。

### ⑤ Part 4 只算了 profit，没有分开 revenue / cost

**位置**：原来的 `profit_cv`。

作业明确问了两个问题：「平均赚多少」和「**要付多少给接线员**」。只返回一个 `profit` 数字，第二个问题根本答不了（你只能反推）。

**修改**：返回一个 DataFrame，包含 `tp / calls / revenue / cost / profit` 五列。这样：

- 平均利润 = **$1813.2 ± 186.8**
- 付给接线员 = **$322.8/折**，占收入的 **15.1%**

而且逐折的表格更容易看出「哪一折特别差」。

### ⑥ Part 4 三个问答题只有图没有答案

**位置**：C 调参后、多次运行后、阈值扫描后、precision/recall 后、裁剪 PR-AUC 后。

原解答都是「跑一段代码，画个图」，但作业每个格子都有一句明确的提问（*Did you get any win? / Is there any pattern? / What are the conclusions?*）。这些是**要打分的**。

**修改**：每一处补上结论段落。其中最关键的一条是阈值那个「规律」：

$$\text{threshold}^\ast \approx \frac{\text{一次电话成本}}{\text{客户期望收益}} = \frac{1}{10} = 0.1$$

### ⑦ 同一个问题被粘贴了两遍

**位置**：Part 2 Task 3 的 markdown 格。`__Question__: why ... AUC-PR` 及其回答出现了两次（一次俄文格式一次英文格式）。已合并成一处。

### ⑧ 对「随机分类器 AUC-PR 为什么是 0.75」的解释是错的

**位置**：Part 1 的两个问答题（`y_random` 为什么是小数、两种 AUC-PR 算法的差别）。

原解答的说法是：「`DummyClassifier(strategy='uniform')` 对每个样本随机抽一个类，`predict_proba` 报告抽中正类的比例」——**这是错的**。实测它的 `predict_proba` 是**恒定 0.5**：

```python
np.unique(dummy.predict_proba(X_test)[:, 1])   # -> array([0.5])
```

`strategy='uniform'` 返回的是**均匀先验** $1/K$，对所有样本都一样；随机性只体现在 `predict`（抽类）上。

进而「梯形法因为曲线锯齿而高估」也是错的。真正的原因是：分数恒定 → 只有 **1 个阈值** → `precision_recall_curve` 只返回 2 个点 $(recall=1, precision=0.5057)$ 和补上的锚点 $(recall=0, precision=1)$ → 梯形法把这两点连成一条斜线，面积 $\frac{1+0.5057}{2}=0.7528$。**和实测值分毫不差**，这才是正确解释。`average_precision_score` 给 0.5057 = 正类比例，是对的。

**修改**：两处答案全部重写，并把机制算给读者看。

### ⑨ 前向选择的候选池只有 40 个，导致「选 40 个」= 全选

**位置**：Part 3 的前向选择。

```python
candidates = list(np.argsort(-t_stat)[:40])   # 候选池 = 40 个
while len(selected) < 40:                     # 目标是选 40 个
```

候选池正好等于目标数量，循环结束时把池子里的 40 个全部选完——**这次「前向选择」什么也没筛，结果和 filter 完全一样**（我跑出来两者都是 0.45162，一模一样，正好暴露了这个问题）。

**修改**：候选池扩到 60 个（`[:60]`），这样才真的在「60 选 40」，结果变成 0.44427，与 filter 的 0.45162 拉开差距。

### ⑩ `n_jobs=-1` 在 Windows 上留下的进程池

**位置**：Part 1 的两处 `GridSearchCV`（原解答只改了第一处，第二处漏了）。

`n_jobs=-1` 会启动约 32 个 loky 后台进程（每个占约 150 MB）。在 Jupyter 里这没什么问题，但如果 notebook 是被嵌入执行（脚本化运行），子进程会反复重新导入主模块，出现卡住或内存爆掉。这类数据量（7000×10、3 万×62）单进程几十毫秒就跑完了，**开并行只会更慢**。

**修改**：两处都去掉 `n_jobs`，用默认的单进程。



## 6.2 措辞或数据不一致（不影响运行，但会被扣分）

| 位置 | 原文 | 问题 | 改成 |
|---|---|---|---|
| Task 1.1 对比结论 | "on data that is **almost linearly separable**" | 实测 AUC-ROC 0.85，属于「中等可分」，不是「几乎可分」。这个说法会误导读者以为 SVM 的间隔性质起了大作用 | 改为「中等可分」，并把「两者打平」归因于**同为线性模型** |
| Task 1.1 对比表的 LR 行 | AUC-PR 那一列是 `—` | 表格有空格，说明没算 | 补上阶梯法数值，并强调两种算法的差异 |
| 直方图那题 | "LR predicted probabilities **pile up at the ends**" | 实测只有 5.0% 落在 0.05 以下或 0.95 以上，谈不上「堆积」；SVM 是 0.2% | 改成基于实测的描述：min-max 是**线性拉伸**，只是把距离换了个刻度，形状没变 |
| Part 3 总结表 | 全部特征写作 "62" 但同时又写 "all 62 features" 与 19 列原始特征混用 | 容易混淆（19 是编码前，62 是 one-hot 后） | 明确标注「编码前 19 / 编码后 62」 |
| Part 2 Task 8（年龄） | 打印 `"age has %d unique values (float, from 17 to 98)"` | 硬编码了「float, 17~98」，换数据就错；实测是 78 个唯一值 | 改成运行时计算 `.nunique()` / `.min()` / `.max()` |
| Part 2 月份那题 | 两行 print 都指向同一个模型（`p_base`），标签不同 | 极易误读为「两个模型的对比」 | 显式打印 `p_base` 与 `p_md` 两个不同模型 |

## 6.3 版本兼容性

| 位置 | 问题 | 处理 |
|---|---|---|
| `LinearSVC(..., dual="auto")` | `dual` 参数在 sklearn < 1.3 不存在，会直接报 `InvalidParameterError` | 删掉该参数（新版本默认就是 `auto`），代码在 1.0+ 都能跑 |
| `GridSearchCV(..., n_jobs=-1)` | Windows 上会启动子进程，在部分环境（如脚本内嵌执行）会失败 | Part 1 的两处去掉 `n_jobs`（数据小，单进程足够快） |
| `X[c].dtype == object or str(X[c].dtype) in ("str", "string")` | pandas 3.0 起字符串默认是 `str` 类型而非 `object` | 原解答已经写对了，保留并加注释说明 |

> 顺带一句：作业原文里有两个「Task 8」（Part 2 的年龄题、Part 3 的嵌入式选择），这是出题人的编号失误，不是你的问题。我在讲解里按位置区分了。

---
# 7. 附录

## 7.1 变量速查表（按出现顺序）

### Part 1

| 名字 | 是什么 | 形状/类型 |
|---|---|---|
| `X`, `y` | 人造数据：特征矩阵 / 标签 | `(10000, 10)`, `(10000,)` |
| `X_train` / `X_test` | 切分后的训练/测试特征 | `(7000,10)` / `(3000,10)` |
| `y_random` | 随机分类器给的正类概率 | `(3000,)`，取值 0~1 |
| `svm_search` | `GridSearchCV` 对象，负责选 `C` | — |
| `svm_search.best_params_` | 选出来的最优参数字典 | `{'C': 3.73}` |
| `svm_search.best_score_` | 最优参数下的交叉验证平均分 | float |
| `svm` | 用最优 `C` 重新训练好的 `LinearSVC` | — |
| `y_svm_score` | SVM 的 `decision_function` 输出（有符号距离） | `(3000,)`，无界 |
| `y_svm_scaled` | 把上面 min-max 到 `[0,1]` | `(3000,)` |
| `lr_search` / `lr` | 逻辑回归的网格搜索 / 最优模型 | — |
| `y_lr_prob` | LR 的正类概率 | `(3000,)`，`[0,1]` |
| `X2`, `y2` | 只有 2 个特征的数据（为了能画图） | `(400,2)`, `(400,)` |
| `model_svm2` | 二维数据上训好的 `SVC(kernel='linear')` | — |
| `model_svm2.support_vectors_` | 支持向量的坐标 | `(n_sv, 2)` |
| `model_lr2` | 二维数据上的逻辑回归 | — |
| `svm_cal` | 套了 Platt 校准的 SVM | — |
| `y_svm_cal` | 校准后的概率 | `(3000,)` |
| `platt` | 自己拟合的一维逻辑回归（校准器） | — |
| `oof_scores` | out-of-fold 分数 | `(7000,)` |

### Part 2

| 名字 | 是什么 |
|---|---|
| `df` | 银行营销原始表，`(41188, 21)` |
| `X`, `y` | 去掉 `duration` 后的特征 / 布尔标签 |
| `cat_cols` | 类别列名列表（10 个） |
| `num_cols` | 数值列名列表（9 个） |
| `make_pipe(mode, C)` | 造「编码 + 逻辑回归」流水线的工厂函数 |
| `pipe_ord` / `p_ord` | 序数编码的流水线 / 其测试集预测 |
| `pipe_ohe` / `p_ohe` | 独热的流水线 / 其预测 |
| `t_ord`, `t_ohe`, `t_cnt` | 各自的「编码 + 训练」耗时（秒） |
| `TargetEncoder` | 自己写的计数器 transformer |
| `.cols_` | `fit` 后确定：要对哪些列编码 |
| `.maps_` | `fit` 后确定：`{列名: Series(类别 → 编码值)}` |
| `.global_mean_` | 训练集的全局正类比例（没见过类别时用它兜底） |
| `counter_pipe(...)` | 造「计数器 + 逻辑回归」流水线 |
| `best_sigma` | 噪声实验中测试集表现最好的 σ |
| `best_smooth` | 平滑实验中表现最好的 `C` |
| `cats_with_age` | `cat_cols` 再加上 `'age'`（把年龄当类别） |
| `month_map` / `dow_map` | 月份/星期 → 数字的映射字典 |

### Part 3

| 名字 | 是什么 |
|---|---|
| `cat3` / `num3` | 类别列 / 数值列（重新算一遍） |
| `ohe` | 稀疏输出的 `OneHotEncoder` |
| `Xtr` / `Xte` | 稀疏的 one-hot 特征矩阵，`(32950, 62)` / `(8238, 62)` |
| `feature_names` | 62 个特征的列名 |
| `fit_eval(Xa, ya, Xb, yb)` | 训练 + 返回 (模型, AUC-PR, 分数) 的小工具 |
| `model_all` / `ap_all` | 用全部特征的模型 / 其 AUC-PR |
| `top40_emb` | 按 `\|coef\|` 选出的 40 个列下标 |
| `scaler` | `StandardScaler(with_mean=False)` |
| `Xtr_s` / `Xte_s` | 标准化后的稀疏矩阵 |
| `t_stat` | 每个特征的 t 统计量，长度 62 |
| `top40_t` | 按 t 值选出的 40 个列下标 |
| `Xw_tr` / `Xw_val` | wrapper 内部再切出的训练子集 / 验证子集 |
| `fast_lr()` | 搜索阶段用的快速逻辑回归 |
| `val_score(cols)` | 「给定列下标 → 验证集 AUC-PR」 |
| `remaining` | 当前还留着的特征下标列表 |
| `fwd_scores` | 前向选择每一步的验证分数（画学习曲线用） |

### Part 4

| 名字 | 是什么 |
|---|---|
| `df4`, `X4`, `y4` | 重新读入的数据（`y4` 是 0/1 整数） |
| `cat4` / `num4` | 类别列 / 数值列（9 个数值） |
| `bank_pipe(C)` | one-hot + 标准化 + 逻辑回归 的流水线 |
| `cv_economics(...)` | 5 折 CV 算经济指标，返回逐折 DataFrame |
| `res` | 固定收益 $10 的结果表 |
| `res_rnd` | 随机收益 $U[0,20)$ 的结果表 |
| `profit_stats(**kw)` | 只要利润数组的便捷包装 |
| `thresholds` | `0.00, 0.01, ..., 1.00` 共 101 个阈值 |
| `threshold_profits(...)` | 一次拟合、评估所有阈值，返回 `(101, 5)` 矩阵 |
| `store` | 上面那个矩阵 |
| `best_t` | 利润最大的阈值 |
| `pr_auc_above_recall(...)` | 裁剪到 `recall ≥ 0.5` 的 PR-AUC |

## 7.2 常用函数速查

| 函数 | 一句话说明 |
|---|---|
| `train_test_split(X, y, test_size, random_state, stratify)` | 切分训练/测试。`stratify=y` 保持类别比例 |
| `make_classification(...)` | 造人造分类数据 |
| `DummyClassifier(strategy=...)` | 不学习的基准模型 |
| `GridSearchCV(est, param_grid, scoring, cv)` | 网格搜索 + 交叉验证 |
| `StratifiedKFold(n_splits, shuffle, random_state)` | 分层 K 折。`.split(X, y)` 产出下标 |
| `LogisticRegression(C, max_iter)` | L2 正则的逻辑回归 |
| `LinearSVC(C, max_iter)` | 线性 SVM，**没有** `predict_proba` |
| `SVC(kernel='linear', C)` | 核 SVM，**有** `support_vectors_` |
| `decision_function(X)` | 到分隔面的有符号距离 |
| `predict_proba(X)[:, 1]` | 正类概率（第 1 列） |
| `precision_recall_curve(y, s)` | 返回 `(precision, recall, thresholds)` |
| `roc_curve(y, s)` | 返回 `(fpr, tpr, thresholds)` |
| `average_precision_score(y, s)` | **标准**的 AUC-PR（阶梯法） |
| `roc_auc_score(y, s)` | AUC-ROC |
| `auc(x, y)` | 梯形法面积。注意横轴在前 |
| `CalibratedClassifierCV(est, method, cv)` | 概率校准。`method` 取 `sigmoid` / `isotonic` |
| `OneHotEncoder(handle_unknown, sparse_output)` | 独热编码 |
| `OrdinalEncoder(handle_unknown, unknown_value)` | 序数编码（按字母序编号） |
| `StandardScaler(with_mean)` | 标准化。稀疏矩阵要用 `with_mean=False` |
| `ColumnTransformer([...])` | 对不同列做不同处理 |
| `Pipeline([...])` | 把预处理和模型焊在一起 |
| `np.logspace(a, b, n)` | 等比数列（`10^a` 到 `10^b`） |
| `np.argsort(-x)[:k]` | 取最大的 k 个的**下标** |
| `np.meshgrid` | 生成二维网格坐标（画等高线/热力图用） |
| `plt.contour(X, Y, Z, levels)` | 画等高线（SVM 的分隔线） |
| `plt.imshow(Z, extent, origin)` | 把二维数组当图片画（概率面） |
| `plt.errorbar(x, y, yerr)` | 带误差棒的折线图 |
| `scipy.sparse.hstack` | 稀疏矩阵横向拼接 |
| `.toarray()` | 稀疏 → 密集（小心内存） |

## 7.3 怎么复现

```bash
# 1. 数据（放在 notebook 同级的 bank-additional/ 目录下）
#    https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
unzip bank-additional.zip

# 2. 依赖
pip install "scikit-learn>=1.3" pandas numpy matplotlib scipy

# 3. 从上到下 Run All
#    Part 1 约 30 秒
#    Part 2 约 1 分钟
#    Part 3 约 2 分钟（wrapper 最慢）
#    Part 4 约 3 分钟（交叉验证反复训练）
```

**注意**：作业里所有答案中的数字都来自一次具体运行。交叉验证的划分是固定的（`random_state` 已给定），所以**大部分数字可以精确复现**；但 Part 4 故意不固定「客户收益」的随机种子，那部分的利润数字每次都会不同，这正是作业想让你看到的现象。

## 7.4 常见报错

| 报错 | 原因 | 解决 |
|---|---|---|
| `FileNotFoundError: bank-additional/...csv` | 数据没解压 / 路径不对 | 解压到 notebook 同级目录 |
| `ConvergenceWarning: lbfgs failed to converge` | 特征没标准化，或 `max_iter` 太小 | 加 `StandardScaler`，或提高 `max_iter` |
| `ValueError: Unknown label type: 'continuous'` | `y` 传成了浮点概率 | `y` 必须是 0/1 或布尔 |
| `ValueError: Input contains NaN` | 编码后出现缺失（没做 `fillna`） | `TargetEncoder` 里 `fillna(global_mean)` |
| `AttributeError: 'LinearSVC' object has no attribute 'predict_proba'` | `LinearSVC` 不支持概率 | 用 `decision_function`，或改用 `SVC(probability=True)` / `CalibratedClassifierCV` |
| `InvalidParameterError: dual` | sklearn 版本 < 1.3 | 删掉 `dual="auto"` |
| `KeyError: "['duration'] not found in axis"` | 重复 drop 了已经删掉的列 | 检查是否已删过 |
| 内存爆掉 | one-hot 后用了密集矩阵 | 用 `sparse_output=True` + `scipy.sparse` |

---
## 最后：这份作业真正想教你的四件事

1. **指标要和目标对齐。** 不平衡数据用 AUC-PR；有成本的决策要算利润；只关心某个召回区间就裁剪指标。没有「万能指标」。
2. **不能偷看答案。** 测试集只能用一次；`duration` 这种「事后才知道」的特征必须删；目标编码必须 out-of-fold。这三件事本质上是同一件事。
3. **方法和数据规模绑定。** one-hot 在小基数上更好，计数器在大基数上才必要；filter 在海量特征时是唯一选择，wrapper 只在特征少时可行。**没有普适最优，只有场景最优。**
4. **区分「理论上应该」和「实测结果」。** 平滑理论上更好，实测没差别；加噪声理论上防泄漏，实测这数据压根没泄漏；年龄用计数器理论上更合理，实测略降。**先测，再下结论。**